In [8]:
"""
Lucro 平台数据周报 v1.0 — 本周: 20260605-20260611
对比基准：上周 20260529-20260604
无道具数据，跳过道具章节
"""
from pathlib import Path
import io, warnings
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec, matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image, PageBreak, HRFlowable, KeepTogether)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# ── 路径 ────────────────────────────────────────────────
DATA_ROOT  = Path(r"D:\周报更新版\Lucro")
OUTPUT_DIR = Path(r"D:\周报更新版\Lucro\输出")

THIS_WEEK = ("20260612", "20260618")
LAST_WEEK = ("20260605", "20260611")
REPORT_END = THIS_WEEK[1]

RET_TARGETS = {"次留": 21.0, "3留": 15.0, "7留": 11.0}

FN, FNB = "WQY", "WQYB"

C_BLUE   = colors.HexColor("#1d4ed8");  C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669");  C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706");  C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b");  C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white;                C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1");  C_ROW    = colors.HexColor("#f8fafc")
C_TEAL   = colors.HexColor("#0f766e");  C_TARGET = colors.HexColor("#0369a1")
C_ORANGE = colors.HexColor("#ea580c")

PW, PH = A4
MARGIN  = 1.6 * cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
CR_HIGH, CR_LOW = 17, 5

MFR_SHORT = {"Rectangle":"RG","Pragmatic Play":"PP","PG Soft":"PG","PlayTech":"PT",
             "Originals":"自研","Fat Panda":"FP","Tada":"Tada","KA Gaming":"KA",
             "3 Oaks Gaming":"3Oaks","Evolution":"Evolutio","SmartSoft":"SmartSof",
             "AvatarUx":"AvatarUx","NetEnt":"NetEnt","Relax Gaming":"Relax"}

TRUNCATE = {"首充次日复充率":1,"首充次日复投率":1,"首充当日复充率":0,
            "首充7日复充率":6,"首充2日复充率":1,"首充3日复充率":2}

FILES: dict = {}

# ── 文件匹配 ─────────────────────────────────────────────
def find_file(patterns):
    if isinstance(patterns, str):
        patterns = [patterns]

    hits = []

    for f in DATA_ROOT.rglob("*"):
        if f.is_file():
            if all(p in f.name for p in patterns):
                hits.append(f)

    return max(hits, key=lambda f: f.stat().st_mtime) if hits else None

def resolve_files():
    global FILES
    km = {
        "platform":      ["平台报表_USD"],
        "daily":         ["日报-大盘日报"],
        "retention":     ["整体 首充留存"],
        "agent_plat":    ["平台报表-总代_USD_"],
        "agent_promo":   ["推广报表-总代_USD"],
        "agent_ret":     ["首充充值留存_全量数据"],
        "vip":           ["VIP报表_USD"],
        "dt_tw":         ["top提款用户_全量数据"],
        "dt_lw":         ["top提款用户_全量数据"],
        "dc_tw":         ["头部充值用户_全量数据"],
        "dc_lw":         ["头部充值用户_全量数据"],
        "pref_tw":       ["本周top500提款用户游戏偏好"],
        "pref_lw":       ["上周top500提款用户游戏偏好"],
        "mfr":           ["厂商投注数据_全量数据"],
        "game_tw":       ["游戏报表-详情_USD_本周"],
        "game_lw":       ["游戏报表-详情_USD_上周"],
        "gift":          ["各活动赠送_全量数据"],
        "vip_ret_chg":   ["VIP充值-充值_近28天"],
        "vip_ret_act":   ["VIP充值-活跃_近28天"],
        "first_dep_ret": ["首次充值活动用户充值留存情况"],
    }
    fail = 0
    for key, kws in km.items():
        p = find_file(kws)
        if p:
            FILES[key] = p
            print(f"  ✅ [{key:15s}] {p.name}")
        else:
            print(f"  ❌ [{key:15s}] 找不到含{kws}的文件")
            fail += 1
    print(f"  ▶ 文件匹配完成（{len(km)-fail}/{len(km)}）\n")

# ── 工具函数 ──────────────────────────────────────────────
def shorten_mfr(n):
    if not isinstance(n, str): return str(n)
    for k, v in MFR_SHORT.items():
        if k in n: return v
    return n[:8]

def ret_end(week_start, lag):
    e = (datetime.strptime(REPORT_END, "%Y%m%d") - timedelta(days=lag)).strftime("%Y%m%d")
    return e if e >= week_start else None

def fmt_lbl(d): return f"{d[4:6]}/{d[6:]}" if d else "-"

def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns if c not in ("日期","总代.名称","name_总代","总代.ID")]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o):   return (n-o)/abs(o)*100 if o and o != 0 else 0.0
def pct_vec(ns, os):
    return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop else s
    nz = v[v > 0]; return nz.mean() if len(nz) else v.mean()

def wavg_series(vals, weights):
    ok = vals.notna() & (weights > 0)
    if not ok.any(): return np.nan
    return float(np.average(vals[ok], weights=weights[ok]))

def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)

def setup():
    import os
    from matplotlib import font_manager as fm

    fonts = [
        r"C:\Windows\Fonts\msyh.ttc",      # 微软雅黑
        r"C:\Windows\Fonts\msyhbd.ttc",
        r"C:\Windows\Fonts\simhei.ttf",    # 黑体
        r"C:\Windows\Fonts\simsun.ttc",    # 宋体
    ]

    font_path = None

    for f in fonts:
        if os.path.exists(f):
            font_path = f
            break

    if font_path is None:
        raise FileNotFoundError("找不到中文字体，请检查 Windows 字体目录")

    print(f"  ▶ 使用字体：{font_path}")

    pdfmetrics.registerFont(TTFont(FN, font_path))

    try:
        pdfmetrics.registerFont(TTFont(FNB, font_path))
    except:
        pass

    fm.fontManager.addfont(font_path)

    plt.rcParams.update({
        "font.family": "Microsoft YaHei",
        "axes.unicode_minus": False,
        "font.size": 8.5
    })

# ── PDF 组件 ──────────────────────────────────────────────
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.4,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_ORANGE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5), HRFlowable(width="100%",thickness=1.5,color=C_ORANGE),
                         Spacer(1,3), P(f"■  {text}",10,True,C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}",8.5,False,C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg), ("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10), ("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),   ("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None, fsize=6.8, extra_style=None):
    full = PW - 2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr), ("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),       ("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2),      ("BOTTOMPADDING",(0,0),(-1,-1),2),
        ("LEFTPADDING",(0,0),(-1,-1),2),     ("RIGHTPADDING",(0,0),(-1,-1),2),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1, len(rows)+1, 2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    if extra_style:
        for cmd in extra_style: st.add(*cmd)
    hrow = [P(h,fsize,True,C_WHITE,TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]),fsize, c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c,(list,tuple)) else P(str(c),fsize)
            for c in row])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t,bold=False,clr=colors.black,align=TA_LEFT): return (t,bold,clr,align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup=good_up
    c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{'+' if v>=0 else ''}{v:.{d}f}%",False,c,TA_RIGHT)
def gclr(v,t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)
def fret(v): return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"

def kpi_card4(items, cols=4):
    fw = (PW-2*MARGIN)/cols - 4
    rows, row = [], []
    for label,tv,lv,chg,_ in items:
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table([[P(label,7.5,False,C_GRAY)],[P(str(tv),14,True,C_DARK)],
                       [P(f"上周：{lv}",7.5,False,C_GRAY)],
                       [P(f"{'+' if chg>=0 else ''}{chg:.1f}%",8,True,pclr)]],
                      colWidths=[fw],
                      style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
                                        ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
                                        ("LEFTPADDING",(0,0),(-1,-1),8),
                                        ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(fw,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ════════════════════════════════════════════════════════════════
# 数据加载
# ════════════════════════════════════════════════════════════════
def load_platform():
    df = pd.read_excel(FILES["platform"]); df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]; lw = df[df["日期"].between(*LAST_WEEK)]

    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str); dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]; lw_d = dd[dd["日期"].between(*LAST_WEEK)]

    K = {}
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数","充值金额",
              "提现金额","提现人数","充提差","投注金额","投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    for c in ["首充转化率","首充当日复充率","首充次日复充率","首充次日复投率","活跃用户付费率","总赠送充值比"]:
        if c not in df.columns: continue
        drop = TRUNCATE.get(c, 0)
        K[f"tw_{c}"] = trunc_mean(tw[c], drop); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])

    K["tw_充提差比"] = tw["充提差"].sum()/tw["充值金额"].sum()*100 if tw["充值金额"].sum()>0 else np.nan
    K["lw_充提差比"] = lw["充提差"].sum()/lw["充值金额"].sum()*100 if lw["充值金额"].sum()>0 else np.nan
    K["pct_充提差比"] = pct(K["tw_充提差比"], K["lw_充提差比"])
    K["tw_盈亏率"] = tw["公司输赢"].sum()/tw["投注金额"].sum()*100 if tw["投注金额"].sum()>0 else np.nan
    K["lw_盈亏率"] = lw["公司输赢"].sum()/lw["投注金额"].sum()*100 if lw["投注金额"].sum()>0 else np.nan
    K["pct_盈亏率"] = pct(K["tw_盈亏率"], K["lw_盈亏率"])

    # 推广消耗（剔除本周最后1天）
    tw_d_asc = tw_d.sort_values("日期")
    tw_cost_valid = tw_d_asc["真实消耗"].iloc[:-1] if "真实消耗" in tw_d_asc.columns else pd.Series([0])
    lw_cost_days = len(lw_d); tw_cost_days = len(tw_cost_valid)
    K["tw_真实消耗_日均"]  = tw_cost_valid.sum()/tw_cost_days if tw_cost_days>0 else 0
    K["lw_真实消耗_日均"]  = lw_d["真实消耗"].sum()/lw_cost_days if lw_cost_days>0 and "真实消耗" in lw_d.columns else 0
    K["tw_真实消耗_有效天"] = tw_cost_days
    K["pct_真实消耗"] = pct(K["tw_真实消耗_日均"], K["lw_真实消耗_日均"])

    tw_sorted = tw.sort_values("日期")
    tw_cd_valid = tw_sorted["充提差"].iloc[:tw_cost_days]
    K["tw_充提差_日均_roi"] = tw_cd_valid.sum()/tw_cost_days if tw_cost_days>0 else 0
    lw_sorted = lw.sort_values("日期")
    K["lw_充提差_日均_roi"] = lw_sorted["充提差"].sum()/lw_cost_days if lw_cost_days>0 else 0
    K["tw_充提差ROI"] = K["tw_充提差_日均_roi"]/K["tw_真实消耗_日均"] if K["tw_真实消耗_日均"]>0 else 0
    K["lw_充提差ROI"] = K["lw_充提差_日均_roi"]/K["lw_真实消耗_日均"] if K["lw_真实消耗_日均"]>0 else 0
    K["pct_充提差ROI"] = pct(K["tw_充提差ROI"], K["lw_充提差ROI"])

    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])

    last14 = df.tail(14)
    trend = {"dates": [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
             "充值":    last14["充值金额"].tolist(),
             "提现":    last14["提现金额"].tolist(),
             "充提差比": last14["充提差比"].tolist(),
             "公司输赢": last14["公司输赢"].tolist(),
             "首充":    last14["首充人数"].tolist(),
             "注册":    last14["注册人数"].tolist()}
    return K, trend


def load_dashboard_retention():
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str)
    # Lucro日报留存列名：首充2日复充率 / 首充3日复充率 / 首充7日复充率
    specs = [("nd","首充2日复充率",1), ("td","首充3日复充率",2), ("sd","首充7日复充率",6)]
    for _,col,_ in specs:
        if col in dd.columns:
            dd[col] = pd.to_numeric(dd[col].astype(str).str.replace("%","").str.strip(), errors="coerce")
    R = {}
    for key, col, lag in specs:
        for ws, we, prefix in [(THIS_WEEK[0],THIS_WEEK[1],"tw_"), (LAST_WEEK[0],LAST_WEEK[1],"lw_")]:
            ec = ret_end(ws, lag)
            sub = dd[(dd["日期"]>=ws)&(dd["日期"]<=we)]
            if ec: sub = sub[sub["日期"]<=ec]
            R[f"{prefix}{key}"] = float(sub[col].mean()) if len(sub) and col in sub else np.nan
        R[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        R[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)
    return R


def load_weekly_retention():
    """Lucro整体首充留存：列名为1日/2日/3日/6日/7日（无"第"字）"""
    df = pd.read_csv(FILES["retention"])
    # 列名适配：Lucro 用 '初始事件发生时间'，列名是 '1日' 而非 '第1日'
    time_col = "初始事件发生时间" if "初始事件发生时间" in df.columns else "初始事件的发生时间"
    daily = df[~df[time_col].astype(str).str.contains("阶段值", na=False)].copy()
    daily["ds"] = daily[time_col].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
    dr = daily[daily["指标"]=="留存率"].copy()
    du = daily[daily["指标"]=="留存人数"].copy()
    ret_cols = []
    
    for c in dr.columns:
        cc = str(c).replace("第","")
        if cc.endswith("日"):
            ret_cols.append(cc)
    
    dr.columns = [str(c).replace("第","") for c in dr.columns]
    du.columns = [str(c).replace("第","") for c in du.columns]
    # ret_cols = ["1日","2日","3日","6日","7日"]
    for c in ret_cols:
        if c in dr.columns:
            dr[c] = pd.to_numeric(dr[c].astype(str).str.replace("%","").str.strip(), errors="coerce")

    user_col = "充值成功事件用户数" if "充值成功事件用户数" in du.columns else du.columns[1]

    def _d(s): return datetime.strptime(s, "%Y%m%d")
    def _f(d): return d.strftime("%Y-%m-%d")
    def _l(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"

    tw_s,tw_e = _d(THIS_WEEK[0]),_d(THIS_WEEK[1])
    lw_s,lw_e = _d(LAST_WEEK[0]),_d(LAST_WEEK[1])
    w2e = lw_s-timedelta(days=1); w2s = w2e-timedelta(days=6)
    w1e = w2s-timedelta(days=1); w1s = w1e-timedelta(days=6)
    weeks = [(f"第1周\n{_l(w1s,w1e)}",_f(w1s),_f(w1e)),
             (f"第2周\n{_l(w2s,w2e)}",_f(w2s),_f(w2e)),
             (f"上周\n{_l(lw_s,lw_e)}",_f(lw_s),_f(lw_e)),
             (f"本周\n{_l(tw_s,tw_e)}",_f(tw_s),_f(tw_e))]
    result = []
    for wk,s,e in weeks:
        mr = dr[(dr["ds"]>=s)&(dr["ds"]<=e)]
        mu = du[(du["ds"]>=s)&(du["ds"]<=e)]
        row = {"week": wk, "users": mu[user_col].sum() if user_col in mu.columns else 0}
        for col in ret_cols:
            rs = mr[col].values if col in mr.columns else np.array([])
            us = mu[user_col].values if user_col in mu.columns else np.ones(len(rs))
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs)
                row[col] = float(np.average(rs[v],weights=us[v])) if v.sum()>0 else np.nan
            else:
                row[col] = float(np.nanmean(rs)) if len(rs)>0 else np.nan
        result.append(row)
    return result


def load_agents():
    dr = pd.read_excel(FILES["agent_promo"]); dr["日期"]=dr["日期"].astype(str); dr=to_num(dr)
    tw_r=dr[dr["日期"].between(*THIS_WEEK)]; lw_r=dr[dr["日期"].between(*LAST_WEEK)]
    dp = pd.read_excel(FILES["agent_plat"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]; lw_p=dp[dp["日期"].between(*LAST_WEEK)]

    sa=["充值金额","提现金额","充提差","首充金额","首充人数","注册人数","充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        avail=[c for c in sa if c in d.columns]
        g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in avail}).reset_index()
        if "充提差" in g.columns and "充值金额" in g.columns:
            g["充提差率"]=g["充提差"]/g["充值金额"]*100
        return g
    tw_pa=agg_p(tw_p); lw_pa=agg_p(lw_p)

    sr=["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        avail=[c for c in sr if c in d.columns]
        g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in avail}).reset_index()
        if "总消耗" in g.columns and "一级首充人数" in g.columns:
            g["一级首充成本"]=g["总消耗"]/g["一级首充人数"].replace(0,np.nan)
        else:
            g["一级首充成本"]=np.nan
        return g
    tw_ra=agg_r(tw_r); lw_ra=agg_r(lw_r)
    if "总消耗" in lw_ra.columns and "一级首充人数" in lw_ra.columns:
        lw_ra["lw_fc_cost"]=lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
    else:
        lw_ra["lw_fc_cost"]=np.nan

    lw_sub = lw_pa[["总代.ID"]+[c for c in ["充值金额","注册人数","充提差率"] if c in lw_pa.columns]].rename(
        columns={"充值金额":"lw_充值","注册人数":"lw_注册","充提差率":"lw_充提差率"})
    m=tw_pa.merge(lw_sub, on="总代.ID", how="left")
    ra_cols=["总代.ID"]+[c for c in ["总消耗","一级首充成本","一级首充人数"] if c in tw_ra.columns]
    m=m.merge(tw_ra[ra_cols], on="总代.ID", how="left")
    lfc_cols=["总代.ID"]+["lw_fc_cost"] if "lw_fc_cost" in lw_ra.columns else ["总代.ID"]
    m=m.merge(lw_ra[lfc_cols], on="总代.ID", how="left")
    m["注册环比"]=pct_vec(m["注册人数"],m["lw_注册"].fillna(1)) if "注册人数" in m.columns and "lw_注册" in m.columns else 0.0
    if "充值金额" not in m.columns: m["充值金额"]=0
    return m.sort_values("充值金额", ascending=False)


def load_agent_ret():
    """Lucro总代留存：列名 1日/2日/7日"""
    df = pd.read_csv(FILES["agent_ret"])
    time_col = "初始事件发生时间" if "初始事件发生时间" in df.columns else df.columns[0]
    df["_d"] = df[time_col].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"] = df["_d"].str.replace("-","").fillna("")

    col_1 = "1日" if "1日" in df.columns else None
    col_2 = "2日" if "2日" in df.columns else None
    col_6 = "7日" if "7日" in df.columns else ("6日" if "6日" in df.columns else None)

    for c in [col_1, col_2, col_6]:
        if c and c in df.columns:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(), errors="coerce")

    user_col = "充值成功事件用户数" if "充值成功事件用户数" in df.columns else df.columns[3]
    if user_col in df.columns:
        df[user_col] = pd.to_numeric(df[user_col], errors="coerce").fillna(0)
    else:
        df[user_col] = 1

    name_col = "name_总代" if "name_总代" in df.columns else "总代.名称"
    ind_col  = "总代" if "总代" in df.columns else "总代.ID"

    daily = df[df["_yyyymmdd"].str.match(r"^\d{8}$", na=False)].copy()
    # 用总代ID匹配（留存文件名称前缀"运营_"，平台报表"运营部_"，名称无法join，改用ID）
    daily["_id_int"] = pd.to_numeric(daily[ind_col], errors="coerce")
    daily_ret = daily[daily["指标"]=="留存率"].reset_index(drop=True) if "指标" in daily.columns else daily.copy()
    daily_ret["_id_int"] = pd.to_numeric(daily_ret[ind_col], errors="coerce")

    ends = {}
    for key, lag in [("nd",1),("3d",2),("7d",6)]:
        ends[f"tw_{key}_end"] = ret_end(THIS_WEEK[0], lag)
        ends[f"lw_{key}_end"] = ret_end(LAST_WEEK[0], lag)

    def _slice(ws, we, ec):
        sub = daily_ret[daily_ret["_yyyymmdd"].between(ws,we)]
        if ec: sub = sub[sub["_yyyymmdd"] <= min(ec,we)]
        return sub

    slices = {
        "tw_nd": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_nd_end"]),
        "tw_3d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_3d_end"]),
        "tw_7d": _slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_7d_end"]),
        "lw_nd": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_nd_end"]),
        "lw_3d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_3d_end"]),
        "lw_7d": _slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_7d_end"]),
    }
    col_map = {}
    if col_1: col_map.update({"tw_nd":col_1,"lw_nd":col_1})
    if col_2: col_map.update({"tw_3d":col_2,"lw_3d":col_2})
    if col_6: col_map.update({"tw_7d":col_6,"lw_7d":col_6})

    def _wavg_by_id(df_sub, col):
        res = {}
        if not col or col not in df_sub.columns: return res
        for aid, grp in df_sub.groupby("_id_int"):
            if pd.isna(aid): continue
            ok = grp[col].notna() & (grp[user_col] > 0)
            if ok.any():
                res[int(aid)] = float(np.average(grp.loc[ok, col], weights=grp.loc[ok, user_col]))
        return res

    per_agent = {k: _wavg_by_id(slices[k], col_map.get(k)) for k in slices}
    all_ids = set().union(*[set(v) for v in per_agent.values()])
    ret = {}
    for aid in all_ids:
        ret[aid] = {k: per_agent[k].get(aid, np.nan) for k in per_agent}
    return ret, ends


def load_vip():
    df=pd.read_excel(FILES["vip"]); df["日期"]=df["日期"].astype(str)
    num=["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df=to_num(df,num)
    tw=df[df["日期"].between(*THIS_WEEK)]; lw=df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()


def load_vip_retention():
    res={}
    for fk,rt in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df=pd.read_csv(FILES[fk])
        time_col2 = "初始事件发生时间" if "初始事件发生时间" in df.columns else "初始事件的发生时间"
        df["ds"]=df[time_col2].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
        df["yyyymmdd"]=df["ds"].str.replace("-","").fillna("")
        # Lucro列名：1日/2日/3日/7日
        for c in ["1日","2日","3日","7日"]:
            if c in df.columns:
                df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
        n_col = "充值成功事件用户数" if "充值成功事件用户数" in df.columns else None
        df["n"]=pd.to_numeric(df[n_col],errors="coerce") if n_col else 1
        df["vip"]=pd.to_numeric(df["vip_level"],errors="coerce") if "vip_level" in df.columns else np.nan
        ind_col2 = "指标" if "指标" in df.columns else None
        if ind_col2:
            daily=df[(df[ind_col2]=="留存率")&df["yyyymmdd"].notna()&df["vip"].notna()]
        else:
            daily=df[df["yyyymmdd"].notna()&df["vip"].notna()]
        tw_d=daily[daily["yyyymmdd"].between(*THIS_WEEK)]; lw_d=daily[daily["yyyymmdd"].between(*LAST_WEEK)]
        def wavg(d,col):
            r={}
            if col not in d.columns: return r
            for v,g in d.groupby("vip"):
                s=g[g[col].notna()]
                if len(s): r[int(v)]=float(np.average(s[col].values,weights=s["n"].values))
            return r
        cols_avail=[c for c in ["1日","2日","3日","7日"] if c in daily.columns]
        res[rt]={"tw":{c:wavg(tw_d,c) for c in cols_avail},
                 "lw":{c:wavg(lw_d,c) for c in cols_avail}}
    return res


def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])

    def _clean(df):
        for col in df.columns:
            for kw in ["充值金额","提款金额","公司输赢","活动奖励","投注金额","充提差"]:
                if kw in col:
                    df[col]=pd.to_numeric(df[col].astype(str).str.replace(",",""),errors="coerce")
        return df
    dc_tw=_clean(dc_tw); dc_lw=_clean(dc_lw)
    dt_tw=_clean(dt_tw); dt_lw=_clean(dt_lw)

    def get_col(df, primary, fallback=None):
        if primary in df.columns: return df[primary]
        if fallback and fallback in df.columns: return df[fallback]
        return pd.Series([0]*len(df))

    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]
    total_c=tw_p["充值金额"].sum(); total_t=tw_p["提现金额"].sum()
    actual_cr=(total_c-total_t)/total_c*100 if total_c>0 else 0
    tot_c=get_col(dc_tw,"充值金额").sum() or 1
    tot_t=get_col(dt_tw,"提款金额").sum() or 1

    dep_t=[]
    for t in [1,10,50,100,200,500]:
        tw_s=dc_tw.head(t); lw_s=dc_lw.head(t)
        tc=get_col(tw_s,"充值金额").sum(); tt=get_col(tw_s,"提款金额").sum()
        lc=get_col(lw_s,"充值金额","对比时段 充值金额").sum()
        lt=get_col(lw_s,"提款金额","对比时段 提款金额").sum()
        win=get_col(tw_s,"公司输赢").sum()
        dep_t.append({"tier":f"Top{t}","tw_chg":tc,"lw_chg":lc,"tw_avg":tc/t,"lw_avg":lc/t,
            "tw_cr":(tc-tt)/tc*100 if tc>0 else 0,"lw_cr":(lc-lt)/lc*100 if lc>0 else 0,
            "tw_win":win,"占全量":tc/tot_c*100 if tot_c>0 else 0})

    wdr_t=[]
    for t in [1,10,50,100,200,500]:
        tw2=dt_tw.head(t); lw2=dt_lw.head(t)
        tt2=get_col(tw2,"提款金额").sum(); tc2=get_col(tw2,"充值金额").sum()
        lt2=get_col(lw2,"提款金额","对比时段 提款金额").sum()
        lc2=get_col(lw2,"充值金额","对比时段 充值金额").sum()
        win2=get_col(tw2,"公司输赢"); wns=(win2<0).sum()
        act_sum=get_col(tw2,"活动奖励").sum()
        act_pct=act_sum/(tc2+act_sum)*100 if (tc2+act_sum)>0 else 0
        excl_c=total_c-tc2; excl_t=total_t-tt2
        excl_cr=(excl_c-excl_t)/excl_c*100 if excl_c>0 else 0
        wdr_t.append({"tier":f"Top{t}","tw_tx":tt2,"lw_tx":lt2,"tw_avg":tt2/t,"lw_avg":lt2/t,
            "tw_cr":(tc2-tt2)/tc2*100 if tc2>0 else 0,"lw_cr":(lc2-lt2)/lc2*100 if lc2>0 else 0,
            "赢家":wns,"总数":t,"赢家率":wns/t*100,"活动占比":act_pct,
            "占全量":tt2/tot_t*100 if tot_t>0 else 0,"大盘影响":excl_cr-actual_cr})
    return dep_t, wdr_t, dc_tw.head(200), dt_tw.head(20)


def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); dl=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean); dl["阶段汇总"]=dl["阶段汇总"].apply(clean)
    mfr_col="show_name_厂商标签id" if "show_name_厂商标签id" in df.columns else df.columns[1]
    game_col="游戏名称" if "游戏名称" in df.columns else df.columns[2]
    ind_col="分析指标" if "分析指标" in df.columns else df.columns[3]
    bet=df[df[ind_col]=="投注金额"]; win=df[df[ind_col]=="公司输赢"]; bl=dl[dl[ind_col]=="投注金额"]
    gb=bet.groupby([mfr_col,game_col])["阶段汇总"].sum().rename("本周投注")
    gw=win.groupby([mfr_col,game_col])["阶段汇总"].sum().rename("公司输赢")
    gl=bl.groupby([mfr_col,game_col])["阶段汇总"].sum().rename("上周投注")
    gu=bet.groupby([mfr_col,game_col])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([gb,gw,gl,gu],axis=1).reset_index()
    g["占比"]=g["本周投注"]/g["本周投注"].sum()*100
    g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr=bet.groupby(mfr_col)["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr


def load_games():
    """Lucro游戏报表有日期列，需按周聚合"""
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        df["日期"]=df["日期"].astype(str)
        for c in ["投注人数","投注局数","投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    tw=tw[tw["日期"].between(*THIS_WEEK)]; lw=lw[lw["日期"].between(*LAST_WEEK)]
    mfr_col="游戏厂商标签.名称" if "游戏厂商标签.名称" in tw.columns else tw.columns[2]
    game_col="游戏.名称" if "游戏.名称" in tw.columns else tw.columns[4]
    tot=tw["投注金额"].sum()
    gt=tw.groupby([mfr_col,game_col]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    gl=lw.groupby(game_col).agg(投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    gt["人均局数"]=gt["投注局数"]/gt["投注人数"]; gt["人均金额"]=gt["投注金额"]/gt["投注人数"]
    gt["盈亏率"]=gt["公司输赢"]/gt["投注金额"]*100; gt["占比"]=gt["投注金额"]/tot*100
    gt["厂商简称"]=gt[mfr_col].apply(shorten_mfr)
    gm=gt.merge(gl,on=game_col,how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)


def load_mfr():
    df=pd.read_csv(FILES["mfr"]); df=df.rename(columns={"盈利率":"盈亏率"})
    time_col="时间" if "时间" in df.columns else df.columns[0]
    df=df[df[time_col]!="阶段汇总"].copy(); df[time_col]=df[time_col].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢","盈亏率"]:
        if c in df.columns: df[c]=df[c].apply(clean)
    tw=df[df[time_col].between(*THIS_WEEK)]; lw=df[df[time_col].between(*LAST_WEEK)]
    mfr_col="show_name_厂商标签id" if "show_name_厂商标签id" in df.columns else df.columns[1]
    def agg(d):
        g=d.groupby(mfr_col).agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]; g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["占比"]=g["投注金额"]/g["投注金额"].sum()*100
        return g
    tg=agg(tw); lg=agg(lw)
    mg=tg.merge(lg[[mfr_col,"投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比","盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on=mfr_col,how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    mg["厂商显示名"]=mg[mfr_col].apply(shorten_mfr)
    mg["show_name_厂商标签id"]=mg[mfr_col]
    return mg.sort_values("投注金额",ascending=False)


def load_mfr_game_delta():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        df["日期"]=df["日期"].astype(str)
        for c in ["投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    tw=tw[tw["日期"].between(*THIS_WEEK)]; lw=lw[lw["日期"].between(*LAST_WEEK)]
    mfr_c="游戏厂商标签.名称" if "游戏厂商标签.名称" in tw.columns else tw.columns[2]
    game_c="游戏.名称" if "游戏.名称" in tw.columns else tw.columns[4]
    gt=tw.groupby([mfr_c,game_c]).agg(投注金额=("投注金额","sum")).reset_index()
    gl=lw.groupby([mfr_c,game_c]).agg(投注金额=("投注金额","sum")).reset_index().rename(columns={"投注金额":"lw_投注"})
    m=gt.merge(gl,on=[mfr_c,game_c],how="outer").fillna(0); m["delta"]=m["投注金额"]-m["lw_投注"]
    mfr_tw=tw.groupby(mfr_c)["投注金额"].sum().sort_values(ascending=False)
    res={}
    for n in mfr_tw.head(8).index:
        sub=m[m[mfr_c]==n].sort_values("delta",ascending=False)
        res[n]={"up":sub.head(1),"dn":sub.tail(1)}
    return res


def load_activities():
    df=pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数","赠送人数.1"]:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0)
    opt_col="账变opt_code" if "账变opt_code" in df.columns else df.columns[0]
    name_col="name_账变opt_id" if "name_账变opt_id" in df.columns else df.columns[1]
    ag=df.groupby([opt_col,name_col]).agg(
        赠送金额=("赠送金额","sum"),lw_赠=("赠送金额.1","sum"),
        赠送人数=("赠送人数","sum"),lw_人数=("赠送人数.1","sum")).reset_index()
    ag["环比"]=(ag["赠送金额"]-ag["lw_赠"])/ag["lw_赠"].replace(0,np.nan).abs()*100
    tot=ag["赠送金额"].sum(); ag["占比"]=ag["赠送金额"]/tot*100
    ag["人均"]=ag["赠送金额"]/ag["赠送人数"].replace(0,np.nan)
    ag["日均_本"]=ag["赠送人数"]/7; ag["日均_上"]=ag["lw_人数"]/7
    ag["人数环比"]=(ag["赠送人数"]-ag["lw_人数"])/ag["lw_人数"].replace(0,np.nan)*100
    ag["name_账变opt_id"]=ag[name_col]
    return ag.sort_values("赠送金额",ascending=False), tot


def load_first_dep_ret():
    """Lucro首充活动留存：列名 1日/2日/…/7日"""
    df=pd.read_csv(FILES["first_dep_ret"])
    df.columns=df.columns.str.strip().str.replace("\ufeff","")
    time_col="初始事件发生时间" if "初始事件发生时间" in df.columns else df.columns[0]
    df["_d"]=df[time_col].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"]=df["_d"].str.replace("-","").fillna("")
    for c in ["当日","1日","2日","3日","4日","5日","6日","7日"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
    user_col="账变事件用户数" if "账变事件用户数" in df.columns else df.columns[1]
    df[user_col]=pd.to_numeric(df[user_col],errors="coerce").fillna(0)
    ind_col="指标" if "指标" in df.columns else None
    stage=df[df[time_col]=="阶段值"].copy() if ind_col else pd.DataFrame()
    daily=df[df["_yyyymmdd"].str.match(r"^\d{8}$",na=False)].copy()
    rr=daily[daily[ind_col]=="留存率"].reset_index(drop=True) if ind_col else daily
    nr=daily[daily[ind_col]=="留存人数"].reset_index(drop=True) if ind_col else daily
    tw_r=rr[rr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_r=rr[rr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    tw_n=nr[nr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_n=nr[nr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    def wa(rd,nd,col):
        if col not in rd.columns or len(rd)==0: return np.nan
        rv=rd[col].to_numpy(dtype=float,na_value=np.nan); ok=~np.isnan(rv)
        if not ok.any(): return np.nan
        w=nd[user_col].to_numpy(dtype=float) if len(nd)==len(rd) else np.ones(len(rv))
        ww=w[ok]; return float(np.nanmean(rv[ok])) if ww.sum()==0 else float(np.average(rv[ok],weights=ww))
    R={"stage":stage,"tw_users":int(tw_n[user_col].sum()),"lw_users":int(lw_n[user_col].sum())}
    for col in ["1日","2日","3日","4日","5日","6日","7日"]:
        R[f"tw_{col}"]=wa(tw_r,tw_n,col); R[f"lw_{col}"]=wa(lw_r,lw_n,col)
    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    R["tw_platform_fc"]=int(dp[dp["日期"].between(*THIS_WEEK)]["首充人数"].sum())
    R["lw_platform_fc"]=int(dp[dp["日期"].between(*LAST_WEEK)]["首充人数"].sum())
    return R


def load_risk(dt_raw, dc_raw):
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    mfr_col="show_name_厂商标签id" if "show_name_厂商标签id" in df.columns else df.columns[1]
    game_col="游戏名称" if "游戏名称" in df.columns else df.columns[2]
    ind_col="分析指标" if "分析指标" in df.columns else df.columns[3]

    ba=df[df[ind_col]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    wa2=df[df[ind_col]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    tg=(df[df[ind_col]=="投注金额"].sort_values("阶段汇总",ascending=False)
        .groupby("账户ID").first()[[mfr_col,game_col,"阶段汇总"]]
        .rename(columns={mfr_col:"主玩厂商",game_col:"主玩游戏","阶段汇总":"主游投注"}).reset_index())
    ug=pd.concat([ba,wa2],axis=1).reset_index()
    ug=ug.merge(tg,on="账户ID",how="left")

    top500=dt_raw.head(500).copy()
    top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0
    top500["投充比"]=top500["投注金额"]/top500["充值金额"].replace(0,np.nan) if "投注金额" in top500.columns else np.nan
    hr=top500[top500["公司输赢"]<-5000].sort_values("公司输赢") if "公司输赢" in top500.columns else pd.DataFrame()

    bg=df[df[ind_col]=="投注金额"].groupby([mfr_col,game_col])["阶段汇总"].sum().rename("投注金额").reset_index()
    wg=df[df[ind_col]=="公司输赢"].groupby([mfr_col,game_col])["阶段汇总"].sum().rename("公司输赢").reset_index()
    ug2=df[df[ind_col]=="投注金额"].groupby([mfr_col,game_col])["账户ID"].nunique().rename("玩家数").reset_index()
    wn=df[df[ind_col]=="公司输赢"].groupby([mfr_col,game_col]).apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index()
    g=bg.merge(wg,on=[mfr_col,game_col],how="left").merge(ug2,on=[mfr_col,game_col],how="left").merge(wn,on=[mfr_col,game_col],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100; g["赢家率"]=g["赢家数"]/g["玩家数"]*100
    g["人均投注额"]=g["投注金额"]/g["玩家数"]
    g["show_name_厂商标签id"]=g[mfr_col]; g["游戏名称"]=g[game_col]
    rg=g[(g["公司输赢"]<-2000)|((g["盈亏率"]<-10)&(g["投注金额"]>2000))].sort_values("公司输赢")
    sp=top500[top500["投充比"]>100].sort_values("提款金额",ascending=False) if "投充比" in top500.columns else pd.DataFrame()
    return hr, sp, g.sort_values("投注金额",ascending=False).head(20), rg, top500


# ════════════════════════════════════════════════════════════════
# 图表
# ════════════════════════════════════════════════════════════════
SPLIT = 7

def _vline(ax):
    ax.axvline(SPLIT-.5, color="#94a3b8", ls="--", lw=1, alpha=.7)
    trans=ax.get_xaxis_transform()
    ax.text(SPLIT-4, 0.93, "上周", ha="center", fontsize=7, color="#64748b", transform=trans)
    ax.text(SPLIT+3, 0.93, "本周", ha="center", fontsize=7, color="#ea580c", transform=trans)

def chart_trend(trend):
    fig=plt.figure(figsize=(16,10),facecolor="white")
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=.45,wspace=.3)
    dates=trend["dates"]; x=range(len(dates))
    def sp(a): a.spines["top"].set_visible(False); a.spines["right"].set_visible(False); a.grid(axis="y",alpha=.25)
    ax=fig.add_subplot(gs[0,0])
    ax.bar(x,[v/10000 for v in trend["充值"]],color=["#fed7aa"]*SPLIT+["#ea580c"]*SPLIT,width=.7,label="充值")
    ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
    ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold")
    ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
    ax.legend(fontsize=7,loc="upper left"); sp(ax); _vline(ax)
    ax2=fig.add_subplot(gs[0,1])
    ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=.7)
    ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold")
    ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7); sp(ax2); _vline(ax2)
    ax3=fig.add_subplot(gs[1,0])
    ax3.bar(x,trend["首充"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=.7,label="首充人数")
    ax3r=ax3.twinx()
    ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold")
    ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
    l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left")
    ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=.25)
    ax3.axvline(SPLIT-.5,color="#94a3b8",ls="--",lw=1,alpha=.7)
    ax4=fig.add_subplot(gs[1,1])
    ax4.bar(x,trend["充提差比"],color=["#fde68a"]*SPLIT+["#d97706"]*SPLIT,width=.7)
    tm_=np.mean(trend["充提差比"][SPLIT:]); lm_=np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tm_,color="#d97706",ls=":",lw=1.5); ax4.axhline(lm_,color="#94a3b8",ls=":",lw=1.2)
    _bbox=dict(boxstyle="round,pad=0.15",fc="white",ec="none",alpha=0.85)
    n=len(dates)
    ax4.text(n-1,tm_+.3,f"本周均{tm_:.1f}%",fontsize=7,color="#d97706",ha="right",clip_on=False,fontweight="bold",bbox=_bbox)
    ax4.text(0,lm_+.3,f"上周均{lm_:.1f}%",fontsize=7,color="#64748b",ha="left",clip_on=False,fontweight="bold",bbox=_bbox)
    ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold")
    ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7)
    ymax=max(trend["充提差比"]); ax4.set_ylim(0,max(ymax*1.20,5)); sp(ax4); _vline(ax4)
    fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
    plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,11)

def chart_ret_weekly(weeks):
    # Lucro列名：1日/2日/6日
    fig,ax=plt.subplots(figsize=(14,6),facecolor="white")
    cols=[("1日","次留","#1d4ed8"),("2日","3留","#059669"),("6日","7留","#7c3aed")]
    x=np.arange(len(weeks)); w=.25
    for i,(col,lbl,clr) in enumerate(cols):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-w,vals,w,label=lbl,color=clr,alpha=.85)
        for bar,v in zip(bars,vals):
            if not(isinstance(v,float) and np.isnan(v)):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
        tgt=RET_TARGETS.get(lbl)
        if tgt: ax.axhline(tgt,color=clr,ls="--",lw=1.2,alpha=.55)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周对比（含目标虚线）",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8,loc="upper left"); ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=.5)
    ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,6)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tc=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lc=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tc,w,label="本周",color="#ea580c",alpha=.85); ax1.bar(x+w/2,lc,w,label="上周",color="#fed7aa",alpha=.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold")
    ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7)
    ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    tb=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lb=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tb,w,label="本周",color="#059669",alpha=.85); ax2.bar(x+w/2,lb,w,label="上周",color="#6ee7b7",alpha=.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7)
    ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100
         if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=.85,width=.6)
    ax3.axhline(0,color="black",lw=.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold")
    ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7)
    ax3.grid(axis="y",alpha=.3); ax3.spines["top"].set_visible(False); ax3.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_retention(vr):
    vips=[1,3,4,5,6,7,8,9,10,11,12]; labels=[f"V{v}" for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white"); x=np.arange(len(labels)); w=.28
    for ax,rt,t in [(ax1,"chg","充值→充值 留存率（%）"),(ax2,"act","充值→活跃 留存率（%）")]:
        cols_avail=list(vr[rt]["tw"].keys())
        c1="1日" if "1日" in cols_avail else (cols_avail[0] if cols_avail else "1日")
        c3="3日" if "3日" in cols_avail else (cols_avail[min(2,len(cols_avail)-1)] if cols_avail else "3日")
        tw1=[vr[rt]["tw"].get(c1,{}).get(v,0) if isinstance(vr[rt]["tw"].get(c1),dict) else 0 for v in vips]
        lw1=[vr[rt]["lw"].get(c1,{}).get(v,0) if isinstance(vr[rt]["lw"].get(c1),dict) else 0 for v in vips]
        tw3=[vr[rt]["tw"].get(c3,{}).get(v,0) if isinstance(vr[rt]["tw"].get(c3),dict) else 0 for v in vips]
        clr=("#1d4ed8","#93c5fd","#059669") if rt=="chg" else ("#7c3aed","#c4b5fd","#d97706")
        ax.bar(x-w,tw1,w,label="次日(本周)",color=clr[0],alpha=.85)
        ax.bar(x,lw1,w,label="次日(上周)",color=clr[1],alpha=.7)
        ax.bar(x+w,tw3,w,label="3日(本周)",color=clr[2],alpha=.75)
        ax.set_title(t,fontsize=10,fontweight="bold"); ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8)
        ax.legend(fontsize=7.5,loc="upper left"); ax.grid(axis="y",alpha=.25)
        ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.suptitle("VIP各等级充值留存率",fontsize=11,fontweight="bold",y=1.02)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10)
    names=top10["厂商显示名"].tolist() if "厂商显示名" in top10.columns else top10.iloc[:,0].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    sh=top10["占比"].tolist(); ls=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,sh,color=["#059669" if c>=l else "#dc2626" for c,l in zip(sh,ls)],alpha=.85,width=.6)
    ax1.plot(names,ls,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,sh,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.2,f"{s:.1f}%",ha="center",fontsize=7)
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),
               plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    ax1.grid(axis="y",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    x=np.arange(len(names)); w=.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=.85)
    ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=.7)
    ax2.axhline(0,color="black",lw=.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5)
    ax2.legend(fontsize=7.5); ax2.grid(axis="y",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    game_col="游戏.名称" if "游戏.名称" in top30.columns else top30.columns[1]
    t15=top30.head(15); names=[str(r[game_col])[:18] for _,r in t15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in t15.iterrows()]
    lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in t15.iterrows()]
    x=np.arange(len(names)); w=.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#ea580c",alpha=.85)
    ax.barh(x-w/2,lw_b,w,label="上周",color="#fed7aa",alpha=.7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8)
    ax.set_title("Top15游戏 投注金额（万USD）",fontsize=9,fontweight="bold")
    ax.legend(fontsize=8); ax.grid(axis="x",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pg,ma):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=ma.index[:8].tolist(); vals=ma.values[:8].tolist(); tot=sum(vals)
    pcts=[v/tot*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",
                          startangle=90,pctdistance=.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{str(n)[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-.05,-.18))
    ax1.set_title("Top500提款用户 厂商偏好",fontsize=9,fontweight="bold")
    game_col="游戏名称" if "游戏名称" in pg.columns else pg.columns[1]
    mfr_col="show_name_厂商标签id" if "show_name_厂商标签id" in pg.columns else pg.columns[0]
    t12=pg.head(12); gn=[f"[{str(r[mfr_col])[:4]}]\n{str(r[game_col])}"[:22] for _,r in t12.iterrows()]
    gv=[r["本周投注"]/10000 for _,r in t12.iterrows()]
    gc=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in t12.iterrows()]
    ax2.barh(range(len(gn))[::-1],gv,color=gc[::-1],alpha=.85)
    ax2.set_yticks(range(len(gn))); ax2.set_yticklabels(gn,fontsize=7.5)
    ax2.set_title("偏好游戏Top12（红=平台亏损）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.1f}万"))
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act):
    name_col="name_账变opt_id" if "name_账变opt_id" in act.columns else act.columns[1]
    t10=act.head(10); names=[str(r[name_col])[:12] for _,r in t10.iterrows()]
    vals=[r["赠送金额"]/10000 for _,r in t10.iterrows()]
    lw=[r["lw_赠"]/10000 for _,r in t10.iterrows()]
    envs=[r["环比"] for _,r in t10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white"); x=np.arange(len(names)); w=.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#ea580c",alpha=.85)
    ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#fed7aa",alpha=.7)
    ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8)
    ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold")
    ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    es=[v if not pd.isna(v) else 0 for v in envs]
    ax2.barh(range(len(names)),es[::-1],color=["#059669" if v>0 else "#dc2626" for v in es[::-1]],alpha=.85)
    ax2.axvline(0,color="black",lw=.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8)
    ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:+.0f}%"))
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_first_dep_ret(R):
    lv=[R.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    tv=[R.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white"); x=np.arange(7)
    ax.plot(x,lv,"-o",color="#fed7aa",lw=2,ms=6,label=f"上周（{LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}）")
    ax.plot(x,tv,"-o",color="#ea580c",lw=2,ms=6,label=f"本周（{THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}）")
    days=["D1","D2","D3","D4","D5","D6","D7"]
    for i,(lval,tval) in enumerate(zip(lv,tv)):
        if lval is not None and not(isinstance(lval,float) and np.isnan(lval)): ax.text(i,lval+.4,f"{lval:.1f}%",ha="center",fontsize=7.5,color="#64748b")
        if tval is not None and not(isinstance(tval,float) and np.isnan(tval)): ax.text(i,tval-1.2,f"{tval:.1f}%",ha="center",fontsize=7.5,color="#ea580c",fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(days,fontsize=9); ax.set_title("首次充值活动用户 充值留存趋势",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8.5,loc="upper right"); ax.grid(axis="y",alpha=.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%"))
    plt.tight_layout(); return fig_img(fig,13,5.5)

def chart_risk_scatter(top500):
    v=top500[top500["投充比"].notna()].copy() if "投充比" in top500.columns else top500.head(0)
    v=v[v["投充比"]<200]
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    if len(v)>0:
        ax.scatter(v["投充比"],v["公司输赢"]/10000,
                   c=["#dc2626" if x<0 else "#059669" for x in v["公司输赢"]],
                   s=[min(abs(x)/200+20,200) for x in v["公司输赢"]],alpha=.55,edgecolors="none")
    ax.axhline(0,color="black",lw=.8,ls="--"); ax.axvline(20,color="#d97706",lw=1,ls=":",alpha=.7)
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（万USD）",fontsize=8)
    ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=.6,label="平台输钱"),
                       mpatches.Patch(color="#059669",alpha=.6,label="平台赢钱")],fontsize=8,loc="upper right")
    ax.grid(alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(rg):
    t12=rg.head(12)
    game_col="游戏名称" if "游戏名称" in t12.columns else t12.columns[1]
    names=[str(r[game_col])[:16] for _,r in t12.iterrows()]
    losses=[abs(r["公司输赢"]) for _,r in t12.iterrows()]
    rates=[r["盈亏率"] for _,r in t12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=.85)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold")
    ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"${v/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=.85)
    ax2.axvline(0,color="black",lw=.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold")
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_:f"{v:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)


# ════════════════════════════════════════════════════════════════
# 章节构建
# ════════════════════════════════════════════════════════════════
def build_overview(K, trend, weekly_ret, dash_ret):
    full=PW-2*MARGIN; S=[sec_title("一、大盘核心数据")]
    kpis=[
        ("充值金额",   f"{K['tw_充值金额']/10000:.1f}万",   f"{K['lw_充值金额']/10000:.1f}万",   K["pct_充值金额"],   True),
        ("提现金额",   f"{K['tw_提现金额']/10000:.1f}万",   f"{K['lw_提现金额']/10000:.1f}万",   K["pct_提现金额"],   False),
        ("充提差",     f"{K['tw_充提差']/10000:.1f}万",     f"{K['lw_充提差']/10000:.1f}万",     K["pct_充提差"],     True),
        ("充提差率",   f"{K['tw_充提差比']:.2f}%",          f"{K['lw_充提差比']:.2f}%",          K["pct_充提差比"],   True),
        ("公司输赢",   f"{K['tw_公司输赢']/10000:.1f}万",   f"{K['lw_公司输赢']/10000:.1f}万",   K["pct_公司输赢"],   True),
        ("盈亏率",     f"{K['tw_盈亏率']:.3f}%",            f"{K['lw_盈亏率']:.3f}%",            K["pct_盈亏率"],     True),
        ("注册人数",   f"{int(K['tw_注册人数']):,}",        f"{int(K['lw_注册人数']):,}",        K["pct_注册人数"],   True),
        ("首充人数",   f"{int(K['tw_首充人数']):,}",        f"{int(K['lw_首充人数']):,}",        K["pct_首充人数"],   True),
        ("日均活跃",   f"{K['tw_活跃人数']/7/10000:.2f}万", f"{K['lw_活跃人数']/7/10000:.2f}万", K["pct_活跃人数"],   True),
        ("投注金额",   f"{K['tw_投注金额']/10000:.1f}万",   f"{K['lw_投注金额']/10000:.1f}万",   K["pct_投注金额"],   True),
        ("全量ARPPU",  f"${K['tw_全量Arppu']:.2f}",         f"${K['lw_全量Arppu']:.2f}",         K["pct_全量Arppu"],  True),
        ("老用户ARPPU",f"${K['tw_老用户ARPPU']:.2f}",       f"${K['lw_老用户ARPPU']:.2f}",       K["pct_老用户ARPPU"],True),
        ("首充ARPPU",  f"${K['tw_首充Arppu']:.2f}",         f"${K['lw_首充Arppu']:.2f}",         K["pct_首充Arppu"],  True),
        ("总赠送金额", f"{K['tw_总赠送金额']/10000:.2f}万", f"{K['lw_总赠送金额']/10000:.2f}万", K["pct_总赠送金额"], False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",        f"{K['lw_赠送充值比']:.2f}%",        K["pct_赠送充值比"], False),
        ("首充转化率", f"{K['tw_首充转化率']:.1f}%",        f"{K['lw_首充转化率']:.1f}%",        K["pct_首充转化率"], True),
        ("首充次日留存",f"{K['tw_首充次日复充率']:.1f}%",   f"{K['lw_首充次日复充率']:.1f}%",   K["pct_首充次日复充率"],True),
        ("推广消耗(日均)",f"{K['tw_真实消耗_日均']/10000:.2f}万",
                          f"{K['lw_真实消耗_日均']/10000:.2f}万",K["pct_真实消耗"],False),
        ("充提差ROI\n(日均充提差/消耗)",
         f"{K['tw_充提差ROI']:.2f}x",f"{K['lw_充提差ROI']:.2f}x",K["pct_充提差ROI"],True),
    ]
    S.append(kpi_card4(kpis,cols=4)); S.append(Spacer(1,8))
    cr_d=K["tw_充提差比"]-K["lw_充提差比"]; rd=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(insight_box([
        f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%）；推广日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（本周{K['tw_真实消耗_有效天']}天有效，{K['pct_真实消耗']:+.1f}%）。",
        f"充提差率{K['tw_充提差比']:.2f}%（上周{K['lw_充提差比']:.2f}%，{cr_d:+.2f}pp）；盈亏率{K['tw_盈亏率']:.3f}%（上周{K['lw_盈亏率']:.3f}%）。",
        f"首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），转化率{K['tw_首充转化率']:.1f}%，首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%）。",
        f"首充次日充值留存{K['tw_首充次日复充率']:.1f}%（上周{K['lw_首充次日复充率']:.1f}%，{rd:+.1f}pp）。",
    ]))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
    S.append(sub_title("大盘首充留存 vs 目标对比（来源：日报 首充2/3/7日复充率）"))
    specs=[("次留","tw_nd","lw_nd","tw_nd_end"),("3留","tw_td","lw_td","tw_td_end"),("7留","tw_sd","lw_sd","tw_sd_end")]
    ret_headers=["留存类型","数据截止","本周实际","上周实际","周环比(pp)","目标","vs目标(pp)"]
    ret_rows=[]
    for rtype,tw_k,lw_k,end_k in specs:
        tv=dash_ret.get(tw_k,np.nan); lv=dash_ret.get(lw_k,np.nan)
        end_lbl=fmt_lbl(dash_ret.get(end_k)); tgt=RET_TARGETS.get(rtype,np.nan)
        wpp=(tv-lv) if not(np.isnan(tv) or np.isnan(lv)) else np.nan
        dpp=(tv-tgt) if not(np.isnan(tv) or np.isnan(tgt)) else np.nan
        tw_clr=C_GREEN if(not np.isnan(tv) and not np.isnan(tgt) and tv>=tgt) else C_RED
        ret_rows.append([
            cell(rtype,True,C_DARK), cell(end_lbl,False,C_GRAY,TA_CENTER),
            cell(fret(tv),False,tw_clr,TA_RIGHT), cell(fret(lv),False,C_GRAY,TA_RIGHT),
            cell(f"{wpp:+.1f}pp" if not np.isnan(wpp) else "-",False,C_GREEN if(not np.isnan(wpp) and wpp>=0) else C_RED,TA_RIGHT),
            cell(f"{tgt:.0f}%" if not np.isnan(tgt) else "-",True,C_TARGET,TA_CENTER),
            cell(f"{dpp:+.1f}pp" if not np.isnan(dpp) else "-",True,C_GREEN if(not np.isnan(dpp) and dpp>=0) else C_RED,TA_RIGHT),
        ])
    S.append(dtable(ret_headers,ret_rows,[full*x for x in [0.16,0.12,0.14,0.14,0.14,0.12,0.14]],fsize=8,
                    extra_style=[("BACKGROUND",(5,1),(5,-1),colors.HexColor("#fff7ed"))]))
    nd_end=fmt_lbl(dash_ret.get("tw_nd_end")); td_end=fmt_lbl(dash_ret.get("tw_td_end")); sd_end=fmt_lbl(dash_ret.get("tw_sd_end"))
    S.append(P(f"本周截止：次留~{nd_end}，3留~{td_end}，7留~{sd_end}",7,False,C_GRAY)); S.append(Spacer(1,8))
    S.append(sub_title("首充用户充值留存 — 近4周对比（含目标线）"))
    S.append(chart_ret_weekly(weekly_ret)); S.append(Spacer(1,4))
    this_w=weekly_ret[-1]; last_w=weekly_ret[-2]
    d1=this_w.get("1日",0) or 0; d2=this_w.get("2日",0) or 0; d6=this_w.get("6日",0) or 0
    ld1=last_w.get("1日",0) or 0; ld6=last_w.get("6日",0) or 0
    S.append(insight_box([
        f"本周首充次日留存{d1:.1f}%，较上周{d1-ld1:+.1f}pp；3留{d2:.1f}%，7留{d6:.1f}%。",
        f"注：本周7日留存因截止日期不完整，以上周7留{ld6:.1f}%作为参考基准。",
    ], clr=C_AMBER))
    return S


def build_agents(agents, agent_ret, ends):
    full=PW-2*MARGIN; S=[sec_title("二、总代分析")]
    tw_nd_lbl=fmt_lbl(ends.get("tw_nd_end")); lw_nd_lbl=fmt_lbl(ends.get("lw_nd_end"))
    tw_3d_lbl=fmt_lbl(ends.get("tw_3d_end")); lw_3d_lbl=fmt_lbl(ends.get("lw_3d_end"))
    lw_7d_lbl=fmt_lbl(ends.get("lw_7d_end"))
    S.append(sub_title("全量总代表现（本周 vs 上周，按充值金额排序）"))
    headers=["ID","总代名称","注册(环比)","首充\n人数","充值\n(万)","充提差率\n(差值pp)","消耗\n(万)","1级首充\n成本(本/上)",
             f"次留(本/上)\n~{tw_nd_lbl}/{lw_nd_lbl}",f"3留(本/上)\n~{tw_3d_lbl}/{lw_3d_lbl}",f"7留上周\n~{lw_7d_lbl}"]
    rows=[]
    id_col="总代.ID" if "总代.ID" in agents.columns else agents.columns[0]
    name_col="总代.名称" if "总代.名称" in agents.columns else agents.columns[1]
    for _,r in agents.iterrows():
        aid=int(r[id_col]) if not pd.isna(r.get(id_col,np.nan)) else "-"
        rname=str(r[name_col])
        cr=r.get("充提差率",0) or 0; lw_cr=r.get("lw_充提差率",0) or 0; cr_d=cr-lw_cr
        cost=(r.get("总消耗",0) or 0)/10000
        fcc=r.get("一级首充成本",0) or 0; lfc=r.get("lw_fc_cost",0) or 0
        reg=int(r.get("注册人数",0) or 0); reg_c=r.get("注册环比",0) or 0
        cr_clr=C_GREEN if cr>=CR_HIGH else (C_RED if cr<CR_LOW else C_DARK)
        d=agent_ret.get(int(aid) if aid!="-" else -1, {})
        tw_nd=d.get("tw_nd",np.nan); lw_nd=d.get("lw_nd",np.nan)
        tw_3d=d.get("tw_3d",np.nan); lw_3d=d.get("lw_3d",np.nan)
        lw_7d=d.get("lw_7d",np.nan)
        def _fmt_pair(tv,lv):
            if not np.isnan(tv) and not np.isnan(lv): return f"{tv:.1f}%/{lv:.1f}%",C_GREEN if tv>=lv else C_RED
            if not np.isnan(tv): return f"{tv:.1f}%/-",C_DARK
            return "-",C_GRAY
        nd_s,nd_clr=_fmt_pair(tw_nd,lw_nd); td_s,td_clr=_fmt_pair(tw_3d,lw_3d)
        rows.append([
            cell(str(aid),False,C_GRAY,TA_CENTER), cell(rname[:14],True,C_DARK,TA_LEFT),
            cell(f"{reg:,}/{'+' if reg_c>=0 else ''}{reg_c:.0f}%",False,C_GREEN if reg_c>=0 else C_RED,TA_RIGHT),
            cell(f"{int(r.get('首充人数',0)):,}",False,C_DARK,TA_RIGHT),
            cell(f"{r.get('充值金额',0)/10000:.1f}",False,C_DARK,TA_RIGHT),
            cell(f"{cr:.1f}%/{'+' if cr_d>=0 else ''}{cr_d:.1f}pp",False,cr_clr,TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-",False,C_DARK,TA_RIGHT),
            cell(f"${fcc:.0f}/${lfc:.0f}" if fcc>0 else "-",False,C_DARK,TA_RIGHT),
            cell(nd_s,False,nd_clr,TA_RIGHT), cell(td_s,False,td_clr,TA_RIGHT),
            cell(fret(lw_7d),False,C_DARK,TA_RIGHT),
        ])
    cw=[full*x for x in [0.04,0.14,0.10,0.06,0.06,0.11,0.05,0.10,0.11,0.11,0.08]]
    S.append(dtable(headers,rows,cw,fsize=6.2))
    S.append(Spacer(1,4))
    S.append(P(f"★ 充提差率≥{CR_HIGH}%绿，<{CR_LOW}%红。",6.5,False,C_GRAY)); S.append(Spacer(1,6))
    ins=[]
    if len(agents)>0:
        t1=agents.iloc[0]; t1_cr_d=t1.get("充提差率",0)-(t1.get("lw_充提差率",0) or 0)
        ins.append(f"体量最大总代「{str(t1[name_col])[:16]}」充值{t1.get('充值金额',0)/10000:.1f}万，充提差率{t1.get('充提差率',0):.1f}%（{t1_cr_d:+.1f}pp），注册{int(t1.get('注册人数',0)):,}人（{t1.get('注册环比',0):+.0f}%）。")
    if "充提差率" in agents.columns:
        best=agents[agents["充值金额"]>1000].nlargest(1,"充提差率") if "充值金额" in agents.columns else pd.DataFrame()
        if len(best)>0:
            b=best.iloc[0]; ins.append(f"充提差率最优渠道「{str(b[name_col])[:16]}」达{b.get('充提差率',0):.1f}%，充值规模{b.get('充值金额',0)/10000:.1f}万。")
    if ins: S.append(insight_box(ins))
    return S


def build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_tw):
    full=PW-2*MARGIN; S=[sec_title("三、用户分析")]
    S.append(sub_title("VIP等级分层分析")); S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
    tot=tw_v["充值金额"].sum()
    hvs=sum(tw_v.loc[v,"充值金额"] for v in [9,10,11] if v in tw_v.index)
    ins_vip=[f"VIP9-11高价值层合计贡献充值{hvs/tot*100:.1f}%，高端用户付费意愿{'强劲' if hvs/tot>0.3 else '有待提升'}。"]
    if 1 in tw_v.index and 1 in lw_v.index:
        ins_vip.append(f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。")
    S.append(insight_box(ins_vip))
    vr=load_vip_retention()
    S.append(sub_title("VIP各等级充值留存率")); S.append(chart_vip_retention(vr)); S.append(Spacer(1,4))
    v9_act=vr.get("act",{}).get("tw",{}).get("1日",{}).get(9,0)
    v10_act=vr.get("act",{}).get("tw",{}).get("1日",{}).get(10,0)
    v10_chg_tw=vr.get("chg",{}).get("tw",{}).get("1日",{}).get(10,0)
    v10_chg_lw=vr.get("chg",{}).get("lw",{}).get("1日",{}).get(10,0)
    S.append(insight_box([
        f"充值→活跃次日留存：VIP9达{v9_act:.1f}%，VIP10达{v10_act:.1f}%。",
        f"充值→充值次日留存：VIP10达{v10_chg_tw:.1f}%（上周{v10_chg_lw:.1f}%）。",
    ])); S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h=["分层","本周充值","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),
                     cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_win']/10000:.2f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    S.append(insight_box([
        f"Top10充值用户人均${dep_t[1]['tw_avg']:,.0f}（{pct(dep_t[1]['tw_avg'],dep_t[1]['lw_avg']):+.1f}%），充提差率{dep_t[1]['tw_cr']:.1f}%。",
        "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。"
    ]))
    S.append(sub_title("头部提款用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","赢家比例","活动占比","占全量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdr_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                      cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                      rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),
                      cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),
                      cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
                      cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),
                      cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(h3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(insight_box([
        f"剔除Top100提款用户后，大盘充提差率影响{wdr_t[3]['大盘影响']:+.2f}pp，头部提款用户对充提差率有明显拖累。",
        f"Top500提款用户活动奖励占资金来源仅{wdr_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。",
    ], clr=C_AMBER))
    return S


def build_games(mfr, top30, delta):
    full=PW-2*MARGIN; S=[sec_title("四、游戏分析")]
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比")); S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    h=["排名","厂商","日均投注人数(本/上)","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),
                     cell(str(r.get("厂商显示名",r["show_name_厂商标签id"])),True),
                     cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),
                     rc(r.get("投注环比",0)),
                     cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
                     cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
                     cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.05,0.12,0.13,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5)); S.append(Spacer(1,4))
    top1=mfr.iloc[0]
    S.append(insight_box([
        f"{top1.get('厂商显示名',top1['show_name_厂商标签id'])}投注份额{top1['占比']:.1f}%（上周{top1.get('lw_占比',0):.1f}%），环比{top1.get('投注环比',0):+.1f}%，为本周最大流量厂商。",
        "盈亏率高于4%的厂商可适当扩大曝光权重；持续偏低厂商建议复核RTP配置。",
    ]))
    if delta:
        S.append(sub_title("▶ 厂商投注额环比主要驱动游戏"))
        dh=["厂商","投注额环比","增量最大游戏(+贡献)","降量最大游戏(-拖累)"]; dr=[]
        for mn,gd in delta.items():
            mrow=mfr[mfr["show_name_厂商标签id"]==mn]; mc=mrow["投注环比"].values[0] if len(mrow)>0 else 0
            game_cn="游戏.名称" if "游戏.名称" in gd["up"].columns else gd["up"].columns[1]
            up=gd["up"]; dn=gd["dn"]
            us=f'{up.iloc[0][game_cn][:16]}（+${up.iloc[0]["delta"]/10000:.2f}万）' if len(up)>0 and up.iloc[0]["delta"]>0 else "-"
            ds=f'{dn.iloc[0][game_cn][:16]}（${dn.iloc[0]["delta"]/10000:.2f}万）' if len(dn)>0 and dn.iloc[0]["delta"]<0 else "-"
            dr.append([cell(shorten_mfr(mn),True,C_DARK),rc(mc),
                       cell(us,False,C_GREEN if us!="-" else C_GRAY),
                       cell(ds,False,C_RED if ds!="-" else C_GRAY)])
        S.append(dtable(dh,dr,[full*x for x in [0.15,0.10,0.37,0.38]],fsize=6.8)); S.append(Spacer(1,6))
    S.append(sub_title("Top30游戏详细数据")); S.append(chart_top30(top30)); S.append(Spacer(1,4))
    game_col="游戏.名称" if "游戏.名称" in top30.columns else top30.columns[1]
    h2=["#","游戏名称","厂商简称","投注人数","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r[game_col])[:18],True),
                      cell(str(r.get("厂商简称",""))[:5]),
                      cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),
                      cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['投注金额']/10000:.2f}",False,C_DARK,TA_RIGHT),
                      rc(r.get("投注环比",0)),
                      cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
                      cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h2,rows2,[full*x for x in [0.04,0.18,0.07,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S


def build_activities(act, tot_gift):
    full=PW-2*MARGIN; S=[sec_title("五、活动分析")]
    S.append(sub_title("各活动赠送效果（全量，含环比）")); S.append(chart_activities(act)); S.append(Spacer(1,4))
    h=["活动名称","本周赠送","上周赠送","金额环比","本周日均\n赠送人数","上周日均\n赠送人数","人数环比","本周人均\n赠送金额","占比"]
    rows=[]
    for _,r in act.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:18],True),
                     cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"${r['lw_赠']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r["环比"] if not pd.isna(r.get("环比",np.nan)) else 0),
                     cell(f"{r.get('日均_本',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r.get('日均_上',0):.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r.get("人数环比",0) if not pd.isna(r.get("人数环比",np.nan)) else 0),
                     cell(f"${r.get('人均',0):.2f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.20,0.11,0.11,0.07,0.12,0.12,0.07,0.11,0.09]],fsize=6.5))
    S.append(Spacer(1,4)); daily=tot_gift/7/10000
    S.append(P(f"本周总赠送金额：${tot_gift/10000:.2f}万 | 日均赠送：${daily:.2f}万",9,True,C_DARK)); S.append(Spacer(1,6))
    S.append(insight_box([f"各类赠送活动全量列出（共{len(act)}个），合计本周日均赠送{daily:.2f}万USD。"]))
    return S


def build_first_dep_ret(fdr):
    full=PW-2*MARGIN; S=[sec_title("六、首充活动用户留存专题", clr=C_TEAL)]
    S.append(sub_title("6.1 首次充值活动用户 充值留存分析（重点）"))
    stage=fdr.get("stage",pd.DataFrame())
    if len(stage)>0 and "指标" in stage.columns:
        lws=LAST_WEEK[0]
        S.append(P(f"两周阶段汇总（{lws[:4]}.{lws[4:6]}.{lws[6:]} - {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}）",9,True,C_DARK)); S.append(Spacer(1,3))
        sr=stage[stage["指标"]=="留存率"]; sn=stage[stage["指标"]=="留存人数"]
        if len(sr)>0:
            user_col="账变事件用户数" if "账变事件用户数" in sn.columns else sn.columns[1]
            u=int(sn[user_col].values[0]) if len(sn)>0 else 0
            S.append(P(f"首充活动用户总数：{u:,}人",8.5,False,C_DARK))
            dc=["当日","1日","2日","3日","4日","5日","6日","7日"]
            dc_avail=[c for c in dc if c in sr.columns]
            srows=[]
            if len(sr)>0: srows.append(["留存率"]+[str(sr.iloc[0].get(c,"-")) for c in dc_avail])
            if len(sn)>0: srows.append(["留存人数"]+[str(sn.iloc[0].get(c,"-")) for c in dc_avail])
            if srows:
                tr2=[[cell(row[0],True,C_DARK)]+[cell(str(v),False,C_DARK,TA_CENTER) for v in row[1:]] for row in srows]
                S.append(dtable(["指标"]+dc_avail,tr2,[full*.12]+[full*(0.88/len(dc_avail))]*len(dc_avail),fsize=6.8))
        S.append(Spacer(1,6))
    S.append(P("本周 vs 上周 首充活动用户留存趋势",9,True,C_DARK)); S.append(Spacer(1,3))
    S.append(chart_first_dep_ret(fdr)); S.append(Spacer(1,4))
    tw_u=fdr.get("tw_users",0); lw_u=fdr.get("lw_users",0)
    tw_pf=fdr.get("tw_platform_fc",0); lw_pf=fdr.get("lw_platform_fc",0)
    tw_r=tw_u/tw_pf*100 if tw_pf>0 else 0; lw_r=lw_u/lw_pf*100 if lw_pf>0 else 0
    tv=[fdr.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    lv=[fdr.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    def _fr(v): return f"{v:.1f}%" if not(v is None or (isinstance(v,float) and np.isnan(v))) else "-"
    rh=["周次","活动用户数","平台首充\n人数","活动用户\n占比","D1留存","D2留存","D3留存","D4留存","D5留存","D6留存","D7留存"]
    rrows=[
        [cell(f"本周 {THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}",True,C_ORANGE),
         cell(f"{tw_u:,}",False,C_DARK,TA_RIGHT),cell(f"{tw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{tw_r:.1f}%",False,C_PURPLE,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GREEN if not pd.isna(v) and not pd.isna(lval) and v>=lval else(C_RED if not pd.isna(v) and not pd.isna(lval) and v<lval else C_DARK),TA_RIGHT) for v,lval in zip(tv,lv)],
        [cell(f"上周 {LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}",True,C_GRAY),
         cell(f"{lw_u:,}",False,C_GRAY,TA_RIGHT),cell(f"{lw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{lw_r:.1f}%",False,C_GRAY,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GRAY,TA_RIGHT) for v in lv],
    ]
    S.append(dtable(rh,rrows,[full*.16,full*.09,full*.09,full*.08]+[full*.084]*7,fsize=7,zebra=False)); S.append(Spacer(1,4))
    tw_d1=fdr.get("tw_1日",np.nan); lw_d1=fdr.get("lw_1日",np.nan); ins=[]
    if not np.isnan(tw_d1) and not np.isnan(lw_d1):
        d=tw_d1-lw_d1; ins.append(f"首充活动用户次日留存{tw_d1:.1f}%（上周{lw_d1:.1f}%，{d:+.1f}pp），{'留存改善' if d>0 else '留存下降，建议优化次日触达策略'}。")
    ins.append(f"本周活动用户{tw_u:,}人，占平台首充{tw_r:.1f}%（上周{lw_u:,}/{lw_r:.1f}%）；属高价值用户来源，建议持续投入。")
    ins.append("建议：对D1/D2留存用户设置阶梯式再充值激励；对D3后流失用户做专项召回（24h内触达效果最佳）。")
    S.append(insight_box(ins, clr=C_AMBER))
    return S


def build_risk(hr, sp, top20g, rg, top500, dt_full):
    full=PW-2*MARGIN; S=[sec_title("七、用户游戏风险专项分析", clr=colors.HexColor("#7c2d12"))]
    tot_tx=top500["提款金额"].sum() if "提款金额" in top500.columns else 0
    tot_win=top500["公司输赢"].sum() if "公司输赢" in top500.columns else 0
    wns=(top500["公司输赢"]<0).sum() if "公司输赢" in top500.columns else 0
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${tot_tx/10000:.2f}万",""),
                           ("平台净赔付",f"${abs(tot_win)/10000:.2f}万","平台向该群体净赔"),
                           ("赢家比例",f"{wns/500*100:.1f}%",f"{wns}赢/{500-wns}输"),
                           ("高风险用户",f"{len(hr)}人","公司净输>$5,000")]:
        fw=full/4-4
        row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],
                           colWidths=[fw],style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
                           ("BOX",(0,0),(-1,-1),0.5,C_BORDER),("LEFTPADDING",(0,0),(-1,-1),8),
                           ("TOPPADDING",(0,0),(-1,-1),5),("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),
                            ("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）")); S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    hl=abs(hr["公司输赢"].sum())/10000 if len(hr)>0 else 0
    S.append(insight_box([
        f"Top500提款用户中赢家{wns}人（{wns/500*100:.1f}%），平台净赔付${abs(tot_win)/10000:.2f}万。",
        f"{len(hr)}名高风险用户（公司净输>$5,000）合计导致平台净输${hl:.2f}万。",
    ]))
    if len(hr)>0:
        S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
        h=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","主玩厂商","主玩游戏","风险标签"]
        rows=[]
        for _,r in hr.iterrows():
            ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0
            tags=[]
            if r.get("充值金额",0)<2000: tags.append("低充高提")
            if ratio>50: tags.append("超高投充")
            rows.append([cell(str(r["账户ID"]),True),cell(f"${r.get('提款金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"${r.get('充值金额',0):,.0f}",False,C_RED if r.get("充值金额",0)<2000 else C_DARK,TA_RIGHT),
                         cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED,TA_RIGHT),
                         cell(f"${r.get('投注金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),
                         cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),
                         cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
        S.append(dtable(h,rows,[full*x for x in [0.12,0.11,0.11,0.11,0.11,0.07,0.09,0.16,0.12]],fsize=6.5)); S.append(Spacer(1,6))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    th=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; tr=[]
    for i,(_,r) in enumerate(dt_full.head(20).iterrows()):
        win=r.get("公司输赢",0)<0
        tr.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),
                   cell(f"${r.get('提款金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r.get('充值金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),
                   cell(f"${r.get('活动奖励',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),
                   cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(th,tr,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5)); S.append(Spacer(1,6))
    pg,ma=load_pref()
    S.append(sub_title("Top500提款用户游戏偏好")); S.append(chart_pref(pg,ma)); S.append(Spacer(1,4))
    if len(rg)>0:
        S.append(sub_title("▶ 高危游戏专项分析")); S.append(chart_risk_games(rg)); S.append(Spacer(1,4))
        gh=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; gr=[]
        for _,r in rg.head(12).iterrows():
            pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-20000 else ("高" if pl<-10 else "关注")
            rc2=C_RED if rl in("极高","高") else C_AMBER
            gr.append([cell(str(r.get("游戏名称",""))[:18],True),cell(str(r.get("show_name_厂商标签id",""))[:8]),
                       cell(f"{int(r.get('玩家数',0))}",False,C_DARK,TA_RIGHT),
                       cell(f"${r.get('投注金额',0)/10000:.2f}万",False,C_DARK,TA_RIGHT),
                       cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED,TA_RIGHT),
                       cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),
                       cell(f"{r.get('赢家率',0):.0f}%",False,C_RED if r.get("赢家率",0)>60 else C_AMBER,TA_RIGHT),
                       cell(f"${r.get('人均投注额',0):,.0f}",False,C_DARK,TA_RIGHT),
                       cell(rl,True,rc2)])
        S.append(dtable(gh,gr,[full*x for x in [0.22,0.10,0.07,0.10,0.11,0.08,0.08,0.10,0.07]],fsize=6.5)); S.append(Spacer(1,6))
        top_rg=rg.iloc[0] if len(rg)>0 else None
        ins1=f"高危游戏「{str(top_rg.get('游戏名称',''))}」公司输赢${top_rg.get('公司输赢',0):,.0f}，盈亏率{top_rg.get('盈亏率',0):.1f}%，建议复审RTP参数。" if top_rg is not None else "本周未检测到极高风险游戏。"
        S.append(insight_box([ins1,"建议对高赢家率游戏实施单账户赢额上限，防范套利风险。"], clr=C_RED))
    return S


def build_conclusion(K, dep_t, wdr_t, mfr, act, rg, hr):
    full = PW - 2 * MARGIN
    S = [sec_title("八、总结与行动建议")]
    
    cr_d = K["tw_充提差比"] - K["lw_充提差比"]
    rd = K["tw_首充次日复充率"] - K["lw_首充次日复充率"]
    
    # ── 1. 本周亮点：只写真实向好/增长的指标 ──
    S.append(sub_title("▶ 本周亮点"))
    bg_g = colors.HexColor("#f0fdf4")  # 使用柔和的绿色底色
    highlights = []
    
    # 修复充提差率判定：上涨（cr_d > 0）归入亮点
    if cr_d > 0:
        highlights.append(f"大盘资金沉淀极佳：充提差率显著反弹至 {K['tw_充提差比']:.2f}%（较上周大幅提升 {cr_d:+.2f}pp），充提差总额达 {K['tw_充提差']/10000:.2f}万 USD（环比大增 {K['pct_充提差']:+.1f}%）。")
    
    # 公司输赢上涨归入亮点
    if K.get("pct_公司输赢", 0) > 0:
        highlights.append(f"大盘盈利能力爆发：公司输赢达 {K['tw_公司输赢']/10000:.1f}万 USD（环比猛增 {K['pct_公司输赢']:+.1f}%），盈亏率提升至 {K['tw_盈亏率']:.3f}%。")
        
    if K.get("pct_活跃人数", 0) > 0:
        highlights.append(f"大盘基本盘稳固：日均活跃用户达 {K['tw_活跃人数']/7/10000:.2f}万（{K['pct_活跃人数']:+.1f}%），总投注额达 {K['tw_投注金额']/10000:.1f}万（{K['pct_投注金额']:+.1f}%）。")
        
    if K.get("tw_充提差ROI", 0) > K.get("lw_充提差ROI", 0):
        highlights.append(f"推广ROI效率倍增：日均充提差ROI升至 {K['tw_充提差ROI']:.2f}x（上周 {K['lw_充提差ROI']:.2f}x，环比大涨 {K['pct_充提差ROI']:+.1f}%）。")

    if not highlights:
        highlights.append("本周各项核心指标整体承压，暂无显著亮点，详见风险预警。")
        
    for hl in highlights:
        S.append(Table([[P(f"• {hl}", 8.5, False, C_DARK)]], colWidths=[full],
                       style=TableStyle([("BACKGROUND", (0, 0), (-1, -1), bg_g),
                                         ("LEFTPADDING", (0, 0), (-1, -1), 12),
                                         ("TOPPADDING", (0, 0), (-1, -1), 4),
                                         ("BOTTOMPADDING", (0, 0), (-1, -1), 4)])))
    S.append(Spacer(1, 8))

    # ── 2. 风险预警：只写真实恶化/下滑的指标 ──
    S.append(sub_title("▶ 风险预警"))
    bg_r = colors.HexColor("#fff5f5")  # 使用柔和的红色底色
    risks = []
    
    # 只有充提差率显着下滑才计入风险
    if cr_d < -3:
        risks.append(f"资金流失风险：充提差率重大下滑至 {K['tw_充提差比']:.2f}%（较上周下降 {cr_d:.2f}pp），大盘流速加剧。")
        
    # 拉新规模下滑判定
    if K.get("pct_注册人数", 0) < 0 or K.get("pct_首充人数", 0) < 0:
        risks.append(f"拉新进量规模全面收缩：注册人数 {int(K['tw_注册人数']):,}人（{K['pct_注册人数']:+.1f}%），首充人数 {int(K['tw_首充人数']):,}人（{K['pct_首充人数']:+.1f}%）。主要受本周日均推广消耗削减 {K['pct_真实消耗']:+.1f}% 的直接压制。")
        
    # 新客留存质量下滑
    if rd < -0.5:
        risks.append(f"新用户后续粘性微跌：首充次日充值留存下降至 {K['tw_复充率']:.1f}%（较上周下滑 {rd:+.1f}pp）。")

    if not risks:
        risks.append("本周大盘运转健康，各项风险监控指标均在安全阈值内。")
        
    for rk in risks:
        S.append(Table([[P(f"• {rk}", 8.5, False, C_DARK)]], colWidths=[full],
                       style=TableStyle([("BACKGROUND", (0, 0), (-1, -1), bg_r),
                                         ("LEFTPADDING", (0, 0), (-1, -1), 12),
                                         ("TOPPADDING", (0, 0), (-1, -1), 4),
                                         ("BOTTOMPADDING", (0, 0), (-1, -1), 4)])))
    S.append(Spacer(1, 8))

    # ── 3. 行动建议：基于真现状，动态化策略 ──
    S.append(sub_title("▶ 行动建议与执行策略"))
    actions = []
    
    # 场景 A：如果大盘ROI和沉淀很好，但是人进少了，说明需要恢复投放
    if cr_d > 3 and K.get("pct_首充人数", 0) < -5:
        actions.append(("[本周]", "稳步恢复高质量渠道推广", 
                        f"鉴于本周大盘充提差率高达 {K['tw_充提差比']:.2f}% 且推广ROI（{K['tw_充提差ROI']:.2f}x）极具性价比，说明现有核心模型非常健康。日均推广消耗削减 {K['pct_真实消耗']:.1f}% 是拉新规模下滑的根本主因。建议在维持现有质量的前提下，逐步恢复并加码优质渠道（如次留≥21%的总代渠道）的投放权重，扩充新客基本盘。"))
    
    # 场景 B：如果充提差率真的跌了
    if cr_d < -3:
        actions.append(("[本周]", "大盘资金沉淀为止跌行动", 
                        f"大盘充提差率触及风险水位（{K['tw_充提差比']:.2f}%），需立即复核头部大客的提款分布和核心高危游戏的RTP参数，提高风控控杀频次。"))
                        
    # 场景 C：新客承接优化建议
    if K.get("pct_首充人数", 0) < 0:
        actions.append(("[常规]", "新客承接链与黄金24小时留存策略", 
                        "首充人数与次留微幅承压，建议针对新客导入的黄金24小时，细化首充落地福利及首充礼包的弹窗承接组合；同时对D1/D2活跃用户增设阶梯式复充奖励，稳固高价值首充活动用户的二次留存率。"))

    # 绘制行动建议表格
    h_act = ["时间窗", "策略核心事项", "执行说明与数据依据"]
    r_act = []
    for prefix, title, desc in actions:
        r_act.append([cell(prefix, True, C_ORANGE if "本周" in prefix else C_GREEN, TA_CENTER),
                      cell(title, True, C_DARK, TA_LEFT),
                      cell(desc, False, C_DARK, TA_LEFT)])
                      
    if r_act:
        S.append(dtable(h_act, r_act, [full * 0.10, full * 0.28, full * 0.62], fsize=8))
        
    return S


def header_footer(c, doc):
    c.saveState(); w,h=A4
    c.setFillColor(colors.HexColor("#7c2d12")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN,h-17,f"Lucro 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
    c.restoreState()


def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print("📊 Lucro 周报 PDF — 本周 20260605-20260611 生成中...")
    resolve_files(); setup()

    print("  ▶ 加载大盘数据...")
    K, trend = load_platform()
    print(f"  ▶ 充值{K['tw_充值金额']/10000:.1f}万，充提差ROI={K['tw_充提差ROI']:.2f}x")
    dash_ret    = load_dashboard_retention()
    weekly_ret  = load_weekly_retention()

    print("  ▶ 加载总代数据...")
    agents      = load_agents()
    agent_ret, ends = load_agent_ret()

    print("  ▶ 加载用户数据...")
    tw_v, lw_v  = load_vip()
    dep_t, wdr_t, dc_tw, dt_top = load_top_users()

    print("  ▶ 加载游戏数据...")
    top30       = load_games()
    mfr         = load_mfr()
    mfr_delta   = load_mfr_game_delta()

    print("  ▶ 加载活动数据...")
    act_df, tot_gift = load_activities()
    fdr         = load_first_dep_ret()

    print("  ▶ 加载风险数据...")
    dt_raw = pd.read_csv(FILES["dt_tw"])
    dc_raw = pd.read_csv(FILES["dc_tw"])
    for df in [dt_raw, dc_raw]:
        for col in df.columns:
            for kw in ["充值金额","提款金额","公司输赢","活动奖励","投注金额","充提差"]:
                if kw in col:
                    df[col]=pd.to_numeric(df[col].astype(str).str.replace(",",""),errors="coerce")
    hr, sp, top20g, rg, top500 = load_risk(dt_raw, dc_raw)

    # 给top20提款用户加主玩游戏
    dpf=pd.read_csv(FILES["pref_tw"]); dpf["阶段汇总"]=dpf["阶段汇总"].apply(clean)
    ind_pf="分析指标" if "分析指标" in dpf.columns else dpf.columns[3]
    game_pf="游戏名称" if "游戏名称" in dpf.columns else dpf.columns[2]
    mfr_pf="show_name_厂商标签id" if "show_name_厂商标签id" in dpf.columns else dpf.columns[1]
    tg2=(dpf[dpf[ind_pf]=="投注金额"].sort_values("阶段汇总",ascending=False)
         .groupby("账户ID").first()[[mfr_pf,game_pf]]
         .rename(columns={mfr_pf:"主玩厂商",game_pf:"主玩游戏"}).reset_index())
    dt_top=dt_top.merge(tg2,on="账户ID",how="left")

    print("  ▶ 导出风控Excel...")
    excel_out = OUTPUT_DIR/f"Lucro_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out),engine="openpyxl") as xw:
        cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","投充比"]
        hr[[c for c in cols_hr if c in hr.columns]].to_excel(xw,sheet_name="高风险用户",index=False)
        if len(sp)>0:
            sp[["账户ID","提款金额","充值金额","公司输赢"]].to_excel(xw,sheet_name="超高投充用户",index=False)
    print(f"  ✅ 风控Excel: {excel_out}")

    print("  ▶ 构建PDF章节...")
    out = OUTPUT_DIR/f"Lucro周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc = SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,
                            topMargin=MARGIN+22,bottomMargin=MARGIN+10,
                            title=f"Lucro平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story=[
        Spacer(1,3*cm),
        P("Lucro 平台数据周报",30,True,C_DARK,TA_CENTER), Spacer(1,.5*cm),
        P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",14,False,C_GRAY,TA_CENTER),
        Spacer(1,.2*cm),
        P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",12,False,C_GRAY,TA_CENTER),
        Spacer(1,.5*cm),
        HRFlowable(width="50%",thickness=2,color=C_ORANGE,hAlign="CENTER"),
        PageBreak()
    ]
    story += build_overview(K,trend,weekly_ret,dash_ret);        story.append(PageBreak())
    story += build_agents(agents,agent_ret,ends);                 story.append(PageBreak())
    story += build_users(tw_v,lw_v,dep_t,wdr_t,dc_tw,dt_top);   story.append(PageBreak())
    story += build_games(mfr,top30,mfr_delta);                   story.append(PageBreak())
    story += build_activities(act_df,tot_gift);                   story.append(PageBreak())
    story += build_first_dep_ret(fdr);                            story.append(PageBreak())
    story += build_risk(hr,sp,top20g,rg,top500,dt_top);          story.append(PageBreak())
    story += build_conclusion(K,dep_t,wdr_t,mfr,act_df,rg,hr)

    print("  ▶ 渲染PDF（可能需要2-3分钟）...")
    doc.build(story, onFirstPage=header_footer, onLaterPages=header_footer)
    size = out.stat().st_size/1024
    print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out), str(excel_out)

if __name__ == "__main__":
    main()

📊 Lucro 周报 PDF — 本周 20260605-20260611 生成中...
  ✅ [platform       ] 平台报表_USD_20260619151840.xlsx
  ✅ [daily          ] 日报-大盘日报_20260619(4).xlsx
  ✅ [retention      ] 整体 首充留存（近7天）_20260522-20260618 (1).csv
  ✅ [agent_plat     ] 平台报表-总代_USD_20260619152732.xlsx
  ✅ [agent_promo    ] 推广报表-总代_USD_20260619152801.xlsx
  ✅ [agent_ret      ] 首充充值留存_全量数据_20260522_20260618.csv
  ✅ [vip            ] VIP报表_USD_20260619153729.xlsx
  ✅ [dt_tw          ] top提款用户_全量数据_20260612_20260618_日期对比20260605_20260611 (1).csv
  ✅ [dt_lw          ] top提款用户_全量数据_20260612_20260618_日期对比20260605_20260611 (1).csv
  ✅ [dc_tw          ] 头部充值用户_全量数据_20260605_20260611_日期对比20260529_20260604 (2).csv
  ✅ [dc_lw          ] 头部充值用户_全量数据_20260605_20260611_日期对比20260529_20260604 (2).csv
  ✅ [pref_tw        ] 本周top500提款用户游戏偏好_全量数据_20260612_20260618 (2).csv
  ✅ [pref_lw        ] 上周top500提款用户游戏偏好_全量数据_20260605_20260611 (2).csv
  ✅ [mfr            ] 厂商投注数据_全量数据_20260605_20260618 (1).csv
  ✅ [game_tw        ] 游戏报表-详情_USD_本周.xlsx
  ✅ [gam

In [2]:
"""
Lucro 平台数据周报 v1.0
统计周期：20260529 - 20260604
对比基准：20260522 - 20260528

使用说明：
  1. 修改下方 DATA_ROOT / OUTPUT_DIR 为本地路径
  2. 确认 THIS_WEEK / LAST_WEEK 日期正确
  3. 运行：python lucro_report_20260529_20260604.py
     或在 Jupyter 中：%run lucro_report_20260529_20260604.py

章节：一、大盘核心数据  二、总代分析  三、用户分析
      四、游戏分析      五、活动分析  六、风险专项  七、总结建议
（道具专项本期暂不生成）
"""

from pathlib import Path
import sys, io, warnings
from datetime import datetime, timedelta
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
warnings.filterwarnings("ignore")

from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.units import cm
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT, TA_CENTER, TA_RIGHT
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer, Table,
    TableStyle, Image, PageBreak, HRFlowable, KeepTogether)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# ══════════════════════════════════════════════
# ★ 路径配置（本地运行时修改这两行）
# ══════════════════════════════════════════════
DATA_ROOT  = Path(r"D:\周报更新版\Lucro")    # ← 改为本地数据根目录
OUTPUT_DIR = Path(r"D:\周报更新版\Lucro\输出") # ← 改为输出目录

THIS_WEEK = ("20260529", "20260604")
LAST_WEEK = ("20260522", "20260528")
REPORT_END = THIS_WEEK[1]
PLATFORM   = "Lucro"

# ── 留存目标 ──────────────────────────────────
RET_TARGETS = {"次留": 21.0, "3留": 15.0, "7留": 11.0}

# ── 颜色 / 字体 ───────────────────────────────
FN, FNB = "WQY", "WQYB"
FILES: dict = {}

C_BLUE   = colors.HexColor("#1d4ed8");  C_BLUE2  = colors.HexColor("#3b82f6")
C_GREEN  = colors.HexColor("#059669");  C_RED    = colors.HexColor("#dc2626")
C_AMBER  = colors.HexColor("#d97706");  C_PURPLE = colors.HexColor("#7c3aed")
C_GRAY   = colors.HexColor("#64748b");  C_LGRAY  = colors.HexColor("#f1f5f9")
C_WHITE  = colors.white;                C_DARK   = colors.HexColor("#1e293b")
C_BORDER = colors.HexColor("#cbd5e1");  C_ROW    = colors.HexColor("#f8fafc")
C_TEAL   = colors.HexColor("#0f766e");  C_TARGET = colors.HexColor("#0369a1")
C_ORANGE = colors.HexColor("#ea580c")

PW, PH = A4
MARGIN = 1.6 * cm
CHART_COLORS = ["#1d4ed8","#059669","#d97706","#7c3aed","#dc2626",
                "#0891b2","#65a30d","#ea580c","#9333ea","#be185d","#475569","#0f766e"]
CR_HIGH, CR_LOW = 17, 5

MFR_SHORT = {
    "Pragmatic Play":"PP","PG Soft":"PG","PlayTech":"PT","Rectangle":"RG",
    "Originals":"自研","Fat Panda":"FP","Nolimit City":"NC","Hacksaw":"HK",
    "Push Gaming":"PUG","Evolution":"EVO","Relax Gaming":"REL",
    "Funky Games":"FUG","AvatarUx":"AVU","EazyGaming":"EZG",
    "759 Gaming":"759","Caleta":"CAL",
}

# ══════════════════════════════════════════════
# 工具函数
# ══════════════════════════════════════════════
def shorten_mfr(n):
    if not isinstance(n, str): return str(n)
    for k, v in MFR_SHORT.items():
        if k in n: return v
    return n[:8]

def ret_end(week_start, lag):
    e = (datetime.strptime(REPORT_END,"%Y%m%d")-timedelta(days=lag)).strftime("%Y%m%d")
    return e if e >= week_start else None

def fmt_lbl(d): return f"{d[4:6]}/{d[6:]}" if d else "-"

def clean(s):
    try: return float(str(s).replace(",","").replace("%","").strip())
    except: return np.nan

def to_num(df, cols=None):
    if cols is None:
        cols = [c for c in df.columns
                if c not in ("日期","总代.名称","name_总代","总代.ID","原始时间","对比时间1")]
    for c in cols:
        if c in df.columns: df[c] = df[c].apply(clean)
    return df

def pct(n, o): return (n-o)/abs(o)*100 if o and o!=0 else 0.0
def pct_vec(ns, os):
    return ((ns-os)/os.abs()*100).replace([np.inf,-np.inf],0).fillna(0)
def trunc_mean(s, drop=0):
    if drop >= len(s): return np.nan
    v = s.iloc[:len(s)-drop] if drop else s
    nz = v[v>0]; return nz.mean() if len(nz) else v.mean()
def wavg_series(vals, weights):
    ok = vals.notna() & (weights>0)
    if not ok.any(): return np.nan
    return float(np.average(vals[ok], weights=weights[ok]))
def fig_img(fig, w=16, h=7):
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight",
                facecolor=fig.get_facecolor())
    buf.seek(0); plt.close(fig)
    return Image(buf, width=w*cm, height=h*cm)
def fret(v): return "-" if v is None or (isinstance(v,float) and np.isnan(v)) else f"{v:.1f}%"
def gclr(v, t=0): return C_GREEN if v>t else (C_RED if v<t else C_GRAY)

# ══════════════════════════════════════════════
# 字体 & 文件匹配
# ══════════════════════════════════════════════
def setup():
    candidates = [
        # Windows
        r"C:\Windows\Fonts\msyh.ttc",
        r"C:\Windows\Fonts\msyhbd.ttc",
        r"C:\Windows\Fonts\simhei.ttf",
        r"C:\Windows\Fonts\simsun.ttc",
        # Linux
        "/usr/share/fonts/truetype/wqy/wqy-zenhei.ttc",
        "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc",
        # macOS
        "/System/Library/Fonts/PingFang.ttc",
        "/Library/Fonts/Arial Unicode.ttf",
    ]
    font_path = next((p for p in candidates if Path(p).exists()), None)
    if not font_path:
        from matplotlib import font_manager as fm
        for f in fm.fontManager.ttflist:
            if any(k in f.name for k in ["YaHei","Hei","SimSun","WenQuanYi","Noto Sans CJK","PingFang"]):
                if Path(f.fname).exists(): font_path = f.fname; break
    if not font_path:
        raise FileNotFoundError("未找到中文字体，请安装微软雅黑或文泉驿字体")
    print(f"  ▶ 字体：{font_path}")
    ttc = font_path.lower().endswith(".ttc")
    pdfmetrics.registerFont(TTFont(FN,  font_path, subfontIndex=0) if ttc else TTFont(FN,  font_path))
    try:    pdfmetrics.registerFont(TTFont(FNB, font_path, subfontIndex=1) if ttc else TTFont(FNB, font_path))
    except: pdfmetrics.registerFont(TTFont(FNB, font_path, subfontIndex=0) if ttc else TTFont(FNB, font_path))
    from matplotlib import font_manager as fm
    fm.fontManager.addfont(font_path)
    added = [f.name for f in fm.fontManager.ttflist if f.fname == font_path]
    plt.rcParams.update({"font.family": added[0] if added else "DejaVu Sans",
                         "axes.unicode_minus": False, "font.size": 8.5})

def resolve_files(root):
    global FILES
    tw0, tw1 = THIS_WEEK; lw0, lw1 = LAST_WEEK
    all_files = []
    try:
        for h in root.rglob("*"):
            try:
                if h.is_file() and not h.name.startswith("~$"):
                    all_files.append(h)
            except (PermissionError, OSError): pass
    except Exception as e:
        print(f"  ⚠️  扫描目录出错: {e}")

    def find(kws):
        kws = kws if isinstance(kws, list) else [kws]
        hits = [h for h in all_files if all(k in h.name for k in kws)]
        return max(hits, key=lambda h: h.stat().st_mtime) if hits else None

    km = {
        "platform":      ["平台报表_USD"],
        "daily":         ["日报-大盘日报"],
        "retention":     ["整体", "首充留存"],
        "agent_plat":    ["平台报表-总代_USD"],
        "agent_promo":   ["推广报表-总代_USD"],
        "agent_ret":     ["首充充值留存_全量数据"],
        "vip":           ["VIP报表_USD"],
        "dt_tw":         [f"top提款用户_全量数据_{tw0}"],
        "dt_lw":         [f"top提款用户_全量数据_{lw0}"],
        "dc_tw":         [f"头部充值用户_全量数据_{tw0}"],
        "dc_lw":         [f"头部充值用户_全量数据_{lw0}"],
        "pref_tw":       [f"本周top500提款用户游戏偏好_全量数据_{tw0}"],
        "pref_lw":       [f"上周top500提款用户游戏偏好_全量数据_{lw0}"],
        "mfr":           ["厂商投注数据_全量数据"],
        "game_tw":       ["游戏报表-详情_USD_本周"],
        "game_lw":       ["游戏报表-详情_USD_上周"],
        "gift":          ["各活动赠送_全量数据"],
        "first_dep_ret": ["首次充值活动用户充值留存情况"],
        "vip_ret_chg":   ["VIP充值-充值_近28天"],
        "vip_ret_act":   ["VIP充值-活跃_近28天"],
    }
    fail = 0
    for key, kws in km.items():
        p = find(kws)
        if p:
            FILES[key] = p
            print(f"     ✅ [{key:15s}] {p.name}")
        else:
            print(f"     ❌ [{key:15s}] 找不到含{kws}的文件")
            fail += 1
    if fail:
        raise FileNotFoundError(
            f"\n❌ 共{fail}个文件未找到，请检查 DATA_ROOT 路径及文件名是否正确。\n"
            f"当前 DATA_ROOT = {root}\n"
        )
    print(f"  ▶ 全部{len(km)}个文件匹配成功\n")

# ══════════════════════════════════════════════
# PDF 组件
# ══════════════════════════════════════════════
def P(txt, sz=8.5, bold=False, clr=colors.black, align=TA_LEFT):
    st = ParagraphStyle("x", fontName=FNB if bold else FN, fontSize=sz,
                        textColor=clr, alignment=align, leading=sz*1.4,
                        spaceAfter=0, spaceBefore=0)
    return Paragraph(str(txt), st)

def sec_title(text, clr=C_BLUE):
    return KeepTogether([Spacer(1,6),
        Table([[P(text,13,True,C_WHITE)]], colWidths=[PW-2*MARGIN],
              style=TableStyle([("BACKGROUND",(0,0),(-1,-1),clr),
                                ("TOPPADDING",(0,0),(-1,-1),7),
                                ("BOTTOMPADDING",(0,0),(-1,-1),7),
                                ("LEFTPADDING",(0,0),(-1,-1),14)])),
        Spacer(1,8)])

def sub_title(text):
    return KeepTogether([Spacer(1,5),
        HRFlowable(width="100%",thickness=1.5,color=C_BLUE2),
        Spacer(1,3), P(f"■  {text}",10,True,C_DARK), Spacer(1,6)])

def insight_box(lines, clr=C_GREEN):
    bkg = (colors.HexColor("#f0fdf4") if clr==C_GREEN else
           colors.HexColor("#fffbeb") if clr==C_AMBER else
           colors.HexColor("#fef2f2"))
    rows = [[P(f"◆ {l}",8.5,False,C_DARK)] for l in lines]
    return KeepTogether([
        Table(rows, colWidths=[PW-2*MARGIN], style=TableStyle([
            ("BACKGROUND",(0,0),(-1,-1),bkg),("BOX",(0,0),(-1,-1),1,clr),
            ("LINEBELOW",(0,0),(-1,-2),0.3,C_BORDER),
            ("LEFTPADDING",(0,0),(-1,-1),10),("RIGHTPADDING",(0,0),(-1,-1),10),
            ("TOPPADDING",(0,0),(-1,-1),4),("BOTTOMPADDING",(0,0),(-1,-1),4),
        ])), Spacer(1,8)])

def dtable(headers, rows, widths=None, zebra=True, hdr_clr=None,
           fsize=6.8, extra_style=None):
    full = PW-2*MARGIN
    if not widths: widths = [full/len(headers)]*len(headers)
    if hdr_clr is None: hdr_clr = C_DARK
    st = TableStyle([
        ("BACKGROUND",(0,0),(-1,0),hdr_clr),("TEXTCOLOR",(0,0),(-1,0),C_WHITE),
        ("FONTNAME",(0,0),(-1,0),FNB),("FONTNAME",(0,1),(-1,-1),FN),
        ("FONTSIZE",(0,0),(-1,-1),fsize),
        ("TOPPADDING",(0,0),(-1,-1),2),("BOTTOMPADDING",(0,0),(-1,-1),2),
        ("LEFTPADDING",(0,0),(-1,-1),2),("RIGHTPADDING",(0,0),(-1,-1),2),
        ("GRID",(0,0),(-1,-1),0.3,C_BORDER),("VALIGN",(0,0),(-1,-1),"MIDDLE"),
    ])
    if zebra:
        for i in range(1,len(rows)+1,2): st.add("BACKGROUND",(0,i),(-1,i),C_ROW)
    if extra_style:
        for cmd in extra_style: st.add(*cmd)
    hrow = [P(h,fsize,True,C_WHITE,TA_CENTER) for h in headers]
    body = []
    for row in rows:
        body.append([
            P(str(c[0]),fsize,c[1] if len(c)>1 else False,
              c[2] if len(c)>2 else colors.black,
              c[3] if len(c)>3 else TA_LEFT)
            if isinstance(c,(list,tuple)) else P(str(c),fsize)
            for c in row
        ])
    return Table([hrow]+body, colWidths=widths, style=st, hAlign="LEFT", repeatRows=1)

def cell(t,bold=False,clr=colors.black,align=TA_LEFT): return (t,bold,clr,align)
def rc(v, gup=True, d=1, good_up=None):
    if good_up is not None: gup=good_up
    c = C_GREEN if (v>0)==gup else C_RED
    return cell(f"{'+' if v>=0 else ''}{v:.{d}f}%",False,c,TA_RIGHT)

def kpi_card4(items, cols=4):
    fw = (PW-2*MARGIN)/cols-4
    rows, row = [], []
    for label,tv,lv,chg,_ in items:
        pclr = C_GREEN if chg>=0 else C_RED
        inner = Table(
            [[P(label,7.5,False,C_GRAY)],[P(str(tv),14,True,C_DARK)],
             [P(f"上周：{lv}",7.5,False,C_GRAY)],
             [P(f"{'+' if chg>=0 else ''}{chg:.1f}%",8,True,pclr)]],
            colWidths=[fw],
            style=TableStyle([("BACKGROUND",(0,0),(-1,-1),C_LGRAY),
                              ("BOX",(0,0),(-1,-1),0.5,C_BORDER),
                              ("LEFTPADDING",(0,0),(-1,-1),8),
                              ("TOPPADDING",(0,0),(-1,-1),5),
                              ("BOTTOMPADDING",(0,0),(-1,-1),5)]))
        row.append(inner)
        if len(row)==cols: rows.append(row); row=[]
    if row:
        while len(row)<cols: row.append(Spacer(fw,1))
        rows.append(row)
    t = Table(rows, colWidths=[(PW-2*MARGIN)/cols]*cols, hAlign="LEFT", vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),
                            ("LEFTPADDING",(0,0),(-1,-1),2),
                            ("RIGHTPADDING",(0,0),(-1,-1),2),
                            ("TOPPADDING",(0,0),(-1,-1),2),
                            ("BOTTOMPADDING",(0,0),(-1,-1),2)]))
    return t

# ══════════════════════════════════════════════
# 数据加载
# ══════════════════════════════════════════════
def load_platform():
    df = pd.read_excel(FILES["platform"]); df["日期"] = df["日期"].astype(str)
    df = df.sort_values("日期"); df = to_num(df)
    tw = df[df["日期"].between(*THIS_WEEK)]
    lw = df[df["日期"].between(*LAST_WEEK)]
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str)
    dd = to_num(dd)
    tw_d = dd[dd["日期"].between(*THIS_WEEK)]
    lw_d = dd[dd["日期"].between(*LAST_WEEK)]

    K = {}
    # 求和指标
    for c in ["注册人数","首充人数","一级首充人数","活跃人数","充值人数",
              "充值金额","提现金额","提现人数","充提差","投注金额",
              "投注人数","公司输赢","总赠送金额","首充金额"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].sum(); K[f"lw_{c}"] = lw[c].sum()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    # 均值指标
    for c in ["全量Arppu","首充Arppu","老用户ARPPU"]:
        if c not in df.columns: continue
        K[f"tw_{c}"] = tw[c].mean(); K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])
    # 截断均值（首充留存等比率指标）
    TRUNC = {"首充次日复充率":1,"首充7日复充率":6,"首充当日复充率":0,
             "首充转化率":0,"总赠送充值比":0,"活跃用户付费率":0}
    for c, drop in TRUNC.items():
        if c not in df.columns: continue
        K[f"tw_{c}"] = trunc_mean(tw[c], drop)
        K[f"lw_{c}"] = lw[c].mean()
        K[f"pct_{c}"] = pct(K[f"tw_{c}"], K[f"lw_{c}"])

    # ★ 充提差率 = sum(充提差)/sum(充值金额)
    K["tw_充提差比"] = tw["充提差"].sum() / tw["充值金额"].sum() * 100 if tw["充值金额"].sum() > 0 else np.nan
    K["lw_充提差比"] = lw["充提差"].sum() / lw["充值金额"].sum() * 100 if lw["充值金额"].sum() > 0 else np.nan
    K["pct_充提差比"] = pct(K["tw_充提差比"], K["lw_充提差比"])
    # ★ 盈亏率 = sum(公司输赢)/sum(投注金额)
    K["tw_盈亏率"] = tw["公司输赢"].sum() / tw["投注金额"].sum() * 100 if tw["投注金额"].sum() > 0 else np.nan
    K["lw_盈亏率"] = lw["公司输赢"].sum() / lw["投注金额"].sum() * 100 if lw["投注金额"].sum() > 0 else np.nan
    K["pct_盈亏率"] = pct(K["tw_盈亏率"], K["lw_盈亏率"])

    # 推广消耗：固定剔除本周最后一天（消耗未录完）
    tw_d_asc = tw_d.sort_values("日期")
    tw_cost_valid = tw_d_asc["真实消耗"].iloc[:-1]
    lw_cost_days  = len(lw_d)
    tw_cost_days  = len(tw_cost_valid)
    K["tw_真实消耗_日均"]  = tw_cost_valid.sum()/tw_cost_days if tw_cost_days>0 else 0
    K["lw_真实消耗_日均"]  = lw_d["真实消耗"].sum()/lw_cost_days if lw_cost_days>0 else 0
    K["tw_真实消耗_有效天"] = tw_cost_days
    K["pct_真实消耗"]      = pct(K["tw_真实消耗_日均"], K["lw_真实消耗_日均"])

    # 充提差ROI（日均充提差/日均消耗，天数对齐）
    tw_sorted = tw.sort_values("日期")
    tw_cd_valid = tw_sorted["充提差"].iloc[:tw_cost_days]
    K["tw_充提差_日均_roi"] = tw_cd_valid.sum()/tw_cost_days if tw_cost_days>0 else 0
    K["lw_充提差_日均_roi"] = lw.sort_values("日期")["充提差"].sum()/lw_cost_days if lw_cost_days>0 else 0
    K["tw_充提差ROI"] = (K["tw_充提差_日均_roi"]/K["tw_真实消耗_日均"]
                         if K["tw_真实消耗_日均"]>0 else 0)
    K["lw_充提差ROI"] = (K["lw_充提差_日均_roi"]/K["lw_真实消耗_日均"]
                         if K["lw_真实消耗_日均"]>0 else 0)
    K["pct_充提差ROI"] = pct(K["tw_充提差ROI"], K["lw_充提差ROI"])

    K["tw_赠送充值比"] = K["tw_总赠送金额"]/K["tw_充值金额"]*100
    K["lw_赠送充值比"] = K["lw_总赠送金额"]/K["lw_充值金额"]*100
    K["pct_赠送充值比"] = pct(K["tw_赠送充值比"], K["lw_赠送充值比"])

    # 近14日趋势
    last14 = df.tail(14)
    trend = {
        "dates":   [f"{d[4:6]}/{d[6:]}" for d in last14["日期"].tolist()],
        "充值":    last14["充值金额"].tolist(),
        "提现":    last14["提现金额"].tolist(),
        "充提差比": last14["充提差比"].tolist(),
        "公司输赢": last14["公司输赢"].tolist(),
        "首充":    last14["首充人数"].tolist(),
        "注册":    last14["注册人数"].tolist(),
    }
    return K, trend


def load_dashboard_retention():
    dd = pd.read_excel(FILES["daily"]); dd["日期"] = dd["日期"].astype(str)
    specs = [("nd","首充2日复充率",1),("td","首充3日复充率",2),("sd","首充7日复充率",6)]
    for _,col,_ in specs:
        if col in dd.columns:
            dd[col] = pd.to_numeric(dd[col].astype(str).str.replace("%","").str.strip(),
                                    errors="coerce")
    R = {}
    for key,col,lag in specs:
        for ws,we,pfx in [(THIS_WEEK[0],THIS_WEEK[1],"tw_"),
                          (LAST_WEEK[0],LAST_WEEK[1],"lw_")]:
            ec = ret_end(ws,lag)
            sub = dd[(dd["日期"]>=ws)&(dd["日期"]<=we)]
            if ec: sub = sub[sub["日期"]<=ec]
            R[f"{pfx}{key}"] = float(sub[col].mean()) if len(sub) and col in sub else np.nan
        R[f"tw_{key}_end"] = ret_end(THIS_WEEK[0],lag)
        R[f"lw_{key}_end"] = ret_end(LAST_WEEK[0],lag)
    return R


def load_weekly_retention():
    df = pd.read_csv(FILES["retention"])
    # 列名适配（列名为"1日"而非"第1日"）
    time_col = "初始事件发生时间" if "初始事件发生时间" in df.columns else df.columns[0]
    user_col = "充值成功事件用户数" if "充值成功事件用户数" in df.columns else df.columns[1]
    daily = df[~df[time_col].astype(str).str.contains("阶段值",na=False)].copy()
    daily["ds"] = daily[time_col].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
    dr = daily[daily["指标"]=="留存率"].copy()
    du = daily[daily["指标"]=="留存人数"].copy()
    for c in ["1日","2日","3日","6日","7日"]:
        if c in dr.columns:
            dr[c] = pd.to_numeric(dr[c].astype(str).str.replace("%","").str.strip(),
                                  errors="coerce")
    def _d(s): return datetime.strptime(s,"%Y%m%d")
    def _f(d): return d.strftime("%Y-%m-%d")
    def _l(s,e): return f"{s.strftime('%m/%d')}-{e.strftime('%m/%d')}"
    tw_s,tw_e = _d(THIS_WEEK[0]),_d(THIS_WEEK[1])
    lw_s,lw_e = _d(LAST_WEEK[0]),_d(LAST_WEEK[1])
    w2e=lw_s-timedelta(days=1); w2s=w2e-timedelta(days=6)
    w1e=w2s-timedelta(days=1); w1s=w1e-timedelta(days=6)
    weeks = [(f"第1周\n{_l(w1s,w1e)}",_f(w1s),_f(w1e)),
             (f"第2周\n{_l(w2s,w2e)}",_f(w2s),_f(w2e)),
             (f"上周\n{_l(lw_s,lw_e)}",_f(lw_s),_f(lw_e)),
             (f"本周\n{_l(tw_s,tw_e)}",_f(tw_s),_f(tw_e))]
    result = []
    for wk,s,e in weeks:
        mr=dr[(dr["ds"]>=s)&(dr["ds"]<=e)]; mu=du[(du["ds"]>=s)&(du["ds"]<=e)]
        row = {"week":wk,"users":mu[user_col].sum() if user_col in mu.columns else 0}
        for col in ["1日","2日","3日","6日","7日"]:
            rs = mr[col].values if col in mr.columns else np.array([])
            us = mu[user_col].values if user_col in mu.columns else np.ones(len(rs))
            if len(rs)>0 and len(rs)==len(us):
                v = ~np.isnan(rs.astype(float))
                row[col] = float(np.average(rs.astype(float)[v],weights=us[v])) if v.sum()>0 else np.nan
            else: row[col] = float(np.nanmean(rs.astype(float))) if len(rs)>0 else np.nan
        result.append(row)
    return result


def load_agents():
    dp = pd.read_excel(FILES["agent_plat"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    dr = pd.read_excel(FILES["agent_promo"]); dr["日期"]=dr["日期"].astype(str); dr=to_num(dr)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]; lw_p=dp[dp["日期"].between(*LAST_WEEK)]
    tw_r=dr[dr["日期"].between(*THIS_WEEK)]; lw_r=dr[dr["日期"].between(*LAST_WEEK)]

    sa=["充值金额","提现金额","充提差","首充金额","首充人数","注册人数",
        "充值人数","投注金额","公司输赢","总赠送金额"]
    def agg_p(d):
        g=d.groupby(["总代.ID","总代.名称"]).agg(
            {c:"sum" for c in sa if c in d.columns}).reset_index()
        for c in ["充提差比","首充次日充值留存"]:
            if c in d.columns:
                g=g.merge(d.groupby("总代.ID")[c].mean().rename(c),on="总代.ID",how="left")
        g["充提差率"]=g["充提差"]/g["充值金额"]*100
        return g

    tw_pa=agg_p(tw_p); lw_pa=agg_p(lw_p)

    sr=["总消耗","注册人数","首充人数","一级首充人数","一级首充金额"]
    def agg_r(d):
        avail=[c for c in sr if c in d.columns]
        g=d.groupby(["总代.ID","总代.名称"]).agg({c:"sum" for c in avail}).reset_index()
        if "总消耗" in g.columns and "一级首充人数" in g.columns:
            g["一级首充成本"]=g["总消耗"]/g["一级首充人数"].replace(0,np.nan)
        else:
            g["一级首充成本"]=np.nan
        return g

    tw_ra=agg_r(tw_r); lw_ra=agg_r(lw_r)
    lw_ra["lw_fc_cost"]=(lw_ra["总消耗"]/lw_ra["一级首充人数"].replace(0,np.nan)
                          if "总消耗" in lw_ra.columns else np.nan)

    lw_sub=lw_pa[["总代.ID"]+
                 [c for c in ["充值金额","注册人数","充提差率"] if c in lw_pa.columns]
                 ].rename(columns={"充值金额":"lw_充值","注册人数":"lw_注册","充提差率":"lw_充提差率"})
    m=tw_pa.merge(lw_sub,on="总代.ID",how="left")
    r_cols=["总代.ID","一级首充成本","一级首充人数"]
    if "总消耗" in tw_ra.columns: r_cols.append("总消耗")
    m=m.merge(tw_ra[[c for c in r_cols if c in tw_ra.columns]],on="总代.ID",how="left")
    fc_cols=["总代.ID","lw_fc_cost"]
    m=m.merge(lw_ra[[c for c in fc_cols if c in lw_ra.columns]],on="总代.ID",how="left")
    m["注册环比"]=pct_vec(m["注册人数"],m["lw_注册"])
    return m.sort_values("充值金额",ascending=False)


def load_agent_ret():
    df=pd.read_csv(FILES["agent_ret"])
    time_col="初始事件发生时间" if "初始事件发生时间" in df.columns else df.columns[0]
    df["_d"]=df[time_col].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"]=df["_d"].str.replace("-","").fillna("")
    for c in ["1日","2日","7日"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
    user_col="充值成功事件用户数" if "充值成功事件用户数" in df.columns else "首充用户数"
    if user_col in df.columns:
        df[user_col]=pd.to_numeric(df[user_col],errors="coerce").fillna(0)
    else: df[user_col]=1
    name_col="name_总代" if "name_总代" in df.columns else "总代.名称"
    ind_col="总代" if "总代" in df.columns else "总代.ID"
    daily=df[df["_yyyymmdd"].str.match(r"^\d{8}$",na=False)].copy()
    daily_ret=daily[daily["指标"]=="留存率"] if "指标" in daily.columns else daily

    ends={}
    for key,lag in [("nd",1),("3d",2),("7d",6)]:
        ends[f"tw_{key}_end"]=ret_end(THIS_WEEK[0],lag)
        ends[f"lw_{key}_end"]=ret_end(LAST_WEEK[0],lag)

    def _slice(ws,we,ec):
        sub=daily_ret[daily_ret["_yyyymmdd"].between(ws,we)]
        if ec: sub=sub[sub["_yyyymmdd"]<=min(ec,we)]
        return sub

    slices={"tw_nd":_slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_nd_end"]),
            "tw_3d":_slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_3d_end"]),
            "tw_7d":_slice(THIS_WEEK[0],THIS_WEEK[1],ends["tw_7d_end"]),
            "lw_nd":_slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_nd_end"]),
            "lw_3d":_slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_3d_end"]),
            "lw_7d":_slice(LAST_WEEK[0],LAST_WEEK[1],ends["lw_7d_end"])}
    col_map={"tw_nd":"1日","tw_3d":"2日","tw_7d":"7日",
             "lw_nd":"1日","lw_3d":"2日","lw_7d":"7日"}

    def _wavg(df_sub,col):
        res={}
        if name_col not in df_sub.columns or col not in df_sub.columns: return res
        for nm,grp in df_sub.groupby(name_col):
            v=wavg_series(grp[col],grp[user_col])
            if not np.isnan(v): res[nm]=v
        return res

    per_agent={k:_wavg(slices[k],col_map[k]) for k in slices}
    id_map={}
    if name_col in daily_ret.columns and ind_col in daily_ret.columns:
        for nm,grp in daily_ret.groupby(name_col):
            vals=pd.to_numeric(grp[ind_col],errors="coerce").dropna().values
            if len(vals): id_map[nm]=int(vals[0])
    all_names=set().union(*[set(v) for v in per_agent.values()])
    ret={}
    for nm in all_names:
        ret[nm]={k:per_agent[k].get(nm,np.nan) for k in per_agent}
        ret[nm]["agent_id"]=id_map.get(nm)
    return ret, ends


def load_vip():
    df=pd.read_excel(FILES["vip"]); df["日期"]=df["日期"].astype(str)
    num=["活跃人数","投注人数","投注局数","投注金额","充值人数","充值金额","提现人数","提现金额"]
    df=to_num(df,num)
    tw=df[df["日期"].between(*THIS_WEEK)]; lw=df[df["日期"].between(*LAST_WEEK)]
    return tw.groupby("VIP等级")[num].sum(), lw.groupby("VIP等级")[num].sum()


def load_vip_retention():
    res={}
    for fk,rt in [("vip_ret_chg","chg"),("vip_ret_act","act")]:
        df=pd.read_csv(FILES[fk])
        time_c="初始事件发生时间" if "初始事件发生时间" in df.columns else df.columns[0]
        df["ds"]=df[time_c].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")
        df["yyyymmdd"]=df["ds"].str.replace("-","").fillna("")
        for c in ["1日","2日","3日","7日"]:
            if c in df.columns:
                df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
        n_col="充值成功事件用户数" if "充值成功事件用户数" in df.columns else df.columns[2]
        df["n"]=pd.to_numeric(df[n_col],errors="coerce").fillna(0)
        df["vip"]=pd.to_numeric(df["vip_level"],errors="coerce") if "vip_level" in df.columns else np.nan
        ind2="指标" if "指标" in df.columns else None
        if ind2: daily=df[(df[ind2]=="留存率")&df["yyyymmdd"].notna()&df["vip"].notna()]
        else: daily=df[df["yyyymmdd"].notna()&df["vip"].notna()]
        tw_d=daily[daily["yyyymmdd"].between(*THIS_WEEK)]
        lw_d=daily[daily["yyyymmdd"].between(*LAST_WEEK)]
        def wavg(d,col):
            r={}
            if col not in d.columns: return r
            for v,g in d.groupby("vip"):
                s=g[g[col].notna()]
                if len(s): r[int(v)]=float(np.average(s[col].values,weights=s["n"].values))
            return r
        cols_avail=[c for c in ["1日","2日","3日","7日"] if c in daily.columns]
        res[rt]={"tw":{c:wavg(tw_d,c) for c in cols_avail},
                 "lw":{c:wavg(lw_d,c) for c in cols_avail}}
    return res


def load_top_users():
    dc_tw=pd.read_csv(FILES["dc_tw"]); dc_lw=pd.read_csv(FILES["dc_lw"])
    dt_tw=pd.read_csv(FILES["dt_tw"]); dt_lw=pd.read_csv(FILES["dt_lw"])
    for df in [dc_tw,dc_lw,dt_tw,dt_lw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","充提差"]:
            if c in df.columns:
                df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")
    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    tw_p=dp[dp["日期"].between(*THIS_WEEK)]
    total_c=tw_p["充值金额"].sum(); total_t=tw_p["提现金额"].sum()
    actual_cr=(total_c-total_t)/total_c*100 if total_c>0 else 0
    tot_c=dc_tw["充值金额"].sum() if "充值金额" in dc_tw.columns else 1
    tot_t=dt_tw["提款金额"].sum() if "提款金额" in dt_tw.columns else 1

    dep_t=[]
    for t in [1,10,50,100,200,500]:
        tw=dc_tw.head(t); lw=dc_lw.head(t)
        tc=tw["充值金额"].sum() if "充值金额" in tw.columns else 0
        tt=tw["提款金额"].sum() if "提款金额" in tw.columns else 0
        lc=lw["充值金额"].sum() if "充值金额" in lw.columns else 0
        lt=lw["提款金额"].sum() if "提款金额" in lw.columns else 0
        win=tw["公司输赢"].sum() if "公司输赢" in tw.columns else 0
        dep_t.append({"tier":f"Top{t}","tw_chg":tc,"lw_chg":lc,"tw_avg":tc/t,
                      "lw_avg":lc/t,"tw_cr":(tc-tt)/tc*100 if tc>0 else 0,
                      "lw_cr":(lc-lt)/lc*100 if lc>0 else 0,
                      "tw_win":win,"占全量":tc/tot_c*100 if tot_c>0 else 0})

    wdr_t=[]
    for t in [1,10,50,100,200,500]:
        tw2=dt_tw.head(t); lw2=dt_lw.head(t)
        tt2=tw2["提款金额"].sum() if "提款金额" in tw2.columns else 0
        tc2=tw2["充值金额"].sum() if "充值金额" in tw2.columns else 0
        lt2=lw2["提款金额"].sum() if "提款金额" in lw2.columns else 0
        lc2=lw2["充值金额"].sum() if "充值金额" in lw2.columns else 0
        wns=(tw2["公司输赢"]<0).sum() if "公司输赢" in tw2.columns else 0
        act_sum=tw2["活动奖励"].sum() if "活动奖励" in tw2.columns else 0
        act_pct=act_sum/(tc2+act_sum)*100 if (tc2+act_sum)>0 else 0
        excl_c=total_c-tc2; excl_t=total_t-tt2
        excl_cr=(excl_c-excl_t)/excl_c*100 if excl_c>0 else 0
        wdr_t.append({"tier":f"Top{t}","tw_tx":tt2,"lw_tx":lt2,"tw_avg":tt2/t,
                      "lw_avg":lt2/t,"tw_cr":(tc2-tt2)/tc2*100 if tc2>0 else 0,
                      "lw_cr":(lc2-lt2)/lc2*100 if lc2>0 else 0,
                      "赢家":wns,"总数":t,"赢家率":wns/t*100,"活动占比":act_pct,
                      "占全量":tt2/tot_t*100 if tot_t>0 else 0,
                      "大盘影响":excl_cr-actual_cr})
    return dep_t,wdr_t,dc_tw.head(200),dt_tw.head(20)


def load_pref():
    df=pd.read_csv(FILES["pref_tw"]); dl=pd.read_csv(FILES["pref_lw"])
    df["阶段汇总"]=df["阶段汇总"].apply(clean)
    dl["阶段汇总"]=dl["阶段汇总"].apply(clean)
    mfr_c="show_name_厂商标签id"; game_c="游戏名称"; ind_c="分析指标"
    bet=df[df[ind_c]=="投注金额"]; win=df[df[ind_c]=="公司输赢"]
    bl=dl[dl[ind_c]=="投注金额"]
    gb=bet.groupby([mfr_c,game_c])["阶段汇总"].sum().rename("本周投注")
    gw=win.groupby([mfr_c,game_c])["阶段汇总"].sum().rename("公司输赢")
    gl=bl.groupby([mfr_c,game_c])["阶段汇总"].sum().rename("上周投注")
    gu=bet.groupby([mfr_c,game_c])["账户ID"].nunique().rename("玩家数")
    g=pd.concat([gb,gw,gl,gu],axis=1).reset_index()
    g["占比"]=g["本周投注"]/g["本周投注"].sum()*100
    g["盈亏率"]=g["公司输赢"]/g["本周投注"]*100
    g["投注环比"]=(g["本周投注"]-g["上周投注"])/g["上周投注"].abs()*100
    mfr_sum=bet.groupby(mfr_c)["阶段汇总"].sum().sort_values(ascending=False)
    return g.sort_values("本周投注",ascending=False).head(20), mfr_sum


def load_games():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注人数","投注局数","投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    mfr_c="游戏厂商标签.名称"; game_c="游戏.名称"
    # 游戏报表按日有多行，先合并
    tot=tw["投注金额"].sum()
    gt=tw.groupby([mfr_c,game_c]).agg(
        投注人数=("投注人数","sum"),投注局数=("投注局数","sum"),
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
    gl=lw.groupby(game_c).agg(
        投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注","公司输赢":"lw_输赢"})
    gt["人均局数"]=gt["投注局数"]/gt["投注人数"]
    gt["人均金额"]=gt["投注金额"]/gt["投注人数"]
    gt["盈亏率"]=gt["公司输赢"]/gt["投注金额"]*100
    gt["占比"]=gt["投注金额"]/tot*100
    gt["厂商简称"]=gt[mfr_c].apply(shorten_mfr)
    gm=gt.merge(gl,on=game_c,how="left")
    gm["投注环比"]=(gm["投注金额"]-gm["lw_投注"])/gm["lw_投注"].abs()*100
    gm["lw_盈亏率"]=gm["lw_输赢"]/gm["lw_投注"]*100
    return gm.sort_values("投注金额",ascending=False).head(30)


def load_mfr():
    df=pd.read_csv(FILES["mfr"]); df=df.rename(columns={"盈利率":"盈亏率"})
    df=df[df["时间"]!="阶段汇总"].copy()
    df["时间"]=df["时间"].astype(str).str.replace("-","")
    for c in ["投注局数","投注人数","投注金额","公司输赢","盈亏率"]:
        if c in df.columns: df[c]=df[c].apply(clean)
    mfr_c="show_name_厂商标签id"
    tw=df[df["时间"].between(*THIS_WEEK)]; lw=df[df["时间"].between(*LAST_WEEK)]
    def agg(d):
        g=d.groupby(mfr_c).agg(
            投注局数=("投注局数","sum"),投注人数=("投注人数","sum"),
            投注金额=("投注金额","sum"),公司输赢=("公司输赢","sum")).reset_index()
        g["日均投注人数"]=(g["投注人数"]/7).round(0).astype(int)
        g["人均局数"]=g["投注局数"]/g["投注人数"]
        g["人均金额"]=g["投注金额"]/g["投注人数"]
        g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100
        g["占比"]=g["投注金额"]/g["投注金额"].sum()*100
        return g
    tg=agg(tw); lg=agg(lw)
    mg=tg.merge(lg[[mfr_c,"投注金额","占比","盈亏率","日均投注人数"]].rename(
        columns={"投注金额":"lw_投注","占比":"lw_占比",
                 "盈亏率":"lw_盈亏率","日均投注人数":"lw_日均人数"}),
        on=mfr_c,how="left")
    mg["投注环比"]=(mg["投注金额"]-mg["lw_投注"])/mg["lw_投注"].abs()*100
    mg["厂商显示名"]=mg[mfr_c].apply(shorten_mfr)
    mg["show_name_厂商标签id"]=mg[mfr_c]
    return mg.sort_values("投注金额",ascending=False)


def load_mfr_delta():
    tw=pd.read_excel(FILES["game_tw"]); lw=pd.read_excel(FILES["game_lw"])
    for df in [tw,lw]:
        for c in ["投注金额","公司输赢"]:
            if c in df.columns: df[c]=df[c].apply(clean)
    mfr_c="游戏厂商标签.名称"; game_c="游戏.名称"
    gt=tw.groupby([mfr_c,game_c]).agg(投注金额=("投注金额","sum")).reset_index()
    gl=lw.groupby([mfr_c,game_c]).agg(投注金额=("投注金额","sum")).reset_index().rename(
        columns={"投注金额":"lw_投注"})
    m=gt.merge(gl,on=[mfr_c,game_c],how="outer").fillna(0)
    m["delta"]=m["投注金额"]-m["lw_投注"]
    mfr_tw=tw.groupby(mfr_c)["投注金额"].sum().sort_values(ascending=False)
    res={}
    for n in mfr_tw.head(10).index:
        sub=m[m[mfr_c]==n].sort_values("delta",ascending=False)
        res[n]={"up":sub.head(1),"dn":sub.tail(1)}
    return res


def load_activities():
    df=pd.read_csv(FILES["gift"])
    for c in ["赠送金额","赠送金额.1","赠送人数","赠送人数.1"]:
        if c in df.columns: df[c]=pd.to_numeric(df[c],errors="coerce").fillna(0)
    opt_c="账变opt_code" if "账变opt_code" in df.columns else df.columns[0]
    name_c="name_账变opt_id" if "name_账变opt_id" in df.columns else df.columns[1]
    ag=df.groupby([opt_c,name_c]).agg(
        赠送金额=("赠送金额","sum"),lw_赠=("赠送金额.1","sum"),
        赠送人数=("赠送人数","sum"),lw_人数=("赠送人数.1","sum")).reset_index()
    ag["环比"]=(ag["赠送金额"]-ag["lw_赠"])/ag["lw_赠"].replace(0,np.nan).abs()*100
    tot=ag["赠送金额"].sum(); ag["占比"]=ag["赠送金额"]/tot*100
    ag["人均"]=ag["赠送金额"]/ag["赠送人数"].replace(0,np.nan)
    ag["日均_本"]=ag["赠送人数"]/7; ag["日均_上"]=ag["lw_人数"]/7
    ag["人数环比"]=(ag["赠送人数"]-ag["lw_人数"])/ag["lw_人数"].replace(0,np.nan)*100
    ag["name_账变opt_id"]=ag[name_c]
    return ag.sort_values("赠送金额",ascending=False), tot


def load_first_dep_ret():
    df=pd.read_csv(FILES["first_dep_ret"])
    df.columns=df.columns.str.strip().str.replace("\ufeff","")
    time_c="初始事件发生时间" if "初始事件发生时间" in df.columns else df.columns[0]
    df["_d"]=df[time_c].astype(str).str.extract(r"(\d{4}-\d{2}-\d{2})")[0]
    df["_yyyymmdd"]=df["_d"].str.replace("-","").fillna("")
    for c in ["当日","1日","2日","3日","4日","5日","6日","7日"]:
        if c in df.columns:
            df[c]=pd.to_numeric(df[c].astype(str).str.replace("%","").str.strip(),errors="coerce")
    user_c="账变事件用户数" if "账变事件用户数" in df.columns else df.columns[1]
    df[user_c]=pd.to_numeric(df[user_c],errors="coerce").fillna(0)
    ind_c="指标" if "指标" in df.columns else None
    stage=df[df[time_c]=="阶段值"].copy()
    daily=df[df["_yyyymmdd"].str.match(r"^\d{8}$",na=False)].copy()
    rr=daily[daily[ind_c]=="留存率"] if ind_c else daily
    nr=daily[daily[ind_c]=="留存人数"] if ind_c else daily
    tw_r=rr[rr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_r=rr[rr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    tw_n=nr[nr["_yyyymmdd"].between(*THIS_WEEK)].reset_index(drop=True)
    lw_n=nr[nr["_yyyymmdd"].between(*LAST_WEEK)].reset_index(drop=True)
    def wa(rd,nd,col):
        if col not in rd.columns or len(rd)==0: return np.nan
        rv=rd[col].to_numpy(dtype=float,na_value=np.nan); ok=~np.isnan(rv)
        if not ok.any(): return np.nan
        w=nd[user_c].to_numpy(dtype=float) if len(nd)==len(rd) else np.ones(len(rv))
        ww=w[ok]
        return float(np.nanmean(rv[ok])) if ww.sum()==0 else float(np.average(rv[ok],weights=ww))
    R={"stage":stage,"tw_users":int(tw_n[user_c].sum()),"lw_users":int(lw_n[user_c].sum())}
    for col in ["1日","2日","3日","4日","5日","6日","7日"]:
        R[f"tw_{col}"]=wa(tw_r,tw_n,col); R[f"lw_{col}"]=wa(lw_r,lw_n,col)
    dp=pd.read_excel(FILES["platform"]); dp["日期"]=dp["日期"].astype(str); dp=to_num(dp)
    R["tw_platform_fc"]=int(dp[dp["日期"].between(*THIS_WEEK)]["首充人数"].sum())
    R["lw_platform_fc"]=int(dp[dp["日期"].between(*LAST_WEEK)]["首充人数"].sum())
    return R


def load_risk(dt_raw, dc_raw):
    df=pd.read_csv(FILES["pref_tw"]); df["阶段汇总"]=df["阶段汇总"].apply(clean)
    ind_c="分析指标"; mfr_c="show_name_厂商标签id"; game_c="游戏名称"
    ba=df[df[ind_c]=="投注金额"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_投注金额")
    wa2=df[df[ind_c]=="公司输赢"].groupby("账户ID")["阶段汇总"].sum().rename("游戏_输赢")
    tg=(df[df[ind_c]=="投注金额"].sort_values("阶段汇总",ascending=False)
        .groupby("账户ID").first()[[mfr_c,game_c,"阶段汇总"]]
        .rename(columns={mfr_c:"主玩厂商",game_c:"主玩游戏","阶段汇总":"主游投注"}).reset_index())
    ug=pd.concat([ba,wa2],axis=1).reset_index().merge(tg,on="账户ID",how="left")
    top500=dt_raw.head(500).copy(); top500=top500.merge(ug,on="账户ID",how="left")
    top500["赢家"]=top500["公司输赢"]<0
    # 投充比 = 游戏_投注金额（来自游戏偏好文件）/ 充值金额
    top500["投充比"]=(top500["游戏_投注金额"]/top500["充值金额"].replace(0,np.nan)
                     if "游戏_投注金额" in top500.columns else np.nan)
    hr=top500[top500["公司输赢"]<-10000].sort_values("公司输赢") if "公司输赢" in top500.columns else pd.DataFrame()
    sp=(top500[((top500["充值金额"]==0)&(top500["提款金额"]>1000))|(top500["投充比"]>150)]
        .sort_values("提款金额",ascending=False) if "投充比" in top500.columns else pd.DataFrame())
    bg=df[df[ind_c]=="投注金额"].groupby([mfr_c,game_c])["阶段汇总"].sum().rename("投注金额").reset_index()
    wg=df[df[ind_c]=="公司输赢"].groupby([mfr_c,game_c])["阶段汇总"].sum().rename("公司输赢").reset_index()
    ug2=df[df[ind_c]=="投注金额"].groupby([mfr_c,game_c])["账户ID"].nunique().rename("玩家数").reset_index()
    wn=(df[df[ind_c]=="公司输赢"].groupby([mfr_c,game_c])
        .apply(lambda x:(x["阶段汇总"]<0).sum()).rename("赢家数").reset_index())
    g=bg.merge(wg,on=[mfr_c,game_c],how="left").merge(ug2,on=[mfr_c,game_c],how="left").merge(wn,on=[mfr_c,game_c],how="left")
    g["盈亏率"]=g["公司输赢"]/g["投注金额"]*100
    g["赢家率"]=g["赢家数"]/g["玩家数"]*100
    g["人均投注额"]=g["投注金额"]/g["玩家数"]
    g["show_name_厂商标签id"]=g[mfr_c]; g["游戏名称"]=g[game_c]
    rg=g[(g["公司输赢"]<-5000)|((g["盈亏率"]<-10)&(g["投注金额"]>5000))].sort_values("公司输赢")
    return hr,sp,g.sort_values("投注金额",ascending=False).head(20),rg,top500


# ══════════════════════════════════════════════
# 图表
# ══════════════════════════════════════════════
SPLIT=7

def _vline(ax):
    ax.axvline(SPLIT-.5,color="#94a3b8",ls="--",lw=1,alpha=.7)
    trans=ax.get_xaxis_transform()
    ax.text(SPLIT-4,.93,"上周",ha="center",fontsize=7,color="#64748b",transform=trans)
    ax.text(SPLIT+3,.93,"本周",ha="center",fontsize=7,color="#1d4ed8",transform=trans)

def _sp(a):
    a.spines["top"].set_visible(False); a.spines["right"].set_visible(False)
    a.grid(axis="y",alpha=.25)

def chart_trend(trend):
    fig=plt.figure(figsize=(16,10),facecolor="white")
    gs=gridspec.GridSpec(2,2,figure=fig,hspace=.45,wspace=.3)
    dates=trend["dates"]; x=range(len(dates))
    ax=fig.add_subplot(gs[0,0])
    ax.bar(x,[v/10000 for v in trend["充值"]],color=["#bfdbfe"]*SPLIT+["#1d4ed8"]*SPLIT,width=.7,label="充值")
    ax.plot(x,[v/10000 for v in trend["提现"]],"-o",color="#dc2626",lw=1.5,ms=3,label="提现")
    ax.set_title("充值 vs 提现（万USD）",fontsize=9,fontweight="bold")
    ax.set_xticks(list(x)); ax.set_xticklabels(dates,rotation=45,fontsize=7)
    ax.legend(fontsize=7,loc="upper left"); _sp(ax); _vline(ax)
    ax2=fig.add_subplot(gs[0,1])
    ax2.bar(x,[v/10000 for v in trend["公司输赢"]],color=["#bbf7d0"]*SPLIT+["#059669"]*SPLIT,width=.7)
    ax2.set_title("公司输赢（万USD）",fontsize=9,fontweight="bold")
    ax2.set_xticks(list(x)); ax2.set_xticklabels(dates,rotation=45,fontsize=7)
    _sp(ax2); _vline(ax2)
    ax3=fig.add_subplot(gs[1,0])
    ax3.bar(x,trend["首充"],color=["#c7d2fe"]*SPLIT+["#6366f1"]*SPLIT,width=.7,label="首充人数")
    ax3r=ax3.twinx()
    ax3r.plot(x,trend["注册"],"--D",color="#d97706",lw=1.5,ms=3,label="注册人数")
    ax3.set_title("首充人数（柱）vs 注册人数（线）",fontsize=9,fontweight="bold")
    ax3.set_xticks(list(x)); ax3.set_xticklabels(dates,rotation=45,fontsize=7)
    l1,lb1=ax3.get_legend_handles_labels(); l2,lb2=ax3r.get_legend_handles_labels()
    ax3.legend(l1+l2,lb1+lb2,fontsize=7,loc="upper left")
    ax3.spines["top"].set_visible(False); ax3r.spines["top"].set_visible(False); ax3.grid(axis="y",alpha=.25)
    ax3.axvline(SPLIT-.5,color="#94a3b8",ls="--",lw=1,alpha=.7)
    trans3=ax3.get_xaxis_transform()
    ax3.text(SPLIT-4,.93,"上周",ha="center",fontsize=7,color="#64748b",transform=trans3)
    ax3.text(SPLIT+3,.93,"本周",ha="center",fontsize=7,color="#1d4ed8",transform=trans3)
    ax4=fig.add_subplot(gs[1,1])
    ax4.bar(x,trend["充提差比"],color=["#e9d5ff"]*SPLIT+["#7c3aed"]*SPLIT,width=.7)
    tm_=np.mean(trend["充提差比"][SPLIT:]); lm_=np.mean(trend["充提差比"][:SPLIT])
    ax4.axhline(tm_,color="#7c3aed",ls=":",lw=1.5); ax4.axhline(lm_,color="#94a3b8",ls=":",lw=1.2)
    bbox_=dict(boxstyle="round,pad=0.15",fc="white",ec="none",alpha=0.85)
    n=len(dates)
    ax4.text(n-1.,tm_+.3,f"本周均{tm_:.1f}%",fontsize=7,color="#7c3aed",ha="right",clip_on=False,fontweight="bold",bbox=bbox_)
    ax4.text(0.,lm_+.3,f"上周均{lm_:.1f}%",fontsize=7,color="#64748b",ha="left",clip_on=False,fontweight="bold",bbox=bbox_)
    ax4.set_title("充提差率（%）",fontsize=9,fontweight="bold")
    ax4.set_xticks(list(x)); ax4.set_xticklabels(dates,rotation=45,fontsize=7)
    ax4.set_ylim(0,max(trend["充提差比"])*1.2)
    _sp(ax4); _vline(ax4)
    fig.suptitle("近14日大盘核心指标趋势",fontsize=11,fontweight="bold",y=1.01)
    plt.tight_layout(rect=[0,0,1,.98]); return fig_img(fig,16,11)

def chart_ret_weekly(weeks):
    fig,ax=plt.subplots(figsize=(14,6),facecolor="white")
    cols=[("1日","次留","#1d4ed8"),("2日","3留","#059669"),("6日","7留","#7c3aed")]
    x=np.arange(len(weeks)); w=.25
    for i,(col,lbl,clr) in enumerate(cols):
        vals=[wk.get(col,np.nan) for wk in weeks]
        bars=ax.bar(x+i*w-w,vals,w,label=lbl,color=clr,alpha=.85)
        for bar,v in zip(bars,vals):
            if not (isinstance(v,float) and np.isnan(v)):
                ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.3,f"{v:.1f}%",ha="center",fontsize=6.5,color=clr)
        tgt=RET_TARGETS.get(lbl)
        if tgt: ax.axhline(tgt,color=clr,ls="--",lw=1.2,alpha=.55)
    ax.set_xticks(x); ax.set_xticklabels([wk["week"] for wk in weeks],fontsize=8)
    ax.set_title("首充用户充值留存率 近4周对比（含目标虚线）",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8,loc="upper left"); ax.axvline(x=2.5,color="#ef4444",ls="--",lw=1,alpha=.5)
    ax.grid(axis="y",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,6)

def chart_vip(tw_v,lw_v):
    vips=sorted(set(tw_v.index)|set(lw_v.index)); labels=[f"V{int(v)}" for v in vips]
    x=np.arange(len(vips)); w=.35
    fig,(ax1,ax2,ax3)=plt.subplots(1,3,figsize=(16,5.5),facecolor="white")
    tc=[tw_v.loc[v,"充值金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lc=[lw_v.loc[v,"充值金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax1.bar(x-w/2,tc,w,label="本周",color="#1d4ed8",alpha=.85); ax1.bar(x+w/2,lc,w,label="上周",color="#93c5fd",alpha=.7)
    ax1.set_title("各VIP充值金额（万USD）",fontsize=9,fontweight="bold"); ax1.set_xticks(x); ax1.set_xticklabels(labels,rotation=45,fontsize=7); ax1.legend(fontsize=7); _sp(ax1)
    tb=[tw_v.loc[v,"投注金额"]/10000 if v in tw_v.index else 0 for v in vips]
    lb=[lw_v.loc[v,"投注金额"]/10000 if v in lw_v.index else 0 for v in vips]
    ax2.bar(x-w/2,tb,w,label="本周",color="#059669",alpha=.85); ax2.bar(x+w/2,lb,w,label="上周",color="#6ee7b7",alpha=.7)
    ax2.set_title("各VIP投注金额（万USD）",fontsize=9,fontweight="bold"); ax2.set_xticks(x); ax2.set_xticklabels(labels,rotation=45,fontsize=7); ax2.legend(fontsize=7); _sp(ax2)
    chg=[(tw_v.loc[v,"充值金额"]-lw_v.loc[v,"充值金额"])/lw_v.loc[v,"充值金额"]*100
         if v in tw_v.index and v in lw_v.index and lw_v.loc[v,"充值金额"]>0 else 0 for v in vips]
    ax3.bar(x,chg,color=["#059669" if c>=0 else "#dc2626" for c in chg],alpha=.85,width=.6)
    ax3.axhline(0,color="black",lw=.8)
    ax3.set_title("各VIP充值金额环比（%）",fontsize=9,fontweight="bold"); ax3.set_xticks(x); ax3.set_xticklabels(labels,rotation=45,fontsize=7); _sp(ax3)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_vip_ret(vr):
    vips=list(range(1,13)); labels=[f"V{v}" for v in vips]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,5.5),facecolor="white")
    x=np.arange(len(labels)); w=.28
    for ax,rt,t in [(ax1,"chg","充值→充值 留存率（%）"),(ax2,"act","充值→活跃 留存率（%）")]:
        c1="1日"; c3="3日" if "3日" in vr[rt]["tw"] else "2日"
        clr=("#1d4ed8","#93c5fd","#059669") if rt=="chg" else ("#7c3aed","#c4b5fd","#d97706")
        tw1=[vr[rt]["tw"].get(c1,{}).get(v,0) for v in vips]
        lw1=[vr[rt]["lw"].get(c1,{}).get(v,0) for v in vips]
        tw3=[vr[rt]["tw"].get(c3,{}).get(v,0) for v in vips]
        ax.bar(x-w,tw1,w,label=f"次日(本周)",color=clr[0],alpha=.85)
        ax.bar(x,lw1,w,label=f"次日(上周)",color=clr[1],alpha=.7)
        ax.bar(x+w,tw3,w,label=f"3日(本周)",color=clr[2],alpha=.75)
        ax.set_title(t,fontsize=10,fontweight="bold"); ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8)
        ax.legend(fontsize=7.5,loc="upper left"); _sp(ax)
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v2,_:f"{v2:.0f}%"))
    plt.suptitle("VIP各等级充值留存率",fontsize=11,fontweight="bold",y=1.02)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_mfr_combo(mfr):
    top10=mfr.head(10)
    names=top10["厂商显示名"].tolist()
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    sh=top10["占比"].tolist(); ls=top10["lw_占比"].fillna(0).tolist()
    bars=ax1.bar(names,sh,color=["#059669" if c>=l else "#dc2626" for c,l in zip(sh,ls)],alpha=.85,width=.6)
    ax1.plot(names,ls,"--o",color="#64748b",lw=1.5,ms=5,zorder=5)
    for bar,s,ic in zip(bars,sh,top10["投注环比"].tolist()):
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.2,f"{s:.1f}%",ha="center",fontsize=7)
        ax1.text(bar.get_x()+bar.get_width()/2,bar.get_height()/2,f"{ic:+.0f}%",ha="center",fontsize=6.5,color="white",fontweight="bold")
    ax1.set_title("Top10厂商 投注份额",fontsize=9,fontweight="bold"); ax1.set_xticklabels(names,rotation=30,ha="right",fontsize=7.5)
    ax1.legend(handles=[mpatches.Patch(color="#059669",label="份额增长"),mpatches.Patch(color="#dc2626",label="份额下降"),plt.Line2D([0],[0],color="#64748b",ls="--",marker="o",ms=4,label="上周份额")],fontsize=7.5)
    _sp(ax1)
    x=np.arange(len(names)); w=.35
    ax2.bar(x-w/2,top10["盈亏率"].tolist(),w,label="本周",color=["#059669" if v>0 else "#dc2626" for v in top10["盈亏率"]],alpha=.85)
    ax2.bar(x+w/2,top10["lw_盈亏率"].fillna(0).tolist(),w,label="上周",color="#93c5fd",alpha=.7)
    ax2.axhline(0,color="black",lw=.8); ax2.set_title("Top10厂商 盈亏率对比（%）",fontsize=9,fontweight="bold")
    ax2.set_xticks(x); ax2.set_xticklabels([n[:8] for n in names],rotation=30,ha="right",fontsize=7.5)
    ax2.legend(fontsize=7.5); _sp(ax2)
    ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v3,_:f"{v3:.1f}%"))
    plt.tight_layout(); return fig_img(fig,16,6.5)

def chart_top30(top30):
    game_c="游戏.名称" if "游戏.名称" in top30.columns else top30.columns[1]
    t15=top30.head(15)
    names=[str(r[game_c])[:18] for _,r in t15.iterrows()]
    bets=[r["投注金额"]/10000 for _,r in t15.iterrows()]
    lw_b=[r["lw_投注"]/10000 if not pd.isna(r.get("lw_投注",np.nan)) else 0 for _,r in t15.iterrows()]
    x=np.arange(len(names)); w=.35; fig,ax=plt.subplots(figsize=(14,7),facecolor="white")
    ax.barh(x+w/2,bets,w,label="本周",color="#1d4ed8",alpha=.85)
    ax.barh(x-w/2,lw_b,w,label="上周",color="#93c5fd",alpha=.7)
    ax.set_yticks(x); ax.set_yticklabels(names,fontsize=8)
    ax.set_title("Top15游戏 投注金额（万USD）",fontsize=9,fontweight="bold")
    ax.legend(fontsize=8); ax.grid(axis="x",alpha=.3); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,14,7)

def chart_pref(pg,ma):
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    names=ma.index[:8].tolist(); vals=ma.values[:8].tolist(); tot=sum(vals)
    pcts=[v/tot*100 for v in vals]; clrs=CHART_COLORS[:len(names)]
    wedges,_,auts=ax1.pie(vals,labels=None,colors=clrs,autopct=lambda p:f"{p:.1f}%" if p>4 else "",startangle=90,pctdistance=.78,wedgeprops=dict(edgecolor="white",linewidth=1.5))
    for at in auts: at.set_fontsize(8); at.set_color("white"); at.set_fontweight("bold")
    ax1.legend(wedges,[f"{str(n)[:12]}({p:.1f}%)" for n,p in zip(names,pcts)],loc="lower left",fontsize=7.5,bbox_to_anchor=(-.05,-.18))
    ax1.set_title("Top500提款用户 厂商偏好",fontsize=9,fontweight="bold")
    game_c6="游戏名称" if "游戏名称" in pg.columns else pg.columns[1]
    mfr_c5="show_name_厂商标签id" if "show_name_厂商标签id" in pg.columns else pg.columns[0]
    t12=pg.head(12)
    gn=[f"[{str(r[mfr_c5])[:4]}]\n{str(r[game_c6])}"[:22] for _,r in t12.iterrows()]
    gv=[r["本周投注"]/10000 for _,r in t12.iterrows()]
    gc=["#dc2626" if r["盈亏率"]<0 else "#1d4ed8" for _,r in t12.iterrows()]
    ax2.barh(range(len(gn))[::-1],gv,color=gc[::-1],alpha=.85)
    ax2.set_yticks(range(len(gn))); ax2.set_yticklabels(gn,fontsize=7.5)
    ax2.set_title("偏好游戏Top12（红=平台亏损）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v4,_:f"{v4:.0f}万"))
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_activities(act):
    name_c="name_账变opt_id" if "name_账变opt_id" in act.columns else act.columns[1]
    t10=act.head(10); names=[str(r[name_c])[:12] for _,r in t10.iterrows()]
    vals=[r["赠送金额"]/10000 for _,r in t10.iterrows()]
    lw=[r["lw_赠"]/10000 for _,r in t10.iterrows()]
    envs=[r["环比"] for _,r in t10.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    x=np.arange(len(names)); w=.35
    ax1.barh(x+w/2,vals[::-1],w,label="本周",color="#1d4ed8",alpha=.85)
    ax1.barh(x-w/2,lw[::-1],w,label="上周",color="#93c5fd",alpha=.7)
    ax1.set_yticks(x); ax1.set_yticklabels(names[::-1],fontsize=8)
    ax1.set_title("Top10活动 赠送金额（万USD）",fontsize=9,fontweight="bold")
    ax1.legend(fontsize=8); ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    es=[v if not pd.isna(v) else 0 for v in envs]
    ax2.barh(range(len(names)),es[::-1],color=["#059669" if v>0 else "#dc2626" for v in es[::-1]],alpha=.85)
    ax2.axvline(0,color="black",lw=.8); ax2.set_yticks(range(len(names))); ax2.set_yticklabels(names[::-1],fontsize=8)
    ax2.set_title("Top10活动 环比变化（%）",fontsize=9,fontweight="bold")
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v5,_:f"{v5:+.0f}%"))
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,16,6)

def chart_first_dep_ret(R):
    lv=[R.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    tv=[R.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    fig,ax=plt.subplots(figsize=(12,5),facecolor="white"); x=np.arange(7)
    ax.plot(x,lv,"-o",color="#93c5fd",lw=2,ms=6,label=f"上周（{LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}）")
    ax.plot(x,tv,"-o",color="#1d4ed8",lw=2,ms=6,label=f"本周（{THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}）")
    days=["D1","D2","D3","D4","D5","D6","D7"]
    for i,(lval,tval) in enumerate(zip(lv,tv)):
        if lval is not None and not (isinstance(lval,float) and np.isnan(lval)):
            ax.text(i,lval+.4,f"{lval:.1f}%",ha="center",fontsize=7.5,color="#64748b")
        if tval is not None and not (isinstance(tval,float) and np.isnan(tval)):
            ax.text(i,tval-1.2,f"{tval:.1f}%",ha="center",fontsize=7.5,color="#1d4ed8",fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(days,fontsize=9)
    ax.set_title("首次充值活动用户 充值留存趋势",fontsize=10,fontweight="bold")
    ax.legend(fontsize=8.5,loc="upper right"); ax.grid(axis="y",alpha=.3)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v6,_:f"{v6:.0f}%"))
    plt.tight_layout(); return fig_img(fig,13,5.5)

def chart_risk_scatter(top500):
    v=top500[top500["投充比"].notna()&(top500["投充比"]<200)].copy() if "投充比" in top500.columns else top500.head(0)
    fig,ax=plt.subplots(figsize=(14,5.5),facecolor="white")
    if len(v)>0:
        ax.scatter(v["投充比"],v["公司输赢"],
                   c=["#dc2626" if x<0 else "#059669" for x in v["公司输赢"]],
                   s=[min(abs(x)/50+20,200) for x in v["公司输赢"]],alpha=.55,edgecolors="none")
    ax.axhline(0,color="black",lw=.8,ls="--")
    ax.set_xlabel("投充比（倍）",fontsize=8); ax.set_ylabel("公司输赢（USD）",fontsize=8)
    ax.set_title("Top500提款用户：投充比 vs 公司输赢",fontsize=9,fontweight="bold")
    ax.legend(handles=[mpatches.Patch(color="#dc2626",alpha=.6,label="平台输钱"),
                       mpatches.Patch(color="#059669",alpha=.6,label="平台赢钱")],fontsize=8,loc="upper right")
    ax.grid(alpha=.25); ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    plt.tight_layout(); return fig_img(fig,15,6)

def chart_risk_games(rg):
    t12=rg.head(12); game_c="游戏名称"
    names=[str(r[game_c])[:16] for _,r in t12.iterrows()]
    losses=[abs(r["公司输赢"]) for _,r in t12.iterrows()]
    rates=[r["盈亏率"] for _,r in t12.iterrows()]
    fig,(ax1,ax2)=plt.subplots(1,2,figsize=(16,6),facecolor="white")
    ax1.barh(names[::-1],losses[::-1],color="#dc2626",alpha=.85)
    ax1.set_title("公司净亏损额（USD）",fontsize=9,fontweight="bold")
    ax1.grid(axis="x",alpha=.3); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
    ax1.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v7,_:f"${v7/1000:.0f}k")); ax1.tick_params(axis="y",labelsize=7.5)
    ax2.barh(names[::-1],rates[::-1],color=["#7c3aed" if r>-10 else "#dc2626" for r in rates[::-1]],alpha=.85)
    ax2.axvline(0,color="black",lw=.8); ax2.set_title("盈亏率（%）",fontsize=9,fontweight="bold")
    ax2.grid(axis="x",alpha=.3); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
    ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v8,_:f"{v8:.0f}%")); ax2.tick_params(axis="y",labelsize=7.5)
    plt.tight_layout(); return fig_img(fig,16,6.5)


# ══════════════════════════════════════════════
# 章节构建
# ══════════════════════════════════════════════
def build_overview(K, trend, weekly_ret, dash_ret):
    full=PW-2*MARGIN; S=[sec_title("一、大盘核心数据")]
    kpis=[
        ("充值金额",   f"{K['tw_充值金额']/10000:.1f}万",   f"{K['lw_充值金额']/10000:.1f}万",   K["pct_充值金额"],   True),
        ("提现金额",   f"{K['tw_提现金额']/10000:.1f}万",   f"{K['lw_提现金额']/10000:.1f}万",   K["pct_提现金额"],   False),
        ("充提差",     f"{K['tw_充提差']/10000:.1f}万",     f"{K['lw_充提差']/10000:.1f}万",     K["pct_充提差"],     True),
        ("充提差率",   f"{K['tw_充提差比']:.2f}%",          f"{K['lw_充提差比']:.2f}%",          K["pct_充提差比"],   True),
        ("公司输赢",   f"{K['tw_公司输赢']/10000:.1f}万",   f"{K['lw_公司输赢']/10000:.1f}万",   K["pct_公司输赢"],   True),
        ("盈亏率",     f"{K['tw_盈亏率']:.3f}%",            f"{K['lw_盈亏率']:.3f}%",            K["pct_盈亏率"],     True),
        ("注册人数",   f"{int(K['tw_注册人数']):,}",        f"{int(K['lw_注册人数']):,}",        K["pct_注册人数"],   True),
        ("首充人数",   f"{int(K['tw_首充人数']):,}",        f"{int(K['lw_首充人数']):,}",        K["pct_首充人数"],   True),
        ("日均活跃",   f"{K['tw_活跃人数']/7/10000:.1f}万", f"{K['lw_活跃人数']/7/10000:.1f}万", K["pct_活跃人数"],   True),
        ("投注金额",   f"{K['tw_投注金额']/10000:.0f}万",   f"{K['lw_投注金额']/10000:.0f}万",   K["pct_投注金额"],   True),
        ("全量ARPPU",  f"${K['tw_全量Arppu']:.2f}",         f"${K['lw_全量Arppu']:.2f}",         K["pct_全量Arppu"],  True),
        ("老用户ARPPU",f"${K['tw_老用户ARPPU']:.2f}",       f"${K['lw_老用户ARPPU']:.2f}",       K["pct_老用户ARPPU"],True),
        ("首充ARPPU",  f"${K['tw_首充Arppu']:.2f}",         f"${K['lw_首充Arppu']:.2f}",         K["pct_首充Arppu"],  True),
        ("总赠送金额", f"{K['tw_总赠送金额']/10000:.1f}万", f"{K['lw_总赠送金额']/10000:.1f}万", K["pct_总赠送金额"], False),
        ("赠送/充值比",f"{K['tw_赠送充值比']:.2f}%",        f"{K['lw_赠送充值比']:.2f}%",        K["pct_赠送充值比"], False),
        ("首充转化率", f"{K['tw_首充转化率']:.1f}%",        f"{K['lw_首充转化率']:.1f}%",        K["pct_首充转化率"], True),
        ("首充次日留存",f"{K['tw_首充次日复充率']:.1f}%",   f"{K['lw_首充次日复充率']:.1f}%",   K["pct_首充次日复充率"],True),
        ("推广消耗(日均)",f"{K['tw_真实消耗_日均']/10000:.2f}万",
                          f"{K['lw_真实消耗_日均']/10000:.2f}万",K["pct_真实消耗"],False),
        ("充提差ROI\n(日均充提差/日均消耗)",
         f"{K['tw_充提差ROI']:.2f}x",f"{K['lw_充提差ROI']:.2f}x",K["pct_充提差ROI"],True),
    ]
    S.append(kpi_card4(kpis,cols=4)); S.append(Spacer(1,8))
    cr_d=K["tw_充提差比"]-K["lw_充提差比"]; rd=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(insight_box([
        f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%）；推广日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（本周{K['tw_真实消耗_有效天']}天有效，{K['pct_真实消耗']:+.1f}%）。",
        f"充提差率{K['tw_充提差比']:.2f}%（上周{K['lw_充提差比']:.2f}%，{cr_d:+.2f}pp）；盈亏率{K['tw_盈亏率']:.3f}%（上周{K['lw_盈亏率']:.3f}%）。",
        f"首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），转化率{K['tw_首充转化率']:.1f}%，首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%）。",
        f"首充次日充值留存{K['tw_首充次日复充率']:.1f}%（上周{K['lw_首充次日复充率']:.1f}%，{rd:+.1f}pp）。",
    ]))
    S.append(sub_title("近14日核心指标趋势")); S.append(chart_trend(trend)); S.append(Spacer(1,8))
    S.append(sub_title("大盘首充留存 vs 目标对比"))
    specs=[("次留","tw_nd","lw_nd","tw_nd_end"),("3留","tw_td","lw_td","tw_td_end"),("7留","tw_sd","lw_sd","tw_sd_end")]
    rh=["留存类型","数据截止","本周实际","上周实际","周环比(pp)","目标","vs目标(pp)"]
    rrows=[]
    for rtype,tw_k,lw_k,ek in specs:
        tv=dash_ret.get(tw_k,np.nan); lv=dash_ret.get(lw_k,np.nan)
        el=fmt_lbl(dash_ret.get(ek)); tgt=RET_TARGETS.get(rtype,np.nan)
        wpp=(tv-lv) if not(np.isnan(tv) or np.isnan(lv)) else np.nan
        dpp=(tv-tgt) if not(np.isnan(tv) or np.isnan(tgt)) else np.nan
        tc=C_GREEN if(not np.isnan(tv) and not np.isnan(tgt) and tv>=tgt) else C_RED
        rrows.append([cell(rtype,True,C_DARK),cell(el,False,C_GRAY,TA_CENTER),
                      cell(fret(tv),False,tc,TA_RIGHT),cell(fret(lv),False,C_GRAY,TA_RIGHT),
                      cell(f"{wpp:+.1f}pp" if not np.isnan(wpp) else "-",False,C_GREEN if(not np.isnan(wpp) and wpp>=0) else C_RED,TA_RIGHT),
                      cell(f"{tgt:.0f}%" if not np.isnan(tgt) else "-",True,C_TARGET,TA_CENTER),
                      cell(f"{dpp:+.1f}pp" if not np.isnan(dpp) else "-",True,C_GREEN if(not np.isnan(dpp) and dpp>=0) else C_RED,TA_RIGHT)])
    S.append(dtable(rh,rrows,[full*x for x in [0.16,0.12,0.14,0.14,0.14,0.12,0.14]],fsize=8,
                    extra_style=[("BACKGROUND",(5,1),(5,-1),colors.HexColor("#eff6ff"))]))
    S.append(Spacer(1,4))
    S.append(P(f"本周截止：次留~{fmt_lbl(dash_ret.get('tw_nd_end'))}，3留~{fmt_lbl(dash_ret.get('tw_td_end'))}，7留~{fmt_lbl(dash_ret.get('tw_sd_end'))}",7,False,C_GRAY))
    S.append(Spacer(1,8))
    S.append(sub_title("首充用户充值留存 — 近4周对比（含目标线）"))
    S.append(chart_ret_weekly(weekly_ret)); S.append(Spacer(1,4))
    tw_=weekly_ret[-1]; lw_=weekly_ret[-2]
    d1=tw_.get("1日",0) or 0; d2=tw_.get("2日",0) or 0; d6=tw_.get("6日",0) or 0
    ld1=lw_.get("1日",0) or 0; ld6=lw_.get("6日",0) or 0
    S.append(insight_box([
        f"本周首充次日留存{d1:.1f}%，较上周{d1-ld1:+.1f}pp；3留{d2:.1f}%，7留{d6:.1f}%。",
        f"注：本周7日留存因截止日期不完整，以上周7留{ld6:.1f}%作为参考基准。",
    ],clr=C_AMBER))
    return S


def build_agents(agents, agent_ret, ends):
    full=PW-2*MARGIN; S=[sec_title("二、总代分析")]
    tw_nd=fmt_lbl(ends.get("tw_nd_end")); lw_nd=fmt_lbl(ends.get("lw_nd_end"))
    tw_3d=fmt_lbl(ends.get("tw_3d_end")); lw_3d=fmt_lbl(ends.get("lw_3d_end"))
    lw_7d=fmt_lbl(ends.get("lw_7d_end"))
    S.append(sub_title("全量总代表现（本周 vs 上周，按充值金额排序）"))
    headers=["ID","总代名称","注册(环比)","首充\n人数","充值\n(万)","充提差率\n(差值pp)",
             "消耗(万)","1级首充\n成本(本/上)",
             f"次留(本/上)\n~{tw_nd}/{lw_nd}",
             f"3留(本/上)\n~{tw_3d}/{lw_3d}",
             f"7留上周\n~{lw_7d}"]
    rows=[]
    for _,r in agents.iterrows():
        aid=int(r["总代.ID"]) if not pd.isna(r.get("总代.ID",np.nan)) else "-"
        rname=str(r["总代.名称"])
        cr=r.get("充提差率",0) or 0; lcr=r.get("lw_充提差率",0) or 0; cr_d=cr-lcr
        cost=(r.get("总消耗",0) or 0)/10000
        fcc=r.get("一级首充成本",0) or 0; lfc=r.get("lw_fc_cost",0) or 0
        reg=int(r.get("注册人数",0) or 0); reg_c=r.get("注册环比",0) or 0
        cr_clr=C_GREEN if cr>=CR_HIGH else (C_RED if cr<CR_LOW else C_DARK)
        d=agent_ret.get(rname,{})
        tw_nd_v=d.get("tw_nd",np.nan); lw_nd_v=d.get("lw_nd",np.nan)
        tw_3d_v=d.get("tw_3d",np.nan); lw_3d_v=d.get("lw_3d",np.nan)
        lw_7d_v=d.get("lw_7d",np.nan)
        def _fp(tv,lv):
            if not np.isnan(tv) and not np.isnan(lv): return f"{tv:.1f}%/{lv:.1f}%",C_GREEN if tv>=lv else C_RED
            if not np.isnan(tv): return f"{tv:.1f}%/-",C_DARK
            return "-",C_GRAY
        nd_s,nd_clr=_fp(tw_nd_v,lw_nd_v); td_s,td_clr=_fp(tw_3d_v,lw_3d_v)
        rows.append([
            cell(str(aid),False,C_GRAY,TA_CENTER),cell(rname[:14],True,C_DARK),
            cell(f"{reg:,}/{'+' if reg_c>=0 else ''}{reg_c:.0f}%",False,C_GREEN if reg_c>=0 else C_RED,TA_RIGHT),
            cell(f"{int(r.get('首充人数',0)):,}",False,C_DARK,TA_RIGHT),
            cell(f"{r.get('充值金额',0)/10000:.0f}",False,C_DARK,TA_RIGHT),
            cell(f"{cr:.1f}%/{'+' if cr_d>=0 else ''}{cr_d:.1f}pp",False,cr_clr,TA_RIGHT),
            cell(f"{cost:.1f}" if cost>0 else "-",False,C_DARK,TA_RIGHT),
            cell(f"${fcc:.0f}/${lfc:.0f}" if fcc>0 else "-",False,C_DARK,TA_RIGHT),
            cell(nd_s,False,nd_clr,TA_RIGHT),cell(td_s,False,td_clr,TA_RIGHT),
            cell(fret(lw_7d_v),False,C_DARK,TA_RIGHT),
        ])
    cw=[full*x for x in [0.04,0.14,0.10,0.06,0.05,0.11,0.05,0.10,0.11,0.11,0.08]]
    S.append(dtable(headers,rows,cw,fsize=6.2)); S.append(Spacer(1,4))
    S.append(P(f"★ 充提差率≥{CR_HIGH}%绿，<{CR_LOW}%红。",6.5,False,C_GRAY)); S.append(Spacer(1,6))
    if len(agents)>0:
        t1=agents.iloc[0]; t1_cr_d=t1.get("充提差率",0)-(t1.get("lw_充提差率",0) or 0)
        ins=[f"体量最大总代「{str(t1['总代.名称'])[:12]}」充值{t1.get('充值金额',0)/10000:.0f}万，充提差率{t1.get('充提差率',0):.1f}%（{t1_cr_d:+.1f}pp），注册{int(t1.get('注册人数',0)):,}人（{t1.get('注册环比',0):+.0f}%）。"]
        best=agents[agents["充值金额"]>10000].nlargest(1,"充提差率") if "充提差率" in agents.columns else pd.DataFrame()
        if len(best)>0:
            b=best.iloc[0]; ins.append(f"充提差率最优渠道「{str(b['总代.名称'])[:12]}」达{b.get('充提差率',0):.1f}%，充值规模{b.get('充值金额',0)/10000:.0f}万。")
        S.append(insight_box(ins))
    return S


def build_users(tw_v, lw_v, dep_t, wdr_t, dc_tw, dt_tw):
    full=PW-2*MARGIN; S=[sec_title("三、用户分析")]
    S.append(sub_title("VIP等级分层分析")); S.append(chart_vip(tw_v,lw_v)); S.append(Spacer(1,4))
    tot=tw_v["充值金额"].sum()
    hvs=sum(tw_v.loc[v,"充值金额"] for v in [9,10,11,12] if v in tw_v.index)
    S.append(insight_box([
        f"VIP9-12高价值层合计贡献充值{hvs/tot*100:.1f}%，高端用户付费意愿持续强劲。",
        f"VIP1基础层活跃{int(tw_v.loc[1,'活跃人数']):,}人（上周{int(lw_v.loc[1,'活跃人数']):,}），是拉新政策直接反映。" if 1 in tw_v.index and 1 in lw_v.index else "VIP基础层数据不可用。"
    ]))
    vr=load_vip_retention()
    S.append(sub_title("VIP各等级充值留存率")); S.append(chart_vip_ret(vr)); S.append(Spacer(1,4))
    v9_act=vr.get("act",{}).get("tw",{}).get("1日",{}).get(9,0)
    v10_act=vr.get("act",{}).get("tw",{}).get("1日",{}).get(10,0)
    v10_chg_tw=vr.get("chg",{}).get("tw",{}).get("1日",{}).get(10,0)
    v10_chg_lw=vr.get("chg",{}).get("lw",{}).get("1日",{}).get(10,0)
    S.append(insight_box([
        f"充值→活跃次日留存：VIP9达{v9_act:.1f}%，VIP10达{v10_act:.1f}%。",
        f"充值→充值次日留存：VIP10达{v10_chg_tw:.1f}%（上周{v10_chg_lw:.1f}%）。",
    ])); S.append(Spacer(1,8))
    S.append(sub_title("头部充值用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h=["分层","本周充值","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","占全量%","公司输赢"]
    rows=[]
    for d in dep_t:
        rows.append([cell(d["tier"],True),cell(f"${d['tw_chg']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(pct(d["tw_avg"],d["lw_avg"])),cell(f"{d['tw_cr']:.1f}%",False,gclr(d["tw_cr"],0),TA_RIGHT),
                     cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                     cell(f"${d['tw_win']/10000:.1f}万",False,gclr(d["tw_win"]),TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.08,0.12,0.11,0.11,0.09,0.1,0.1,0.09,0.1]])); S.append(Spacer(1,6))
    S.append(insight_box([
        f"Top10充值用户人均${dep_t[1]['tw_avg']:,.0f}（{pct(dep_t[1]['tw_avg'],dep_t[1]['lw_avg']):+.1f}%），充提差率{dep_t[1]['tw_cr']:.1f}%。",
        "Top50以下充提差率均已转正，中腰部用户资金沉淀健康，是大盘充提差率的主要贡献层。"
    ]))
    S.append(sub_title("头部提款用户分层分析"))
    S.append(P("注：两组为各自周次独立排名，并非同一批人的跨周对比。",7.5,False,C_GRAY)); S.append(Spacer(1,3))
    h3=["分层","本周提款","本周人均","上周人均","人均环比","充提差率(本)","充提差率(上)","赢家比例","活动占比","占全量%","剔除后大盘\n充提差影响"]
    rows3=[]
    for d in wdr_t:
        rows3.append([cell(d["tier"],True),cell(f"${d['tw_tx']/10000:.2f}万",False,C_DARK,TA_RIGHT),
                      cell(f"${d['tw_avg']:,.0f}",False,C_DARK,TA_RIGHT),cell(f"${d['lw_avg']:,.0f}",False,C_GRAY,TA_RIGHT),
                      rc(pct(d["tw_avg"],d["lw_avg"]),good_up=False),
                      cell(f"{d['tw_cr']:.1f}%",False,C_RED if d["tw_cr"]<-10 else C_AMBER,TA_RIGHT),
                      cell(f"{d['lw_cr']:.1f}%",False,C_GRAY,TA_RIGHT),
                      cell(f"{d['赢家']}/{d['赢家率']:.0f}%",False,C_RED if d["赢家率"]>65 else C_AMBER,TA_RIGHT),
                      cell(f"{d['活动占比']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['占全量']:.1f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{d['大盘影响']:+.2f}pp",False,C_GREEN if d["大盘影响"]>0 else C_DARK,TA_RIGHT)])
    S.append(dtable(h3,rows3,[full*x for x in [0.07,0.1,0.09,0.09,0.07,0.08,0.08,0.09,0.07,0.07,0.13]],fsize=6.5))
    S.append(Spacer(1,6))
    S.append(insight_box([
        f"剔除Top100提款用户后，大盘充提差率影响{wdr_t[3]['大盘影响']:+.2f}pp，头部提款用户对充提差率有明显拖累。",
        f"Top500提款用户活动奖励占资金来源仅{wdr_t[5]['活动占比']:.1f}%，主要靠真实赢钱后提款，非活动套利。",
    ],clr=C_AMBER))
    return S


def build_games(mfr, top30, delta):
    full=PW-2*MARGIN; S=[sec_title("四、游戏分析")]
    S.append(sub_title("Top10厂商 份额对比 & 盈亏率周对比"))
    S.append(chart_mfr_combo(mfr)); S.append(Spacer(1,4))
    h=["排名","厂商","日均投注人数(本/上)","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows=[]
    for i,(_,r) in enumerate(mfr.head(10).iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4 else C_DARK)
        rows.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r.get("厂商显示名","")),True),
                     cell(f"{r['日均投注人数']:,}/{r.get('lw_日均人数',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                     cell(f"{r['占比']:.2f}%",False,C_PURPLE if r["占比"]>10 else C_DARK,TA_RIGHT),
                     cell(f"{pl:.2f}%",False,pc,TA_RIGHT),cell(f"{r.get('lw_盈亏率',0):.2f}%",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.05,0.10,0.15,0.07,0.08,0.08,0.08,0.08,0.09,0.09]],fsize=6.5))
    S.append(Spacer(1,4))
    top1=mfr.iloc[0]
    S.append(insight_box([
        f"{top1.get('厂商显示名','')}投注份额{top1['占比']:.1f}%（上周{top1.get('lw_占比',0):.1f}%），稳居首位；环比{top1.get('投注环比',0):+.1f}%。",
        "盈亏率高于平台均值的厂商可适当扩大曝光权重；关注盈亏率持续偏低厂商的RTP设置。",
    ]))
    if delta:
        S.append(sub_title("▶ 厂商投注额环比主要驱动游戏"))
        dh=["厂商","投注额环比","增量最大游戏(+贡献)","降量最大游戏(-拖累)"]; dr=[]
        for mn,gd in delta.items():
            mrow=mfr[mfr["show_name_厂商标签id"]==mn]; mc=mrow["投注环比"].values[0] if len(mrow)>0 else 0
            game_c="游戏.名称" if "游戏.名称" in gd["up"].columns else gd["up"].columns[1]
            up=gd["up"]; dn=gd["dn"]
            us=f'{up.iloc[0][game_c][:16]}（+${up.iloc[0]["delta"]/10000:.1f}万）' if len(up)>0 and up.iloc[0]["delta"]>0 else "-"
            ds=f'{dn.iloc[0][game_c][:16]}（${dn.iloc[0]["delta"]/10000:.1f}万）' if len(dn)>0 and dn.iloc[0]["delta"]<0 else "-"
            dr.append([cell(shorten_mfr(mn),True,C_DARK),rc(mc),
                       cell(us,False,C_GREEN if us!="-" else C_GRAY),
                       cell(ds,False,C_RED if ds!="-" else C_GRAY)])
        S.append(dtable(dh,dr,[full*x for x in [0.15,0.10,0.37,0.38]],fsize=6.8)); S.append(Spacer(1,6))
    S.append(sub_title("Top30游戏详细数据"))
    S.append(chart_top30(top30)); S.append(Spacer(1,4))
    game_c8="游戏.名称" if "游戏.名称" in top30.columns else top30.columns[1]
    h2=["#","游戏名称","厂商简称","投注人数","人均局数","人均金额","投注额(万)","投注额环比","占比","盈亏率(本)","盈亏率(上)"]
    rows2=[]
    for i,(_,r) in enumerate(top30.iterrows()):
        pl=r["盈亏率"]; pc=C_RED if pl<0 else (C_GREEN if pl>4.5 else C_DARK)
        rows2.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r[game_c8])[:18],True),
                      cell(str(r.get("厂商简称",""))[:5]),
                      cell(f"{int(r['投注人数']):,}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['人均局数']:.0f}",False,C_DARK,TA_RIGHT),
                      cell(f"${r['人均金额']:.1f}",False,C_DARK,TA_RIGHT),
                      cell(f"{r['投注金额']/10000:.1f}",False,C_DARK,TA_RIGHT),rc(r.get("投注环比",0)),
                      cell(f"{r['占比']:.2f}%",False,C_DARK,TA_RIGHT),
                      cell(f"{pl:.2f}%",False,pc,TA_RIGHT),
                      cell(f"{r.get('lw_盈亏率',0):.2f}%" if not pd.isna(r.get("lw_盈亏率",np.nan)) else "-",False,C_GRAY,TA_RIGHT)])
    S.append(dtable(h2,rows2,[full*x for x in [0.04,0.18,0.07,0.08,0.06,0.08,0.08,0.08,0.07,0.08,0.08]],fsize=6.5))
    return S


def build_activities(act, tot_gift, fdr):
    full=PW-2*MARGIN; S=[sec_title("五、活动分析")]
    S.append(sub_title("各活动赠送效果（全量，含环比）"))
    S.append(chart_activities(act)); S.append(Spacer(1,4))
    h=["活动名称","本周赠送","上周赠送","金额环比","本周日均\n赠送人数","上周日均\n赠送人数","人数环比","本周人均\n赠送金额","占比"]
    rows=[]
    for _,r in act.iterrows():
        rows.append([cell(str(r["name_账变opt_id"])[:18],True),
                     cell(f"${r['赠送金额']:,.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"${r['lw_赠']:,.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r["环比"] if not pd.isna(r.get("环比",np.nan)) else 0),
                     cell(f"{r.get('日均_本',0):.0f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r.get('日均_上',0):.0f}",False,C_GRAY,TA_RIGHT),
                     rc(r.get("人数环比",0) if not pd.isna(r.get("人数环比",np.nan)) else 0),
                     cell(f"${r.get('人均',0):.2f}",False,C_DARK,TA_RIGHT),
                     cell(f"{r['占比']:.1f}%",False,C_DARK,TA_RIGHT)])
    S.append(dtable(h,rows,[full*x for x in [0.20,0.11,0.11,0.07,0.12,0.12,0.07,0.11,0.09]],fsize=6.5))
    S.append(Spacer(1,4)); daily=tot_gift/7/10000
    S.append(P(f"本周总赠送金额：${tot_gift/10000:.2f}万 | 日均赠送：${daily:.2f}万",9,True,C_DARK))
    S.append(Spacer(1,6))
    S.append(insight_box([f"各类赠送活动全量列出（共{len(act)}个），合计本周日均赠送{daily:.1f}万USD。"]))
    S.append(Spacer(1,8))

    # 首充活动用户留存
    S.append(sub_title("首次充值活动用户 充值留存分析"))
    stage=fdr.get("stage",pd.DataFrame())
    if len(stage)>0 and "指标" in stage.columns:
        lws=LAST_WEEK[0]
        S.append(P(f"▸ 两周阶段汇总（{lws[:4]}.{lws[4:6]}.{lws[6:]} - {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}）",9,True,C_DARK))
        S.append(Spacer(1,3))
        sr=stage[stage["指标"]=="留存率"]; sn=stage[stage["指标"]=="留存人数"]
        sa=stage[stage["指标"]=="人均充值金额"]
        if len(sr)>0:
            user_c3="账变事件用户数" if "账变事件用户数" in sn.columns else sn.columns[1]
            u=int(sn[user_c3].values[0]) if len(sn)>0 else 0
            S.append(P(f"首充活动用户总数：{u:,}人",8.5,False,C_DARK))
            dc=["当日","1日","2日","3日","4日","5日","6日","7日"]
            dc_avail=[c for c in dc if c in sr.columns]
            srows=[]
            if len(sr)>0: srows.append(["留存率"]+[str(sr.iloc[0].get(c,"-")) for c in dc_avail])
            if len(sn)>0: srows.append(["留存人数"]+[str(sn.iloc[0].get(c,"-")) for c in dc_avail])
            if len(sa)>0: srows.append(["人均充值($)"]+[str(sa.iloc[0].get(c,"-")) for c in dc_avail])
            if srows:
                tr2=[[cell(row[0],True,C_DARK)]+[cell(str(v),False,C_DARK,TA_CENTER) for v in row[1:]] for row in srows]
                S.append(dtable(["指标"]+dc_avail,tr2,[full*.12]+[full*(0.88/len(dc_avail))]*len(dc_avail),fsize=6.8))
        S.append(Spacer(1,6))

    S.append(P("▸ 本周 vs 上周 首充活动用户留存趋势",9,True,C_DARK)); S.append(Spacer(1,3))
    S.append(chart_first_dep_ret(fdr)); S.append(Spacer(1,4))
    tw_u=fdr.get("tw_users",0); lw_u=fdr.get("lw_users",0)
    tw_pf=fdr.get("tw_platform_fc",0); lw_pf=fdr.get("lw_platform_fc",0)
    tw_r=tw_u/tw_pf*100 if tw_pf>0 else 0; lw_r=lw_u/lw_pf*100 if lw_pf>0 else 0
    tv=[fdr.get(f"tw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    lv=[fdr.get(f"lw_{d}日",np.nan) for d in ["1","2","3","4","5","6","7"]]
    def _fr(v): return f"{v:.1f}%" if not(v is None or(isinstance(v,float) and np.isnan(v))) else "-"
    rh=["周次","活动用户数","平台首充\n人数","活动用户\n占比","D1留存","D2留存","D3留存","D4留存","D5留存","D6留存","D7留存"]
    rrows=[
        [cell(f"本周 {THIS_WEEK[0][4:6]}/{THIS_WEEK[0][6:]}-{THIS_WEEK[1][4:6]}/{THIS_WEEK[1][6:]}",True,C_BLUE),
         cell(f"{tw_u:,}",False,C_DARK,TA_RIGHT),cell(f"{tw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{tw_r:.1f}%",False,C_PURPLE,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GREEN if not pd.isna(v) and not pd.isna(lval) and v>=lval else(C_RED if not pd.isna(v) and not pd.isna(lval) and v<lval else C_DARK),TA_RIGHT)
           for v,lval in zip(tv,lv)],
        [cell(f"上周 {LAST_WEEK[0][4:6]}/{LAST_WEEK[0][6:]}-{LAST_WEEK[1][4:6]}/{LAST_WEEK[1][6:]}",True,C_GRAY),
         cell(f"{lw_u:,}",False,C_GRAY,TA_RIGHT),cell(f"{lw_pf:,}",False,C_GRAY,TA_RIGHT),
         cell(f"{lw_r:.1f}%",False,C_GRAY,TA_RIGHT),
        ]+[cell(_fr(v),False,C_GRAY,TA_RIGHT) for v in lv],
    ]
    S.append(dtable(rh,rrows,[full*.16,full*.09,full*.09,full*.08]+[full*.084]*7,fsize=7,zebra=False))
    S.append(Spacer(1,4))
    tw_d1=fdr.get("tw_1日",np.nan); lw_d1=fdr.get("lw_1日",np.nan); ins=[]
    if not np.isnan(tw_d1) and not np.isnan(lw_d1):
        d=tw_d1-lw_d1; ins.append(f"首充活动用户次日留存{tw_d1:.1f}%（上周{lw_d1:.1f}%，{d:+.1f}pp），{'留存改善' if d>0 else '留存下降，建议优化次日触达策略'}。")
    ins.append(f"本周活动用户{tw_u:,}人，占平台首充{tw_r:.1f}%（上周{lw_u:,}/{lw_r:.1f}%）。")
    ins.append("建议：对D1/D2留存用户设置阶梯式再充值道具激励；对D3后流失用户做专项召回（24h内触达效果最佳）。")
    S.append(insight_box(ins,clr=C_AMBER))
    return S


def build_risk(hr, sp, top20g, rg, top500, dt_full):
    full=PW-2*MARGIN; S=[sec_title("六、用户游戏风险专项分析",clr=colors.HexColor("#7c2d12"))]
    tot_tx=top500["提款金额"].sum() if "提款金额" in top500.columns else 0
    tot_win=top500["公司输赢"].sum() if "公司输赢" in top500.columns else 0
    wns=(top500["公司输赢"]<0).sum() if "公司输赢" in top500.columns else 0
    S.append(sub_title("Top500提款用户总览"))
    row_k=[]
    for label,val,sub in [("Top500提款总额",f"${tot_tx/10000:.1f}万",""),
                           ("平台净赔付",f"${abs(tot_win)/10000:.1f}万","平台向该群体净赔"),
                           ("赢家比例",f"{wns/500*100:.1f}%",f"{wns}赢/{500-wns}输"),
                           ("高风险用户",f"{len(hr)}人","公司净输>$10,000")]:
        fw=full/4-4
        row_k.append(Table([[P(label,7.5,False,C_GRAY)],[P(val,15,True,C_DARK)],[P(sub,7,False,C_GRAY)]],
                           colWidths=[fw],style=TableStyle([
                               ("BACKGROUND",(0,0),(-1,-1),C_LGRAY),("BOX",(0,0),(-1,-1),0.5,C_BORDER),
                               ("LEFTPADDING",(0,0),(-1,-1),8),("TOPPADDING",(0,0),(-1,-1),5),
                               ("BOTTOMPADDING",(0,0),(-1,-1),5)])))
    t=Table([row_k],colWidths=[full/4]*4,hAlign="LEFT",vAlign="TOP")
    t.setStyle(TableStyle([("VALIGN",(0,0),(-1,-1),"TOP"),("LEFTPADDING",(0,0),(-1,-1),2),
                            ("RIGHTPADDING",(0,0),(-1,-1),2),("TOPPADDING",(0,0),(-1,-1),2),
                            ("BOTTOMPADDING",(0,0),(-1,-1),2)])); S.append(t); S.append(Spacer(1,8))
    S.append(sub_title("用户风险分布（投充比 vs 公司输赢）"))
    S.append(chart_risk_scatter(top500)); S.append(Spacer(1,6))
    hl=abs(hr["公司输赢"].sum())/10000 if len(hr)>0 else 0
    S.append(insight_box([
        f"Top500提款用户中赢家{wns}人（{wns/500*100:.1f}%），平台净赔付${abs(tot_win)/10000:.1f}万。",
        f"{len(hr)}名超级赢家（公司净输>$10,000）合计导致平台净输${hl:.1f}万。",
    ]))
    if len(hr)>0:
        S.append(sub_title("▶ 高风险用户明细（需立即人工审核）"))
        h=["账户ID","提款金额","充值金额","公司输赢","投注金额","投充比","主玩厂商","主玩游戏","风险标签"]
        rows=[]
        for _,r in hr.iterrows():
            ratio=r.get("投充比",0) if not pd.isna(r.get("投充比",0)) else 0
            tags=[]
            if r.get("充值金额",0)<5000: tags.append("低充高提")
            if ratio>50: tags.append("超高投充")
            rows.append([cell(str(r["账户ID"]),True),
                         cell(f"${r.get('提款金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"${r.get('充值金额',0):,.0f}",False,C_RED if r.get("充值金额",0)<5000 else C_DARK,TA_RIGHT),
                         cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED,TA_RIGHT),
                         cell(f"${r.get('投注金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                         cell(f"{ratio:.0f}x",False,C_RED if ratio>50 else C_AMBER,TA_RIGHT),
                         cell(str(r.get("主玩厂商",""))[:8]),cell(str(r.get("主玩游戏",""))[:16]),
                         cell("+".join(tags) if tags else "超级赢家",True,C_RED)])
        S.append(dtable(h,rows,[full*x for x in [0.12,0.11,0.11,0.11,0.11,0.07,0.09,0.16,0.12]],fsize=6.5))
        S.append(Spacer(1,6))
    S.append(Spacer(1,8)); S.append(sub_title("Top20提款用户明细（含主玩游戏）"))
    th=["#","账户ID","提款金额","充值金额","公司输赢","活动奖励","主玩游戏","状态"]; tr=[]
    for i,(_,r) in enumerate(dt_full.head(20).iterrows()):
        win=r.get("公司输赢",0)<0
        tr.append([cell(str(i+1),False,C_GRAY,TA_CENTER),cell(str(r["账户ID"])),
                   cell(f"${r.get('提款金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r.get('充值金额',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED if win else C_GREEN,TA_RIGHT),
                   cell(f"${r.get('活动奖励',0):,.0f}",False,C_DARK,TA_RIGHT),
                   cell(str(r.get("主玩游戏",""))[:16] if "主玩游戏" in r and not pd.isna(r.get("主玩游戏","")) else "-",False,C_GRAY),
                   cell("玩家赢" if win else "公司赢",False,C_RED if win else C_GREEN,TA_CENTER)])
    S.append(dtable(th,tr,[full*x for x in [0.04,0.13,0.11,0.11,0.11,0.09,0.28,0.1]],fsize=6.5))
    S.append(Spacer(1,6))
    pg,ma=load_pref()
    S.append(sub_title("Top500提款用户游戏偏好")); S.append(chart_pref(pg,ma)); S.append(Spacer(1,4))
    if len(rg)>0:
        S.append(sub_title("▶ 高危游戏专项分析")); S.append(chart_risk_games(rg)); S.append(Spacer(1,4))
        gh=["游戏名称","厂商","玩家数","投注金额","公司输赢","盈亏率","赢家率","人均投注","风险"]; gr=[]
        for _,r in rg.head(12).iterrows():
            pl=r["盈亏率"]; rl="极高" if abs(pl)>40 or r["公司输赢"]<-50000 else("高" if pl<-10 else "关注")
            rc2=C_RED if rl in("极高","高") else C_AMBER
            gr.append([cell(str(r.get("游戏名称",""))[:18],True),cell(str(r.get("show_name_厂商标签id",""))[:8]),
                       cell(f"{int(r.get('玩家数',0))}",False,C_DARK,TA_RIGHT),
                       cell(f"${r.get('投注金额',0)/10000:.1f}万",False,C_DARK,TA_RIGHT),
                       cell(f"${r.get('公司输赢',0):,.0f}",False,C_RED,TA_RIGHT),
                       cell(f"{pl:.1f}%",False,C_RED,TA_RIGHT),
                       cell(f"{r.get('赢家率',0):.0f}%",False,C_RED if r.get("赢家率",0)>60 else C_AMBER,TA_RIGHT),
                       cell(f"${r.get('人均投注额',0):,.0f}",False,C_DARK,TA_RIGHT),
                       cell(rl,True,rc2)])
        S.append(dtable(gh,gr,[full*x for x in [0.21,0.10,0.07,0.10,0.11,0.08,0.08,0.10,0.07]],fsize=6.5))
        S.append(Spacer(1,6))
        top_rg=rg.iloc[0]
        S.append(insight_box([
            f"高危游戏{top_rg.get('游戏名称','')}（{top_rg.get('show_name_厂商标签id','')}）公司输赢${top_rg.get('公司输赢',0):,.0f}，盈亏率{top_rg.get('盈亏率',0):.1f}%，建议复审RTP参数。",
            "建议对高赢家率游戏实施单账户赢额上限，防范套利风险。"
        ],clr=C_RED))
    return S


def build_conclusion(K, dep_t, wdr_t, mfr, act, rg, hr):
    full=PW-2*MARGIN; S=[sec_title("七、总结与行动建议")]
    cr_d=K["tw_充提差比"]-K["lw_充提差比"]; rd=K["tw_首充次日复充率"]-K["lw_首充次日复充率"]
    S.append(sub_title("▶ 本周亮点")); bg_g=colors.HexColor("#f0fdf4")
    highlights=[
        f"充值{K['tw_充值金额']/10000:.1f}万（{K['pct_充值金额']:+.1f}%），充提差{K['tw_充提差']/10000:.1f}万，充提差率{K['tw_充提差比']:.2f}%（{cr_d:+.2f}pp）。"
    ]
    if K["pct_首充人数"]>0:
        highlights.append(f"注册人数{int(K['tw_注册人数']):,}（{K['pct_注册人数']:+.1f}%），首充人数{int(K['tw_首充人数']):,}（{K['pct_首充人数']:+.1f}%），拉新规模扩大。")
    if K["pct_公司输赢"]>0:
        highlights.append(f"公司输赢{K['tw_公司输赢']/10000:.1f}万（{K['pct_公司输赢']:+.1f}%），盈亏率{K['tw_盈亏率']:.3f}%，较上周改善。")
    top1=mfr.iloc[0]
    highlights.append(f"{top1.get('厂商显示名','')}投注份额{top1['占比']:.1f}%（{top1.get('投注环比',0):+.1f}%）。")
    for hl in highlights:
        S.append(Table([[P(f"• {hl}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_g),
                                         ("LEFTPADDING",(0,0),(-1,-1),12),
                                         ("TOPPADDING",(0,0),(-1,-1),3),
                                         ("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))
    S.append(sub_title("▶ 风险预警")); bg_r=colors.HexColor("#fff5f5")
    risks=[]
    if K["pct_首充Arppu"]<-5 or rd<-1:
        risks.append(f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），首充次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp），新用户质量需关注。")
    if len(hr)>0:
        hl2=abs(hr["公司输赢"].sum())/10000
        risks.append(f"{len(hr)}名高风险用户（公司净输>$10,000）合计净输${hl2:.1f}万，需立即人工审核。")
    top_rg_r=rg.iloc[0] if len(rg)>0 else None
    if top_rg_r is not None and top_rg_r.get("公司输赢",0)<-20000:
        risks.append(f"高危游戏「{str(top_rg_r.get('游戏名称',''))[:16]}」公司输赢${top_rg_r.get('公司输赢',0):,.0f}，赢家率{top_rg_r.get('赢家率',0):.0f}%，建议复核RTP参数。")
    if wdr_t[3]["大盘影响"]<-1:
        risks.append(f"头部提款用户对充提差率拖累{wdr_t[3]['大盘影响']:+.2f}pp（剔除Top100后）。")
    if K["pct_真实消耗"]<-15:
        risks.append(f"推广日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（{K['pct_真实消耗']:+.1f}%），建议确认是否计划内削减。")
    if not risks: risks.append("本周暂无重大风险预警，各指标维持正常区间。")
    for r2 in risks:
        S.append(Table([[P(f"• {r2}",8.5,False,C_DARK)]],colWidths=[full],
                       style=TableStyle([("BACKGROUND",(0,0),(-1,-1),bg_r),
                                         ("LEFTPADDING",(0,0),(-1,-1),12),
                                         ("TOPPADDING",(0,0),(-1,-1),3),
                                         ("BOTTOMPADDING",(0,0),(-1,-1),3)])))
    S.append(Spacer(1,8))
    S.append(sub_title("▶ 行动建议"))
    actions=[]
    if len(hr)>0:
        top_hr=hr.sort_values("公司输赢").iloc[0]
        actions.append(("[紧急]","处置高风险提款用户",
                        f"账户{top_hr['账户ID']}：提款${top_hr.get('提款金额',0):,.0f}，公司输赢${top_hr.get('公司输赢',0):,.0f}，高度异常，建议立即人工审核。"))
    if K["pct_首充Arppu"]<-5 or rd<-1:
        actions.append(("[本周]","优化首充质量与次日激活",
                        f"首充ARPPU${K['tw_首充Arppu']:.2f}（{K['pct_首充Arppu']:+.1f}%），次日留存{K['tw_首充次日复充率']:.1f}%（{rd:+.1f}pp）。建议注册后1h/24h内推送首充引导。"))
    if top_rg_r is not None and top_rg_r.get("公司输赢",0)<-20000:
        actions.append(("[本周]","高危游戏风险管控",
                        f"对赢家率>60%的高危游戏实施单账户赢额上限，同步复核「{str(top_rg_r.get('游戏名称',''))[:16]}」RTP参数。"))
    if K["pct_真实消耗"]<-15:
        actions.append(("[本周]","关注推广消耗削减影响",
                        f"日均消耗{K['tw_真实消耗_日均']/10000:.2f}万（{K['pct_真实消耗']:+.1f}%），若非计划内，建议与投放团队确认。"))
    if not actions:
        actions.append(("[常规]","持续监控核心指标","各核心指标维持正常，建议持续监控并保持现有运营节奏。"))
    pmap={"[紧急]":C_RED,"[本周]":C_AMBER,"[常规]":C_GREEN}
    rows=[[cell(pri,True,pmap.get(pri[:4],C_GRAY),TA_CENTER),cell(title,True,C_DARK),cell(desc,False,C_GRAY)]
          for pri,title,desc in actions]
    S.append(dtable(["优先级","建议事项","执行说明与数据依据"],rows,[full*x for x in [0.1,0.22,0.68]]))
    return S


def header_footer(c, doc):
    c.saveState(); w,h=A4
    c.setFillColor(colors.HexColor("#0f172a")); c.rect(0,h-26,w,26,fill=1,stroke=0)
    c.setFillColor(colors.white); c.setFont(FNB,10)
    c.drawString(MARGIN,h-17,f"{PLATFORM} 平台数据周报  {THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}")
    c.setFont(FN,8); c.drawRightString(w-MARGIN,h-17,f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}")
    c.setFillColor(colors.HexColor("#f1f5f9")); c.rect(0,0,w,16,fill=1,stroke=0)
    c.setFillColor(colors.HexColor("#64748b")); c.setFont(FN,7); c.drawRightString(w-MARGIN,5,f"第 {doc.page} 页")
    c.restoreState()


# ══════════════════════════════════════════════
# 主函数
# ══════════════════════════════════════════════
def main():
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"📊 {PLATFORM} 周报 PDF 生成中 — {THIS_WEEK[0]}-{THIS_WEEK[1]} ...")
    resolve_files(DATA_ROOT)
    setup()
    print("  ▶ 加载数据...")

    K, trend         = load_platform()
    print(f"  ▶ 推广消耗日均{K['tw_真实消耗_日均']/10000:.2f}万，充提差ROI={K['tw_充提差ROI']:.2f}x")
    dash_ret         = load_dashboard_retention()
    weekly_ret       = load_weekly_retention()
    agents           = load_agents()
    agent_ret, ends  = load_agent_ret()
    tw_v, lw_v       = load_vip()
    dep_t,wdr_t,dc_tw,dt_top = load_top_users()
    top30            = load_games()
    mfr              = load_mfr()
    mfr_delta        = load_mfr_delta()
    act_df, tot_gift = load_activities()
    fdr              = load_first_dep_ret()

    # 风险分析
    dt_raw=pd.read_csv(FILES["dt_tw"]); dc_raw=pd.read_csv(FILES["dc_tw"])
    for df in [dt_raw, dc_raw]:
        for c in ["充值金额","提款金额","公司输赢","活动奖励","投注金额","充提差"]:
            if c in df.columns:
                df[c]=pd.to_numeric(df[c].astype(str).str.replace(",",""),errors="coerce")

    hr,sp,top20g,rg,top500 = load_risk(dt_raw, dc_raw)

    # 给top20提款用户加主玩游戏
    dpf=pd.read_csv(FILES["pref_tw"]); dpf["阶段汇总"]=dpf["阶段汇总"].apply(clean)
    tg2=(dpf[dpf["分析指标"]=="投注金额"].sort_values("阶段汇总",ascending=False)
         .groupby("账户ID").first()[["show_name_厂商标签id","游戏名称"]]
         .rename(columns={"show_name_厂商标签id":"主玩厂商","游戏名称":"主玩游戏"}).reset_index())
    dt_top=dt_top.merge(tg2,on="账户ID",how="left")

    # 导出风控Excel
    print("  ▶ 导出风控Excel...")
    excel_out=OUTPUT_DIR/f"{PLATFORM}_风控名单_{THIS_WEEK[0]}-{THIS_WEEK[1]}.xlsx"
    with pd.ExcelWriter(str(excel_out),engine="openpyxl") as xw:
        cols_hr=["账户ID","提款金额","充值金额","公司输赢","投注金额","活动奖励","主玩厂商","主玩游戏","投充比"]
        hr[[c for c in cols_hr if c in hr.columns]].to_excel(xw,sheet_name="高风险用户",index=False)
        if len(sp)>0:
            sp[[c for c in ["账户ID","提款金额","充值金额","公司输赢","投充比"] if c in sp.columns]].to_excel(xw,sheet_name="特殊异常用户",index=False)
    print(f"  ✅ 风控Excel: {excel_out}")

    # 构建PDF
    print("  ▶ 构建PDF章节...")
    out=OUTPUT_DIR/f"{PLATFORM}周报_{THIS_WEEK[0]}-{THIS_WEEK[1]}.pdf"
    doc=SimpleDocTemplate(str(out),pagesize=A4,leftMargin=MARGIN,rightMargin=MARGIN,
                          topMargin=MARGIN+22,bottomMargin=MARGIN+10,
                          title=f"{PLATFORM}平台周报 {THIS_WEEK[0]}-{THIS_WEEK[1]}")
    story=[
        Spacer(1,3*cm),
        P(f"{PLATFORM} 平台数据周报",30,True,C_DARK,TA_CENTER),Spacer(1,.5*cm),
        P(f"统计周期：{THIS_WEEK[0][:4]}.{THIS_WEEK[0][4:6]}.{THIS_WEEK[0][6:]} — {THIS_WEEK[1][:4]}.{THIS_WEEK[1][4:6]}.{THIS_WEEK[1][6:]}",
          14,False,C_GRAY,TA_CENTER),Spacer(1,.2*cm),
        P(f"对比基准：{LAST_WEEK[0][:4]}.{LAST_WEEK[0][4:6]}.{LAST_WEEK[0][6:]} — {LAST_WEEK[1][:4]}.{LAST_WEEK[1][4:6]}.{LAST_WEEK[1][6:]}",
          12,False,C_GRAY,TA_CENTER),Spacer(1,.5*cm),
        HRFlowable(width="50%",thickness=2,color=C_BLUE,hAlign="CENTER"),PageBreak()
    ]
    story+=build_overview(K,trend,weekly_ret,dash_ret);               story.append(PageBreak())
    story+=build_agents(agents,agent_ret,ends);                        story.append(PageBreak())
    story+=build_users(tw_v,lw_v,dep_t,wdr_t,dc_tw,dt_top);           story.append(PageBreak())
    story+=build_games(mfr,top30,mfr_delta);                          story.append(PageBreak())
    story+=build_activities(act_df,tot_gift,fdr);                     story.append(PageBreak())
    story+=build_risk(hr,sp,top20g,rg,top500,dt_top);                 story.append(PageBreak())
    story+=build_conclusion(K,dep_t,wdr_t,mfr,act_df,rg,hr)

    print("  ▶ 渲染PDF...")
    doc.build(story,onFirstPage=header_footer,onLaterPages=header_footer)
    size=out.stat().st_size/1024
    print(f"\n✅ PDF完成：{out}  ({size:.0f} KB)")
    return str(out), str(excel_out)


if __name__ == "__main__":
    main()

📊 Lucro 周报 PDF 生成中 — 20260529-20260604 ...
     ✅ [platform       ] 平台报表_USD_20260605151312.xlsx
     ✅ [daily          ] 日报-大盘日报_20260605(2).xlsx
     ✅ [retention      ] 整体 首充留存（近7天）_全量数据_20260508_20260604 (1).csv
     ✅ [agent_plat     ] 平台报表-总代_USD_20260605185511.xlsx
     ✅ [agent_promo    ] 推广报表-总代_USD_20260605185533.xlsx
     ✅ [agent_ret      ] 首充充值留存_全量数据_20260508_20260604.csv
     ✅ [vip            ] VIP报表_USD_20260605190039.xlsx
     ✅ [dt_tw          ] top提款用户_全量数据_20260529_20260604_日期对比20260522_20260528 (2).csv
     ✅ [dt_lw          ] top提款用户_全量数据_20260522_20260528_日期对比20260515_20260521 (1).csv
     ✅ [dc_tw          ] 头部充值用户_全量数据_20260529_20260604_日期对比20260522_20260528 (1).csv
     ✅ [dc_lw          ] 头部充值用户_全量数据_20260522_20260528_日期对比20260515_20260521 (1).csv
     ✅ [pref_tw        ] 本周top500提款用户游戏偏好_全量数据_20260529_20260604 (1).csv
     ✅ [pref_lw        ] 上周top500提款用户游戏偏好_全量数据_20260522_20260528 (1).csv
     ✅ [mfr            ] 厂商投注数据_全量数据_20260522_20260604 (1).csv
     